# AM2020 main case — growth, deformation and detachment

**19 children, four series, about 3 h 30 min. This is the PI-facing case.**

The smoke gates are done: `AM2020_mural_geometry_gate` returned `MURAL_READY`
twice with identical numbers (runs `20260810-075435` and `20260810-192247`).
This case is not a gate. It has no pass/fail verdict — it produces a **results
table** plus a per-child admissibility check, so a child that failed is named
rather than averaged into a row.

## The three series are the PI's three questions

**`growth_*` — how does growth rate depend on flow rate?** Three children at
wall shear 50, 100 and 200 s⁻¹, growth on, 30 s each. The assessor fits the
exponent in `Vn ∝ U^n` and compares it with the transport-limited **1/3**. Shear
enhancement is inactive by construction (`gammaLow = 3000 s⁻¹` is far above
every shear rate here), so any dependence is platelet transport, not activation
— and the realised maximum shear rate is measured from the written field to
confirm that rather than assert it.

**`req_*` — the competition.** Nine children spanning flow (50/100/200 s⁻¹),
stiffness (58.6 Pa / 1 kPa / 11.6 kPa, all sourced) and permeability (a decade
either side of 3.5e-14 m²). The bond is **welded** and the run measures the
peak traction it carries. That number *is* the adhesive strength detachment
would require at that operating point — which is how this case handles the one
parameter with no source in the literature: it never picks a value.

There is a prediction attached, and it can fail: to first order the load should
be set by the **flow**, not the stiffness, because drag is a fluid quantity and
`G` cancels in `traction = G × strain`. If the required strength turns out to
track `G` across the 198× span, that is the more interesting answer and the
assessor says so.

**`demo_*` — the thing to watch.** Two children with the bond live, at 1.2 and
1.8 Pa against the measured 2.65 Pa peak. Growth on, adhesion on, finest write
interval of any series (24 frames) because these are for ParaView. The gate
already showed 1.2 Pa releasing 59 of 60 bonded faces over 18 timesteps.

## What to tell the PI, and what not to

`Gc` is restored to the **physiological 5.547 J/m²** (source 24). The smoke
gates used an unreachable value so any release was unambiguously adhesive; the
main case lets cohesive failure compete instead of excluding it by
construction. Bulk fracture is not expected to fire — run `20260809-113416`
established that AT2 needs about 2900% strain on this formulation — so if
cohesive release *does* appear, that is a finding, and it is reported
separately from adhesive release.

Six limitations travel with the result and are printed by the last cell. The
two that matter most: the partitioned coupling is first-order with a measured
**38% splitting error**, so trends are meaningful and absolute numbers are not;
and adhesive strength has **no literature source**, so the deliverable is the
strength *required*, never a claim that a value is physiological.

## Cost

Growth 36 min, required-strength 104 min, demonstration 26 min. Children run in
that order and each writes its artifacts as it finishes, so an interrupted
session still returns every completed series.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from datetime import datetime
from pathlib import Path
import hashlib, json, os, re, shutil, subprocess, zipfile

ARTIFACT = 'main_case_v1'
CONTENT = Path('/content')

# WHERE THE CODE COMES FROM (L1: research/designs/2026-10-05-L1-colab-from-github.md
# rev. 2, APPROVED by Alireza 2026-10-05).
#
#   SOURCE = 'github'  (default) clone the private repo at EXPECTED_TAG, verify
#                      its content against EXPECTED_REPO_DIGEST, build the
#                      package from the verified clone, run it.
#   SOURCE = 'zip'     today's path: the Drive package pinned by EXPECTED_SHA.
#                      The fallback if a github launch is refused.
#
# The three pins are written by scripts/build_main_case_colab_package.sh. Do
# NOT edit them by hand. EXPECTED_REPO_DIGEST is the content of the release in
# the form the repo holds it, so a moved tag, a shallow clone or any edited
# file is refused before a session is spent. This notebook's own pin lines are
# blanked before hashing, so a pin never changes the digest it pins.
#
# The token: a fine-grained personal access token, Contents read-only, this
# repo only, stored in Colab Secrets as GITHUB_TOKEN (setup steps are printed
# if it is missing). It reaches git ONLY through the environment and a
# GIT_ASKPASS helper (username x-access-token, password $GH_TOKEN): never in
# argv, never in .git/config, scrubbed from every message this cell prints.
SOURCE = 'github'
REPO_URL = 'https://github.com/alirezakianim/openfoam-blood.git'
EXPECTED_REPO_DIGEST = '8497d46a7010a30e85b9458690ed169121cd4225848fe4a0c50594de45dc42c2'
EXPECTED_TAG = 'digest-8497d46a7010'
# S21 VALIDATION ONLY (BATCH 70; research/designs/2026-10-05-S21-clean-solver.md
# V21-6/V21-7). Run this cell TWICE IN THE SAME RUNTIME, without restarting:
# first with R0_FIRST = True (the R0 reference package: release-4198bb17 plus
# three controls-only identity twins), then with R0_FIRST = False (this
# release). Both then run on the same CPU, which the block-coupled identity
# needs: UMFPACK/OpenBLAS choose kernels per CPU, so a block-coupled field from
# another VM can differ in its last bits. Every other batch: leave it False.
R0_FIRST = False
R0_TAG = 'digest-8b79a466e8e4'
R0_REPO_DIGEST = '8b79a466e8e40b7aa29040f155fffbf5bc2830ed9eeee45b72abfe82d5fff414'
if R0_FIRST:
    EXPECTED_REPO_DIGEST, EXPECTED_TAG = R0_REPO_DIGEST, R0_TAG
    print(f'R0_FIRST: running the S21 R0 reference package {R0_TAG}, not this release')
GITHUB_BUILD = CONTENT / 'github_build'

# >>> repo_digest
# ONE implementation. The build (scripts/build_main_case_colab_package.sh),
# release_check.py R5 and the runner (via REPO_STAMP.json) all execute THIS
# block, extracted from this cell; none carries a second copy.
def manifest_entries(manifest):
    """tools/manifest.txt's text -> its entries (blank and # lines skipped)."""
    entries = [line.strip() for line in manifest.splitlines()
               if line.strip() and not line.strip().startswith('#')]
    if len(entries) != len(set(entries)):
        raise ValueError('manifest has duplicate entries')
    return entries


def repo_digest(root, manifest):
    """sha256 over the sorted (relative path, sha256) of every manifest file
    under root, in the form the repo holds it. A notebook's pin values
    (EXPECTED_SHA, EXPECTED_REPO_DIGEST: 64 hex; EXPECTED_TAG: digest-<12 hex>)
    are blanked first. A missing manifest file raises: refuse, never skip."""
    import hashlib, re
    from pathlib import Path
    pin = re.compile(rb"((?:EXPECTED_SHA|EXPECTED_REPO_DIGEST)\s*=\s*\\?['\"])[0-9a-f]{64}"
                     rb"|(EXPECTED_TAG\s*=\s*\\?['\"]digest-)[0-9a-f]{12}")
    root = Path(root)
    total = hashlib.sha256()
    for rel in sorted(manifest_entries(manifest)):
        data = (root / rel).read_bytes()
        if rel.endswith('.ipynb'):
            data = pin.sub(lambda m: (m.group(1) or m.group(2)) + b'<pinned>', data)
        total.update(f'{rel}\0{hashlib.sha256(data).hexdigest()}\n'.encode())
    return total.hexdigest()
# <<< repo_digest

if SOURCE == 'zip':
    package = CONTENT / 'drive/MyDrive/OpenFOAM_Blood/packages/AM2020_main_case_colab_package.zip'
    if not package.is_file():
        raise FileNotFoundError(f'Upload the package first: {package}')

    # HARD ASSERT ON THE BINARY. This notebook had NO package check of any kind
    # until 2026-09-02, while the 7-minute gates notebook has had one for days --
    # exactly backwards, since a batch here costs up to 9 HOURS and the failure is
    # silent: the run completes and the numbers get filed against the wrong solver.
    #
    # Not hypothetical. Three builds in two days had to be discarded:
    #   3277f1f3  splits the c/Wp convection operator (gate_a_exact 1.784e-05)
    #   32cbd2d9  harmonic mean divides by a zero D field
    #   c7ee87a2  gammaPlateletf built by assignment, so its boundary patch types
    #             were `calculated` rather than the interpolation's own -- and
    #             fvc::laplacian(gammaPlateletf, w_) READS that boundary. It would
    #             have changed every D > 0 child while looking like a D = 0 guard.
    #
    # Any of the three would have run to completion here and produced plausible
    # numbers. Do NOT edit this sha to match what is in Drive -- replace the
    # package.
    # The package also CONTAINS a copy of this notebook, which is one
    # revision behind whatever you are reading -- editing the notebook
    # changes the zip, which changes the sha, which changes the notebook.
    # The copy inside the zip is never the one that runs. Ignore it.
    EXPECTED_SHA = '2d3eed1ef3cbb06562db963e2b0afaaf73de9f940b0354275a9b4894a7cfd8e5'
    actual_sha = hashlib.sha256(package.read_bytes()).hexdigest()
    print('package sha256:', actual_sha)
    if actual_sha != EXPECTED_SHA:
        raise SystemExit(
            'WRONG PACKAGE -- refusing to spend a session on an unknown binary.\n'
            f'  in Drive : {actual_sha}\n'
            f'  expected : {EXPECTED_SHA}\n'
            'Replace the package in Drive and re-run this cell.'
        )
    if not package.is_file():
        raise FileNotFoundError(f'Upload the exact package first: {package}')
    runner_args = []
elif SOURCE == 'github':
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
        why = 'the secret is empty'
    except Exception as error:   # SecretNotFoundError, NotebookAccessError
        token, why = None, type(error).__name__
    if not token:
        raise SystemExit(
            f'NO GITHUB TOKEN ({why}) -- refusing. One-time setup:\n'
            '  1. github.com -> Settings -> Developer settings -> Fine-grained personal\n'
            '     access tokens -> Generate. Repository access: ONLY\n'
            '     alirezakianim/openfoam-blood. Permissions: Contents Read-only.\n'
            '     Expiry: 90 days (tokens expire; a refusal after that means a new one).\n'
            '  2. Colab -> the key icon (Secrets) -> Add new secret GITHUB_TOKEN ->\n'
            '     paste -> turn ON "Notebook access". Never paste it into a cell.\n'
            "  Fallback: SOURCE = 'zip' with this release's Drive package."
        )
    clone = CONTENT / 'openfoam-blood'
    shutil.rmtree(clone, ignore_errors=True)
    shutil.rmtree(GITHUB_BUILD, ignore_errors=True)
    askpass = CONTENT / 'git_askpass.sh'
    askpass.write_text('#!/bin/sh\ncase "$1" in\n  Username*) echo x-access-token ;;\n'
                       '  *) printf \'%s\\n\' "$GH_TOKEN" ;;\nesac\n')
    askpass.chmod(0o700)
    # `git -c` (top level) resets any credential helper so ONLY the askpass
    # answers; it is never `git clone -c`, which would write .git/config.
    cloned = subprocess.run(
        ['git', '-c', 'credential.helper=', 'clone', '--quiet', '--depth', '1',
         '--branch', EXPECTED_TAG, REPO_URL, str(clone)],
        env=dict(os.environ, GH_TOKEN=token, GIT_ASKPASS=str(askpass),
                 GIT_TERMINAL_PROMPT='0'),
        capture_output=True, text=True)
    if cloned.returncode != 0:
        message = (cloned.stdout + cloned.stderr).replace(token, '<token>')
        del token
        raise SystemExit(
            f'GIT CLONE FAILED (return code {cloned.returncode}) for tag '
            f'{EXPECTED_TAG} of {REPO_URL} -- refusing.\n{message}'
            'A missing tag means this notebook pins a release that was never pushed;\n'
            'an authentication error means the token is wrong, expired or lacks\n'
            "Contents read access. Fallback: SOURCE = 'zip'."
        )
    del token
    manifest_text = (clone / 'tools' / 'manifest.txt').read_text()
    actual_digest = repo_digest(clone, manifest_text)
    if actual_digest != EXPECTED_REPO_DIGEST:
        raise SystemExit(
            'WRONG REPO CONTENT -- refusing to spend a session on unverified code.\n'
            f'  clone of {EXPECTED_TAG}: {actual_digest}\n'
            f'  expected        : {EXPECTED_REPO_DIGEST}\n'
            'The tag was moved or the repo differs from the release this notebook pins.'
        )
    print(f'source=github tag={EXPECTED_TAG} repo_digest={actual_digest} verified')
    # THE PACKAGE, BUILT FROM THE VERIFIED CLONE: exactly the manifest files plus
    # REPO_STAMP.json ({tag, repo_digest, manifest, function}). The runner
    # recomputes the digest over what it unpacks with the function and manifest
    # carried here, so a file changed after this verify is refused (L-G2). Built
    # in github_build/, NOT /content/: the runner stages a copy to
    # /content/<zip name>, and the same path would be a self-copy.
    _cells = json.loads((clone / 'notebooks' / 'AM2020_main_case_colab.ipynb').read_text())['cells']
    _block = re.search(r'^# >>> repo_digest\n.*?^# <<< repo_digest\n',
                       '\n'.join(''.join(c['source']) for c in _cells), re.M | re.S)
    if _block is None:
        raise SystemExit('the cloned notebook carries no repo_digest block -- refusing')
    GITHUB_BUILD.mkdir(parents=True)
    package = GITHUB_BUILD / 'AM2020_main_case_colab_package.zip'
    with zipfile.ZipFile(package, 'w', zipfile.ZIP_DEFLATED) as archive:
        for rel in sorted(manifest_entries(manifest_text)):
            archive.write(clone / rel, rel)
        archive.writestr('REPO_STAMP.json', json.dumps(
            {'tag': EXPECTED_TAG, 'repo_digest': actual_digest,
             'manifest': manifest_text, 'function': _block.group(0)}, indent=1) + '\n')
    print(f'package built from the clone: {package} '
          f'({len(manifest_entries(manifest_text))} files + REPO_STAMP.json)')
    runner_args = ['--package', str(package), '--repo-digest', EXPECTED_REPO_DIGEST]
else:
    raise SystemExit(f"SOURCE must be 'github' or 'zip', not {SOURCE!r}")
runner = CONTENT / 'run_main_case.py'
member = 'cases/AM2020_planar_mural_detachment/scripts/run_main_case_colab.py'
with zipfile.ZipFile(package) as archive:
    runner.write_bytes(archive.read(member))
# FIVE CHILDREN, ~7.3 h: HOW MUCH OF THE RESPONSE IS THE CORE/SHELL CONTRAST?
# Series `shell_modulus_sweep`, 30 ms. Requested by Dr He Li, 2026-09-01.
#
#   g04  G_shell  4 Pa  -- THE CONTROL, runs first (source 33's value)
#   g40  G_shell 40 Pa  -- the homogeneous endpoint, runs second
#   g20  G_shell 20 Pa  -- the midpoint
#   g10  G_shell 10 Pa
#   g30  G_shell 30 Pa
#
# Every child is run 20260828-040652's cf_xu with TWO keys changed, G_pa and
# end_time_s (50 -> 30 ms); g04 changes only the window. VERIFIED at build time
# by generating all 119 children: g04 differs from cf_xu in exactly one solver
# input (controlDict endTime), g40 differs from g04 in exactly one
# (phaseClotProperties G), and Gcore 40.0 / k0p 1e-11 / k0pcore 1e-14 are
# identical in all five. Brinkman ON, bond hold OFF, bond unbreakable at 1e9 Pa,
# growth multiplier 1000.
#
# THE QUESTION. Dr Li asked to hold the core at 40 Pa and raise the SHELL
# through 10, 20, 30 and 40. At 40 the shell MEETS the core and the clot becomes
# one material, so the sweep runs from source 33's structured clot to a
# stiffness-homogeneous one. THE g04-TO-g40 DIFFERENCE IS THE ENTIRE
# CONTRIBUTION OF THE CORE/SHELL STIFFNESS STRUCTURE. That is the number worth
# sending him, and it is not one the request asked for directly.
#
# HIS WORDING, AND THE READING BUILT HERE. The email says "the shear modulus of
# the core can increase from 4 to 10, 20, 30 and 40 Pa", but 4 Pa is the SHELL
# value -- the core is 40 -- and the same sentence says to keep the core fixed.
# Built as a SHELL sweep, the only reading consistent with both halves. IF HE
# MEANT THE CORE, EVERY CHILD HERE IS WRONG, and that rebuild is not free: dt
# goes as 1/sqrt(G), so a core sweep upward gets progressively dearer.
#
# WHY THIS SWEEP IS FREE. maxDeltaT() bounds dt on gMax(GField_), the maximum
# modulus over the WHOLE field, which is the CORE's 40 Pa in every child:
# dt = 0.5*7.99e-7/sqrt(40/1050) = 2.045e-06 s, and cf_xu measured
# 50 ms/24450 steps = 2.045e-06 s exactly. Raising the SHELL never exceeds 40,
# so gMax does not move and no child is dearer than the control.
#
# PRE-REGISTERED. g04 MUST reproduce cf_xu step for step to 30 ms -- same dt,
# same step count, same maxHeight and cellsOverInjury series, same traction at
# 30 ms. This is bk_ctl's role in eulerian_brinkman, where it reproduced
# gr_hold to every printed digit and thereby made the rest of the series
# readable. IF g04 DOES NOT REPRODUCE cf_xu, STOP: the other four would be
# measuring an unknown change as well as the intended one.
#
# Three outcomes for the sweep itself, all informative:
#  (A) STEEP. Flattening falls monotonically as the shell stiffens and g40
#      holds a visibly taller clot than g04. Shell stiffness governs it.
#  (B) FLAT. g10 looks like g04 and so does g40. Then the flattening is set by
#      the flow and the permeability, not the modulus. THAT IS THE ANSWER, not
#      a failed series -- report it as such.
#  (C) SATURATING. The curve moves between 4 and 10-20 Pa then stops. There is
#      a stiffness above which the shell no longer yields to this flow, which
#      is more useful than either extreme.
#
# WHAT IT CANNOT ANSWER. NOTHING DETACHES IN ANY CHILD: the bond is unbreakable
# at 1e9 Pa throughout and cf_xu's core moved 0.08 um in 50 ms. This measures
# DEFORMATION against shell stiffness -- maxHeight, cellsOverInjury, clot_be_xy,
# peak_traction_final. A null detachment finding is by construction, not by
# discovery. Nothing about permeability either: k0p is fixed at 1e-11/1e-14 and
# the permeability sweep is a SEPARATE series, because Dr Li asked for the two
# independently and moving both at once makes neither attributable. And g40 is
# stiffness-homogeneous, NOT homogeneous -- the permeability contrast is
# untouched. Do not describe it as the latter.
#
# INSTRUMENT TRAPS. `coverage` is NOT growth: it runs 0.160 -> 0.982 within
# 4 ms and DECLINES from there, so the 0.160 is an initialisation transient and
# quoting initial against final reads as growth. That error reached a draft
# email to Dr Li on 2026-08-28. Use peak_traction_final, not peakTractionAny.
# clotVolume halved in cf_xu's last 5 ms (1.359e-15 at 45 ms -> 6.485e-16 at 50)
# with the attached clot's xMax pinned at 94.6 um -- NOT outflow, and still
# unexplained; the 30 ms window sits before it but do not assume it cannot
# appear earlier in a child whose shell holds together differently.
#
# COST: 14670 steps per child. Run 20260902-101600 MEASURED 0.418 s/step
# (6136 s / 14670 steps), revising the 0.3575 recorded here = 1.70 h each,
# ~7.3 h for the five -- inside the 8.2 h session 20260827-162326 returned
# complete, but only just. If the session is cut, the run order returns
# g04/g40/g20 first, which is a readable three-point curve on its own.
#
# PARAVIEW: colour by `c` and OVERLAY a `shell = 0.5` contour for the core. In
# a plain `c` field core and shell are BOTH c < 0 and both render deep blue. Do
# NOT colour by poroPerm -- and note it carries NO information in this series
# at all, since only G moves and it is identical in all five children.

# ------------------------------------------------------------------------
# BATCHES, SEGREGATED BY MECHANICS PATH. Read this before picking one.
#
# THE MATRIX IS NOT ONE MODEL. 95 of the 119 children set `eulerianSolid yes`;
# the other 24 set `eulerianSolid no` and run the PARTITIONED FSI path -- the
# staggered loop with outer-corrector iterations, and the one carrying the
# measured 38% first-order splitting error. The FSI children are the whole of
# `growth`, `required_strength`, `deformation`, `cohesive_tear`,
# `demonstration`, `source33_replication`, plus one deliberate comparison
# child in each of `eulerian_diag_xu`, `eulerian_stage3_li`, `eulerian_stage3_xu`.
#
# An earlier version of this list was ordered by COST and by "never run",
# which front-loaded the FSI path -- 16 FSI children first and an 8.5 h FSI
# child second -- before a single Eulerian child had been re-run on the fixed
# build. That was wrong ordering, not a wrong list.
#
# Batches are now EULERIAN FIRST, FSI last, and each carries its path.
# Batch 0 is the shell-modulus sweep: the direct re-run of run 20260902-101600
# on a build with the shared operator, i.e. the one that turns the closure
# result from ~1% into exact.
#
# THE FSI BATCHES ARE STILL WORTH RUNNING -- 18 of those children have NEVER
# executed and cost 0.1 h between them (fixed dt, 30-40 steps). But they sit at
# a material Courant of ~644 (dt 0.02) and ~32000 (dt 1.0), scaled from a
# MEASURED 0.0658 at dt 2.045e-06, with `adjustTimeStep no`. Everything this
# project has validated ran at Co <= 0.066. Treat that batch as a probe of
# whether those configurations are viable at all, not as a result.
#
# The full matrix is 76.8 h at 0.418 s/step (measured, run 20260902-101600:
# g04, 14670 steps, 6136 s). That is 3.2 days -- not one Colab session.
#
# (child ids, estimated hours, mechanics path)
BATCHES = {
    0: ("g04 g40 g20 g10 g30", 8.52, 'eulerian'),
    1: ("pp_p11 pp_hold pp_p15", 8.52, 'eulerian'),
    2: ("cf_soft cf_xu st_xu st_ctl", 8.40, 'eulerian'),
    3: ("dr_off dr_on ad_d20 ad_d16 ad_d12 gr_hold", 8.18, 'eulerian'),
    4: ("gr_p30 gr_p15 bk_ctl bk_on bk_both bk_soft", 8.18, 'eulerian'),
    5: ("sat_G4 sat_G4grow fx_ren_long fx_upw_long drift_long bond_sat an_all an_break an_ctl an_hold", 8.80, 'eulerian'),
    6: ("an_pen an_sup05 an_sup09 sd_ctl sd_deep sd_taper sat_G40 bt_ctl bt_dt4 bt_lin bt_renorm bond_see fx_dt fx_upw be_k02 be_k1 be_linear diag_xu_base diag_xu_dt4 diag_xu_nogrowth diag_xu_sweeps10 fix_xu_both fix_xu_devl fix_xu_outer5 flux_new flux_old force_cell force_face gate_e_long grad_new grad_old mom_Ghi mom_Glo", 9.00, 'eulerian'),
    7: ("mom_eul mom_nogrowth plt_audit plt_ctl plt_g100 plt_qfloor plt_vnoff renorm_off renorm_on diag_li_base fix_li_both stage3_xu_eul stage3_li_eul gate_a_advect gate_a_content gate_a_exact gate_a_floor gate_a_linear gate_a_nodiff gate_a_tol gate_e_content gate_e_vn anchor_Ghi anchor_Glo anchor_Gmid anchor_base trace_Ghi trace_base", 2.93, 'eulerian'),
    8: ("deform_s1000 deform_s2000 deform_s4000 deform_s8000 growth_shear050 growth_shear100 growth_shear200 req_s050_G0059 req_s100_G0059 req_s100_G1000 req_s100_G11600 req_s200_G0059 req_s200_G1000 req_s200_G11600 req_s200_kHigh req_s200_kLow stage3_li_off stage3_xu_off", 0.14, 'FSI -- never run, Co ~ 644 to 32000'),
    9: ("demo_full_release", 8.52, 'FSI'),
    10: ("tear_gc1em8 tear_gc1em7 xu_core_shell_growth tear_gc1em9 mom_part", 5.56, 'FSI'),
    # BATCH 11 -- THE OCCLUSION RUN. Dr Li, 2026-09-01: connect the channel
    # to a narrower one so a clot can block its entry, driven by PRESSURE
    # rather than a fixed inlet velocity. He is right about the BC: with a
    # velocity inlet the flow rate cannot fall and the effect is unobservable
    # by construction.
    #
    # NOT A DETACHMENT RUN. The mechanism is the one MEASURED in Batch 0: the
    # core centroid moves 75.00 -> 75.99 um in 30 ms while the SHELL centroid
    # moves 75.00 -> 136.26 um and its front reaches 202.5 um. The shell
    # separates and migrates as a body 19-20 um tall, with min c = -0.9986
    # behind the front. That body is what meets the narrowing.
    #
    # th_hold  bond 1e9 Pa  -- released stays 0, so the shell keeps its full
    #                         Brinkman drag and CAN obstruct. The result.
    # th_break bond 0.8 Pa  -- the bond fails early, released -> 1,
    #                         brinkmanHold -> 0, and the freed clot should pass
    #                         with little drop. THE NULL IS THE POINT.
    #
    # RAN 2026-09-03 AND DID NOT OCCLUDE -- superseded by the MEASUREMENT.
    # The clot reached the throat and deposited a 2.0-2.6 um film against a
    # Brinkman screening length sqrt(1e-11) = 3.16 um, so it was hydraulically
    # close to invisible: 22.5% occlusion against the open-channel reference,
    # not the ~32% predicted here. The raw flux ROSE 24%, because the clot
    # vacated the wide section (5.4% of the resistance) faster than it filled
    # the throat (70.7%). See STAGE 6.69; batches 12 and 13 are the response. Validated against Batch 0 rather than asserted -- an impermeable
    # clot would raise dp to 7x the empty channel and g04 measured 2.0-3.0x, so
    # flow demonstrably passes THROUGH the clot. This is the same knob as Dr
    # Li's permeability request: the two halves of his email are one question.
    #
    # GEOMETRY: wide 0-120 | cosine taper 120-220 | throat 220-320 um at 20 um.
    # The taper is at 120 because the window sets the geometry -- 30 ms buys
    # only ~106 um of shell travel, so a taper at 200 um is never reached. The
    # cost is that the clot sits 22 um from the narrowing at t=0, so the
    # "before" state is NOT a clean unobstructed channel. Say so; do not let it
    # read as a baseline.
    #
    # THE MESH IS BENT AFTER blockMesh by apply_throat.py, scaling y by H(x)/H
    # with a raised-cosine profile that has ZERO SLOPE at both joins. A
    # piecewise-linear taper would put a salient corner at each join, in the
    # one place the measurement is taken. Verified offline by
    # check_throat_mesh.py: 0 of 31360 cells inverted, wall error 4.9e-17 m,
    # max cell skew 36.1 deg, clot region exactly untouched. The case ships
    # nNonOrthogonalCorrectors 2 -- every other child runs 0 on an orthogonal
    # block, which would be a divergence risk on a bent one.
    #
    # COST: dt = 1.5082e-06 s, set by the CUBE ROOT OF MIN CELL VOLUME (the
    # throat's dy is 0.2857 um), so 19891 steps for 30 ms. 2.53 h per child at
    # 0.418 s/step scaled to 31360 cells; ~5.1 h serial, ~3 h at CONCURRENCY 2.
    #
    # READ FIRST: `occlusion` in result.json -- outlet_flux_m3_s against time is
    # the velocity drop, and released_max_over_run is a TRIPWIRE. In th_hold it
    # must stay 0; if the stretching shell pinches off, brinkmanHold goes to
    # zero and the run has silently stopped answering the question.
    11: ("th_hold th_break", 5.06, 'eulerian, pressure-driven, THROAT'),
    # BATCH 12 -- THE LODGED ARM. Run 20260903-114913 measured that the journey
    # and the block cannot be had at one permeability: transport inside the clot
    # is Darcy, U = (k/nu)*|grad p| (measured 2.132e-03 m/s against a predicted
    # 2.036e-03, 4.7%), so travel goes AS k -- 95 um in 29 ms at 1e-11, 9.5 um
    # at 1e-12. Blocking needs the opposite: the deposit must be thick against
    # sqrt(k), and at 1e-11 sqrt(k) = 3.16 um exceeds the 2.0-2.6 um deposited.
    #
    # So this arm PLACES the fragment where a detached one would lodge -- a cap
    # filling half the 20 um throat at x = 225-275 um -- and sweeps k. It
    # answers the hydraulic half on its own, and it is also the permeability
    # half of Dr Li's 2026-08-25 request in the one setting where it has a
    # visible consequence.
    #
    # THIS IS A PLACEMENT, NOT A PREDICTION. Never report it as though a clot
    # travelled there; run 20260903-114913 is the honest record of what the
    # journey does. The bond is unbreakable because this model has no CONTACT
    # mechanics -- a real lodged embolus is held by the throat walls, and the
    # bond is the stand-in. Say that to Dr Li.
    #
    # PREDICTED BEFORE THE RUN, from a 1D Brinkman model validated against
    # 20260903-114913's own written fields to 3.3-4.0%, computed for THIS
    # geometry (a 10 um cap over 50 um of the 100 um throat):
    #   Q/Q_open = 0.666 (1e-11), 0.394 (1e-12), 0.315 (1e-13)
    #   i.e. 33.4% / 60.6% / 68.5% occlusion.
    # An earlier draft said 0.438 / 0.208 / 0.159; that was a layer over the
    # whole 200 um taper-plus-throat, not the 50 um actually placed, and is
    # WITHDRAWN. Noted rather than silently replaced.
    #
    # Growth thickens the cap and the assessor MEASURES the fill: at 12 um
    # the three become 0.573 / 0.265 / 0.186, at 14 um 0.498 / 0.166 / 0.091.
    # So a result below prediction is most likely growth, and the measured
    # fill fraction settles which. Lubrication reads ~3.5% high, so measured
    # values should sit slightly BELOW the number for the measured fill.
    #
    # THE FRAGMENT WASHES OUT, AND THAT IS HALF THE RESULT. It is bonded at
    # its base but the material above the bond is advected at the Darcy
    # velocity like any other clot. U = (k/nu)*|grad p| over the 10 ms window
    # gives 114 um of travel at 1e-11, 14.9 at 1e-12, 1.6 at 1e-13. The cap
    # is 50 um long, so lo_k11 should LEAVE the throat while lo_k13 stays.
    # So this series measures two things at once -- how much a lodged
    # fragment blocks, and whether it can stay lodged -- and BOTH go as k in
    # the same direction. That is why permeability is the governing
    # parameter for occlusion rather than a secondary one, and it is the
    # cleanest answer available to Dr Li's permeability question.
    #
    # The predicted Q/Q_open above are for the cap AS PLACED, so they are the
    # right comparison for the EARLY samples. Write interval is 0.2 ms (50
    # samples) precisely so the clean initial state is not smeared into the
    # washout: at 1e-11 the cap moves 1.14 um every 0.1 ms.
    #
    # READ FIRST: occlusion.flux_fraction_of_open_final in result.json, and
    # the per-time series next to max_blockage_fraction. The old
    # flux_drop_fraction is kept but is NOT the number -- it references the
    # first sample and read 0.0 on a run with a real 22.5% occlusion.
    #
    # COST, CORRECTED 2026-09-04 FROM THE RUNNING LOG. dt here is NOT the
    # 1.5083e-6 s of the throat_occlusion children. maxDeltaT's elastic
    # Courant uses gMax(GField_), and these children WAIVE THE CORE, so the
    # largest modulus is the shell's 4 Pa, not the core's 40. dt goes as
    # 1/sqrt(G): predicted 4.7695e-6 s, logged 4.7619e-6. So ~2100 steps,
    # ~0.5 h/child at the observed 0.85 s/step -- 3.16x cheaper than the
    # 1.9 h this line used to claim.
    12: ("lo_k11 lo_k12 lo_k13", 1.1, 'eulerian, pressure-driven, THROAT, lodged'),
    # BATCH 13 -- THE COHERENCE TEST, and it is FREE. th_hold with the SHELL
    # modulus 4 -> 40 Pa and nothing else changed. Run 20260903-114913 measured
    # the cause of the thin deposit: the taper extrudes the body, which is
    # 16.22 um tall at x = 150 um and 1.77 um by x = 210 um, under tau =
    # rho*nu*gammadot = 13.7 Pa against G = 4 Pa. tau/G = 3.4, so it is drawn
    # out like a fluid. At 40 Pa, tau/G = 0.34.
    #
    # FREE because maxDeltaT's elastic Courant uses gMax(GField_) over the
    # WHOLE field, and GfieldMax was ALREADY 39.997 from the core. dt stays
    # 1.5083e-6 s and the step count is th_hold's exactly. 40 Pa is the top of
    # Dr Li's 2026-08-25 list, so this is also his stiffness request.
    #
    # Run it ALONE at CONCURRENCY 1: ~3.3 h, rather than the 5.6 h two children
    # of this length cost each other.
    #
    # WHAT WOULD SETTLE IT: the deposit thickness in the throat. 10 um puts
    # Q/Q_open at 0.44, 18 um at 0.22. If it stays near 2 um the extrusion is
    # not stiffness-limited and Batch 12 is the only route at Xu's k.
    13: ("th_G40", 3.3, 'eulerian, pressure-driven, THROAT'),
    # BATCH 14 -- THE EMBOLUS. The only batch that depends on the solver
    # change of 2026-09-04, and the only one that can show the thing Dr Li
    # actually asked for: a DETACHED clot blocking the vessel.
    #
    # Until that change the Brinkman drag was gated as
    # fvm::Sp(dragCoeff*(1 - released), U), so released material felt ZERO
    # drag. A detached embolus was hydraulically invisible and "moves" and
    # "blocks" were mutually exclusive at every parameter. The drag is now
    # the relative-velocity pair -(mu/k)(u_f - u_s) -- the same term the
    # skeleton path already used -- so a moving body still resists flow.
    #
    # HOW THE SHELL LEAVES AND THE CORE STAYS: no new code. Support is
    # seeded at INTACT bonds and flooded through clot; releaseCoreExempt
    # holds the core back. Once EVERY bond fails there is no seed, so the
    # shell releases and the core is exempt. th_break stalled at 0.8 Pa --
    # 40 of 56 faces, survivors at 0.534 Pa peak traction -- so the bond is
    # 0.4 Pa here, below that and above the 0.304 Pa seen at 0.9 ms.
    #
    # em_free     G = 4 Pa,  source 33 shell value, the reference
    # em_freeG40  G = 40 Pa, less extruded BEFORE it lets go. Confounded and
    #             said so: a stiffer shell also raises wall traction, so it
    #             may debond sooner as well.
    #
    # READ released_cells_max FIRST. If it stays 0 the bond stalled again
    # and neither child says anything about the drag change. Only then read
    # occlusion.flux_fraction_of_open_final.
    #
    # TWO WAYS IT CAN FAIL, both informative. (1) A released body is
    # advected RIGIDLY at one Ubody, so a ~34.5 um body cannot deform into a
    # 20 um throat: it either jams at the entry -- which IS the result -- or
    # pushes material at a wall. The wall half was closed on 2026-09-04 by
    # zeroing the rigid-body mask on wall patches, because the taper top
    # wall slopes up to 25.2 deg and Ubody & Sf is NOT zero there. Watch
    # PHASE_CLOT_PHASE_BOUNDS anyway. (2) The bond may stall again.
    #
    # COST: 19890 steps, ~5.6 h for the pair at CONCURRENCY 2.
    14: ("em_free em_freeG40", 5.6, 'eulerian, pressure-driven, THROAT, embolus'),
    # BATCH 15 -- THE COUPLED TWO-VELOCITY SKELETON. A controlled A/B
    # against em_free: one key differs, two_velocity_skeleton.
    #
    # THIS NOTE WAS REWRITTEN 2026-09-05. It described an ALGEBRAIC closure
    # that has since been DELETED, and running against a stale description
    # is how a wrong result gets believed.
    #
    # WHY. Run 20260904-093228 measured that the clot is advected at the
    # DARCY SEEPAGE velocity -- median |U|/(|grad p|/clotDrag) = 1.081 over
    # 4072 clot cells. That is fluid percolating THROUGH the skeleton,
    # mistaken for the skeleton moving. The wall layer that looks frozen is
    # the part LEAST wrongly advected, not the part most wrongly held.
    #
    # WHAT WAS TRIED AND FAILED. Run 20260904-214345 solved the fluid
    # equation and EVALUATED a pointwise skeleton balance for Vs. It crashed,
    # and 88.3% of clot cells had |Vs| > |U| -- a skeleton outrunning the
    # fluid that drags it, which the fluid equation alone forbids, since it
    # gives u - v = -phi_f*grad(p)/K, downstream unconditionally. The cause
    # was structural: p was solved from FLUID continuity, so nothing could
    # force grad(p) consistent with div(sigma_e), and the mixture-balance
    # residual landed in Vs. That closure is deleted, not disabled.
    #
    # WHAT RUNS NOW. Vs is the unknown of a skeleton momentum equation
    # coupled to U by a per-cell 2x2 block inversion, with the pressure
    # equation assembled from BOTH momenta and enforcing MIXTURE continuity
    # div(phi_f*U + phi_s*Vs) = 0. The drag is in NEITHER matrix -- it is the
    # block off-diagonal, i.e. Spalding partial elimination, which Rusche
    # (2002 sec 3.2) reports is the only treatment that converges "when the
    # drag multiplier is high or very non-uniform". Ours is both: 3.3e+05 in
    # the shell to 3.3e+08 in the core. Sources 47-51 and OpenFOAM-13's own
    # multiphaseEuler independently require the same three things.
    #
    # phi_s = chi * skeletonVolumeFraction, default 0.01. chi = 1-q is an
    # INDICATOR, not a volume fraction -- median q over clot cells is 0.0066
    # -- so phi_f = q would have left NO BLOOD inside the clot. 0.01 is
    # anchored to Wufsus via He/Kim/Ku (1% platelet volume fraction at
    # k = 1.1e-14 m2, against k0pcore = 1e-14). IT IS THE LARGEST
    # UNCERTAINTY IN THIS RUN and is a dictionary entry, so sweep it.
    #
    # READ THE GATE FIRST, AND YOU CAN READ IT IN THE FIRST TEN MINUTES.
    # PHASE_CLOT_SKELETON_VELOCITY prints every step. exceedFractionMargin
    # (|Vs| > 1.1|U|) must fall below 0.01; meanVsOverU should sit at or just
    # below 1. If exceedFractionMargin is still large after a few hundred
    # steps, KILL THE RUN -- the coupling is wrong and nothing else in the
    # run means anything. Do not spend 5.6 h to learn that.
    #
    # NOT the bare exceedFraction: a free element genuinely runs slightly
    # fast or slow depending on deltaT (the two slip terms balance at
    # deltaT* = U/|grad p| = 5.9e-06 s), so a bare count sits on a sign
    # change and is flaky. The margin count is the gate.
    #
    # ONE DELIBERATE BEHAVIOUR CHANGE BEYOND THE SWITCH: the bond-hold
    # anchor now acts on Vs, not U. On the one-velocity path it had to
    # resist FLOW because U was the only velocity; it now holds the
    # MATERIAL. Bonded material staying put is an expected result here and
    # was not achievable before. If it still slides the anchor is too weak
    # -- do NOT return to a displacement skeleton.
    #
    # WATCH FOR: outer correctors failing to converge, which would mean
    # sigma_e must be linearised in the strain rate rather than left
    # explicit. That is the recorded next step, not a surprise.
    #
    # CONCURRENCY 1 so it is not slowed by a neighbour: ~5.6 h.
    15: ("em_2vel", 5.6, 'eulerian, pressure-driven, THROAT, two-velocity'),
    # BATCH 16 -- THE SAME CHILD AGAIN, ON A SOLVER THAT CONSERVES THE CLOT.
    # Identical child.json to Batch 15. Every difference is in the solver.
    #
    # WHY BATCH 15 IS NOT AN ANSWER. It grew the clot 4.0x: 2620 -> 18797 clot
    # cells, int chi dV 1.35e-15 -> 5.36e-15 in 14 ms, with c never leaving
    # [-1,1] -- so cells were CONVERTED, not blown up. PHASE_CLOT_MASS had been
    # measuring it every step for 9391 steps and no gate read that line. The
    # physical growth source was 3% of dV/dt; clotResidual was 98%.
    #
    # THE CAUSE, and it is one line of algebra. With `bounded` convection the
    # solved equation is ddt(c) + [div(phi c)]_scheme - c*SI(carrier) = S, and
    # working the budget against it leaves
    #     clotResidual == -0.5 * int c * div(carrier) dV
    # which is zero for a solenoidal carrier and ONLY for one. phi_ is
    # solenoidal; phiSkelOwn_ is not and CANNOT be -- Cogan & Guy 2010, after
    # their Eq. 7: the individual phase velocities of a mixture "are not
    # divergence-free". Measured drift, every run on disk:
    #     phi_                      +0.00031      (g04..g40)
    #     phi_*(1-bondHold)         +0.0048       (em_free)
    #     phiSkelOwn_*(1-bondHold)  +1.12, +3.21  (em_2vel 09-04, 09-05)
    #
    # NOT the pressure solve: sum local continuity error was flat at 1.3e-14
    # from the first step to the last. The 2x2 block and the shared pressure
    # were doing their job the whole time.
    #
    # WHAT CHANGED IN THE SOLVER.
    #  1. c is transported CONSERVATIVELY -- phaseConservativeConvection()
    #     (the bounded Sp dropped) plus carrierDivergence() on the right.
    #     Together that is Cogan & Guy Eq. 21.
    #  2. Wp gets the IDENTICAL treatment. Not optional: because the carrier
    #     is not solenoidal, R[1] = SI != 0, so leaving Wp bounded would break
    #     Wp = q by chi*SI. With both conservative + SI the invariant reduces
    #     to the stoichiometric closure the solver already reports as
    #     VnClosureResidual = 0. advectMaterialLabel keeps the bounded form --
    #     shell_ is a label, not a content.
    #  3. Both-sides diffusion on the skeleton momentum,
    #     Gamma = phi_s*gD*G*dt/rho. Derived, no new parameter, the halves
    #     cancel at convergence. Batch 15 died at t=0.0142 to the explicit
    #     elastic CFL -- shell 1.31x over, core 4.10x over, predicted growth
    #     1.31/step against 1.19 observed. Respecting it instead costs ~45 h.
    #  4. bondHoldCarrier FORCED OFF here -- the anchor is already on Vs, and
    #     masking a flux destroys its solenoidality. Look for
    #     PHASE_CLOT_CONFIG bondHoldCarrier=forced-off in the first lines.
    #  5. The mass budget re-derived to match: clotResidual drops the
    #     -0.5*clotCarrierNetFlux term. Without this the new gate would be
    #     UNPASSABLE -- a correctly conserving run would have read ~4% drift
    #     against a 5% limit.
    #
    # READ THIS FIRST, AND YOU CAN READ IT IN THE FIRST TEN MINUTES.
    # PHASE_CLOT_MASS prints every step. clotVolume must stay near its initial
    # 1.348e-15 -- the one-velocity children all finish at 1.10x. If clotVolume
    # is climbing past ~1.5x after a few thousand steps, KILL THE RUN: the
    # transport is still wrong and nothing downstream means anything.
    # Then PHASE_CLOT_SKELETON_VELOCITY exceedFractionMargin < 0.01, as before.
    #
    # THE GATES THAT NOW EXIST. assess_case.py fails a child on cumulative
    # clot-volume drift > 5% (floor 1e-5, so 5000x margin) and on
    # max|Wp - q| > 0.25 (widest one-velocity run 0.111; em_2vel 09-04 hit
    # 0.9926). Retro-applied to every log on disk they pass all nine
    # one-velocity children and fail both two-velocity runs.
    #
    # THIS IS THE FIRST MAIN-CASE PACKAGE IN A WHILE BUILT THROUGH THE FULL
    # GATE SUITE. The Batch 15 zip fails check_fvschemes_coverage --self-test
    # on two unnamed divs, which that script runs FIRST under -e, so it cannot
    # have been built by build_main_case_colab_package.sh. Both are now named
    # explicitly (a runtime no-op) and the suite passes end to end.
    #
    # WHAT IT DECIDES. If clotVolume holds near 1.1x AND it reaches t=0.03,
    # the two-fluid layer is settled and the tear/clog work can start on it.
    # If volume conserves but the clot goes locally dense or sparse, that is
    # the signal that chi must become a genuine volume fraction, with a
    # network pressure and xi(phi_s) -- Du & Fogelson Eq. 2.25 after Wufsus.
    # Deliberately NOT built yet, pending this measurement.
    #
    # CONCURRENCY 1: ~5.6 h.
    16: ("em_2vel", 5.6, 'eulerian, pressure-driven, THROAT, two-velocity, conservative'),
    # BATCH 17 -- THE SKELETON GETS A VOLUMETRIC RESPONSE. Same child again.
    # Every difference is in the solver and in one dictionary key.
    #
    # BATCH 16 FIXED WHAT IT WAS BUILT TO FIX. Conservative transport of c
    # (Cogan & Guy Eq. 21) took the clot volume ratio from 4.00x to 0.931x and
    # the cumulative drift from +3.21451 to -0.000024 -- the discretisation
    # floor is 1e-05. The conservation gate PASSES. Both-sides diffusion held
    # too: deltaT flat at 1.506e-06, no elastic-CFL runaway, maxVsOverU 1.6-3.0
    # instead of doubling every three steps. meanVsOverU 0.94-0.96 and
    # exceedFractionMargin 0.013-0.028 against 0.144.
    #
    # AND THEN IT COMPACTED. c left [-1, 1] at STEP 4 and reached -2.74. 988
    # cells (3.15%) below -1, at x = 55-112 um and y = 0.4-29 um -- the CLOT
    # BODY, only 4% of them near the wall and none on the interface. chi
    # reached 1.87, so phi_s NEARLY DOUBLED. cMax stayed exactly 1.0: only the
    # solid end is unbounded, because the SI*(1-c) source vanishes identically
    # at c = +1. SIGFPE in solvePhaseField at t = 0.0028.
    #
    # WHY. elasticStress() was dev(be) ONLY, and `be` is the elastic left
    # Cauchy-Green of an INCOMPRESSIBLE neo-Hookean with det(be) = 1 enforced
    # -- there was no bulk modulus anywhere in the constitutive model. Nothing
    # could object to the network densifying. Both-sides diffusion cannot
    # supply one: its halves cancel at convergence by construction, so it damps
    # and cannot restore. Du & Fogelson Sec. 2.3 describe this exact failure in
    # the opposite sign -- a network velocity that is "in general, not
    # incompressible" would otherwise "collapse an aggregate to a point".
    # Batch 15 dilated from wrong TRANSPORT; Batch 16 compacted from an
    # incomplete CONSTITUTIVE MODEL.
    #
    # WHAT CHANGED.
    #  1. NETWORK PRESSURE, Cogan & Guy Eq. 19:
    #        p_n = K_n*(phi_s - phi_s0)/(phi_s_max - phi_s)
    #     Zero at nominal density, so 0.01 is an EQUILIBRIUM rather than a
    #     starting value that drifts. Divergent at phi_s_max = 0.6 (Du &
    #     Fogelson Eq. 2.25, after Wufsus 2013), so close packing is
    #     unreachable by force balance rather than by clamping a field.
    #     NEGATIVE below nominal, so it resists dilation too -- Batch 15 and
    #     Batch 16 failed in opposite directions and a one-sided law would
    #     have fixed one and left the other.
    #     K_n = networkPressureCoeff * gD * G, default 1.0: NO NEW NUMBER. It
    #     reuses GField_, so the core/shell contrast is inherited, and the same
    #     gD as the deviatoric stress means damaged material loses bulk and
    #     shear stiffness together. Anchored, not fitted -- sweep it.
    #     Entered as -phi_s*p_n*I inside elasticStress(), so div(sigma) gives
    #     -grad(phi_s p_n) and it inherits the existing FACE-form treatment
    #     through correctPressureTwoVelocity with no new plumbing.
    #  2. permeabilityModel = kozenyCarmanSolid, on THIS CHILD ONLY. Porosity
    #     is phi_f = 1 - phi_s, not q. The old kozenyCarman fed it q -- the
    #     clot/blood INDICATOR, ~0 inside a clot -- so run 20260814-055141 ran
    #     all 168 bulk-clot cells at the numerical floor and a three-value k0p
    #     sweep was three runs of the same permeability. That branch is left
    #     untouched; this is a new name.
    #     AT NOMINAL DENSITY IT RETURNS k0pField EXACTLY, so it reproduces
    #     `constant` at the undeformed state and deviates only under
    #     compaction: k/k0p = 0.24 at phi_s = 0.02, 1.8e-05 at close packing.
    #     That is Du & Fogelson's xi(theta_b) in derived rather than fitted
    #     form. It is NOT a substitute for p_n: K multiplies (u_f - v_s), so a
    #     uniformly compacting network with u_f = v_s feels none of it.
    #  3. phi_s was ALREADY a genuine volume fraction -- conservative transport
    #     made int chi dV conserved and chi continuous in Batch 16. What it
    #     lacked was a constitutive law, which is 1 and 2.
    #
    # READ THIS FIRST, IN THE FIRST TEN MINUTES. Two lines, both every step:
    #   PHASE_CLOT_PHASE_BOUNDS cUnder  -- must stay at ~1e-13. Every run on
    #     disk except 20260905-113709 has it EXACTLY 0. If it passes 0.05 the
    #     network pressure is not holding and nothing downstream means
    #     anything: q = 0.5(c+1) goes negative, qEff clamps to qFloor, and
    #     w = Wp/qEff explodes (it hit -2.8e+05 last run). KILL IT.
    #   PHASE_CLOT_NETWORK phiSOverNominal -- 1.0 is equilibrium. This is the
    #     same failure in the variable the physics actually acts on; it read
    #     1.87 last run and there was no diagnostic to show it.
    # Then clotVolume near 1.348e-15 as before, and exceedFractionMargin.
    #
    # THREE GATES NOW, and they are not redundant -- each of the three
    # two-velocity runs on disk fails a DIFFERENT combination:
    #   conservation drift > 5%      (floor 1e-05)
    #   max|Wp - q| > 0.25           (widest 1-vel 0.111)
    #   cUnder/cOver > 0.05          (every healthy run 0 / 1.1e-13)
    # All nine one-velocity children pass all three.
    #
    # NOT CHANGED, DELIBERATELY: th_hold, th_break, em_free, em_freeG40 and
    # every g-child still generate permeabilityModel `constant` and no network
    # pressure keys. Verified by generating them. No existing result moves.
    #
    # WHAT IT DECIDES. If cUnder stays at round-off and clotVolume holds near
    # 1.348e-15 to t = 0.03, the two-fluid layer is DONE -- conservation,
    # coupling, CFL and constitutive response all closed -- and the tear/clog
    # work starts on a foundation that can be trusted. The remaining blockers
    # for detachment are then the all-or-nothing release criterion and
    # mobility = 0, neither of which touches the momentum formulation.
    #
    # CONCURRENCY 1: ~5.6 h.
    17: ("em_2vel", 5.6, 'eulerian, THROAT, two-velocity, conservative + network pressure'),

    # ------------------------------------------------------------------------
    # BATCH 18. THE SAME CHILD AGAIN. ONE EXPRESSION OF SOLVER CHANGED.
    #
    #   elasticStress(), two-velocity branch:
    #     -   phiS*networkPressure()*symmTensor::I
    #     +   solidFraction()*networkPressure()*symmTensor::I
    #
    # `phiS` there is a LOCAL variable, and it is not the volume fraction:
    #
    #     volScalarField phiS("elasticSolidWeight", chi);
    #     if (skeletonDiffuseDomain_) { phiS = max(chi, skeletonSolidFloor_); }
    #
    # -- the elastic DIFFUSE-DOMAIN WEIGHT, chi floored at 1e-04 so the elastic
    # operator does not vanish outside the clot. Correct for dev(be), which is
    # what it was written for. Wrong for p_n, which is a function OF the volume
    # fraction and must be weighted BY it. Two consequences, and the second is
    # the one that killed the run:
    #
    #   IN THE CLOT it was 100x too strong -- chi ~ 1 against phi_s = 0.01.
    #   IN THE LUMEN it was nonzero at all. networkPressure() is evaluated
    #     everywhere, so outside the clot it returns the constant
    #     -K_n*phi_s0/(phiSMax - phi_s0) = -0.0667 Pa, and max(chi, 1e-04)
    #     carried it into VsEqn. THE LUMEN IS THE WORST PLACE FOR A SPURIOUS
    #     FORCE: every term of VsEqn there carries phiSkel_ = 0, so
    #     VsEqn.A() = 0 and the ONLY diagonal is Kres = 66.7 1/s, against
    #     3.3e+05 of Darcy drag inside the clot. A tiny force, no stiffness,
    #     enormous velocity.
    #
    # HOW IT WAS LOCATED, because the first reading was wrong. Batch 17 grew
    # 1.21x/step near the end and Batch 15 had grown 1.19x/step from an elastic
    # CFL violation, so the CFL was the first answer. The matched-time
    # comparison against Batch 16 refuted it: whole-field maxVs was 0.0294
    # against 0.00061 -- 48x -- at t = 2e-05, the FIRST samples, while every
    # clot-cell metric was identical between the two runs. A CFL instability
    # does not start 48x high, and identical clot metrics put the difference
    # outside the clot. NOT the CFL, and not in the clot.
    #
    # WHAT BATCH 17 ESTABLISHED -- AND THE CLAIM THAT DID NOT SURVIVE.
    #
    # It was first written up here as "the network pressure works", on
    # phi_s/phi_s0 1.87 -> 1.0149 and cUnder 1.78 -> 0.0297 against Batch 16.
    # THAT COMPARED DIFFERENT TIMES AND IT IS WRONG. Batch 16's 1.87 and 1.78
    # are its values at t = 2.794e-03; Batch 17's are its values at
    # t = 2.214e-04, where it died -- 12.6x earlier. At MATCHED time the two
    # runs are indistinguishable:
    #
    #      t            B16 cUnder (no p_n)   B17 cUnder (p_n on)
    #      4.970e-05        0.001784              0.001782
    #      9.940e-05        0.007068              0.007062
    #      1.491e-04        0.01466               0.01464
    #      1.988e-04        0.02456               0.02452
    #      2.214e-04        0.0298                0.02974      <- B17 dies
    #
    # phi_s/phi_s0 is 1.0149 in BOTH at that time. THE NETWORK PRESSURE
    # CHANGED NOTHING. Batch 17 did not stop the compaction, it died before
    # the compaction arrived.
    #
    # WHY, and it is arithmetic not a bug. Against the deviatoric term
    # chi*gD*G*dev(be) the isotropic term phi_s*p_n has ratio
    #     0.01*(phi_s - phi_s0) / ((phi_s_max - phi_s)*|dev be|)
    # and G CANCELS -- it depends only on networkPressureCoeff, phi_s0 = 0.01
    # and phi_s_max = 0.6. Along Batch 16's own trajectory, using its logged
    # clotBeXYmax: 4.1e-05 at t = 2.2e-04, 1.4e-04 at 1.0e-03, 2.6e-04 at its
    # worst compaction. Parity needs phi_s = 0.59, i.e. 59x NOMINAL. With the
    # two anchors sixty apart the law is a CLOSE-PACKING BACKSTOP and nothing
    # else. Even Batch 17's 100x-too-strong chi weighting only reaches 2.6e-02
    # at peak compaction, and it never got near peak compaction to find out.
    #
    # WHAT DOES HOLD UP from Batch 17: conservation (drift +1e-06, clot volume
    # ratio 1.0005) and kozenyCarmanSolid (kClotMin/Max 1.0e-14 / 1.11e-11,
    # flooredCells = 0 -- three decades of real permeability instead of every
    # bulk-clot cell pinned to the floor).
    #
    # SO WHAT IS THIS BATCH FOR. It is a FALSIFICATION TEST, not an attempt to
    # succeed, and it is not a 5.6 h commitment. Pre-registered:
    #   * cUnder tracks Batch 16 step for step and crosses the 0.05 gate limit
    #     at t = 2.97e-04 -- Batch 16 step 197 of 1855, about 3 MINUTES of
    #     solver time. Compaction then runs to ~1.87x and the run dies near
    #     t = 2.8e-03, which cost Batch 16 1645 s. Budget ~30 min, not 5.6 h.
    #   * whole-field maxVs/maxU stays near Batch 16's 6.36 instead of 237.7.
    #     THAT IS THE ONLY THING THIS RUN ACTUALLY TESTS -- whether the lumen
    #     fix holds.
    # If cUnder departs from Batch 16's trajectory the analysis above is wrong
    # and that is worth 30 minutes. KILL IT once cUnder passes 0.05 and
    # maxVs/maxU has been read; there is nothing after that point this run can
    # tell you.
    #
    # A FOURTH GATE, AND WHY THREE WERE NOT ENOUGH. Batch 17 PASSED all three:
    # conservation, Wp-q and c-bounds. It passed them honestly -- the clot was
    # fine. meanVsOverU, maxVsOverU and exceedFractionMargin are computed over
    # CLOT CELLS ONLY, while maxVs is whole-field, so a lumen runaway is
    # STRUCTURALLY INVISIBLE to all three. New gate:
    #   VS_WHOLE_FIELD_LIMIT = 20   on peak maxVs/maxU
    # THE THRESHOLD IS NOT CALIBRATED AND SHOULD NOT BE READ AS IF IT WERE. No
    # two-velocity run has ever survived, so there is no healthy peak to
    # calibrate against; the four on disk read 2127, 27.7, 6.36 and 237.7. It
    # is a "pathological on any reading" bound, to be tightened the first time
    # something reaches 0.03.
    #
    # Four gates, four distinct signatures -- 191135 is caught by the new one
    # ALONE:
    #                 conserv   Wp-q   c-bounds   Vs-field
    #   214345          FAIL     FAIL    pass       FAIL
    #   054441          FAIL     pass    pass       FAIL
    #   113709          pass     FAIL    FAIL       pass
    #   191135          pass     pass    pass       FAIL
    #
    # READ THIS FIRST, IN THE FIRST TEN MINUTES.
    #  1. PHASE_CLOT_SKELETON_VELOCITY maxVs vs maxVsClot. Equal-ish is
    #     healthy. maxVs >> maxVsClot is the lumen again, and maxVsX/maxVsY
    #     say where. These four fields are new; the run before had no way to
    #     answer "where" at all.
    #  2. PHASE_CLOT_NETWORK pnTermMax/pnTermMin -- phi_s*p_n, the term as
    #     it enters the stress (pnMax/pnMin are the RAW law and are left
    #     unchanged, so pnMin still reads -0.0667 in the lumen; that is the
    #     law evaluated where there is no network, it is inert now, DO NOT
    #     CHASE IT -- I printed it every line last run and walked past it).
    #     pnTermMin is NOT expected to be zero: predicted -1.7e-04 Pa at
    #     G = 4 and -1.7e-03 Pa at G = 40, from the diffuse edge where
    #     phi_s < phi_s0. What these two are for is the MAGNITUDE question --
    #     compare them against clotSigmaXY, which reached 1.77 Pa in Batch 16.
    #  3. phiSOverNominal near 1.0 (1.0149 last run).
    #  4. cUnder at ~1e-13, clotVolume near 1.348e-15.
    #
    # WHAT IT DOES NOT DECIDE. It cannot close the two-fluid layer, because
    # the compaction is unopposed. The open problem after this run is the
    # volumetric response, and there are three routes, none anchored yet:
    # a stiffer networkPressureCoeff (parity at 1.5x nominal needs ~6000, which
    # is an admission that K_n = G was the wrong scale); a different
    # phi_s_max (0.01 is a FIBRIN fraction from Wufsus, 0.6 is a PLATELET
    # close-packing limit from Du & Fogelson -- mixing them may be the real
    # error, and that PDF is not in this repo to check); or the drag backstop,
    # kozenyCarmanSolid reaching k/k0p = 1.8e-05 at close packing, which acts
    # on u_f - v_s and is at least available since Batch 16 measured
    # maxVsOverU = 10.2 at death. A bulk modulus on be is NOT a fourth route:
    # be is isochoric by construction and beDetRenormalise exists because any
    # det(be) departure from 1 is advective mixing error, not volumetric
    # strain -- there is no trustworthy J for a K(J-1) term to act on.
    #
    # STILL THE ONE UNMEASURED ASSUMPTION: networkPressureCoeff = 1.0, i.e.
    # K_n = G. Not fitted to anything. First thing to sweep once a run lives.
    #
    # TWO CHILDREN, AND THE SECOND IS THE ONE THAT MIGHT ACTUALLY WORK.
    # Added 2026-09-05 after reading sources 47 and 52 in full (audit:
    # research/literature/AM.2020.0056-19/NETWORK_COMPACTION_AUDIT.md). Three
    # findings, and they change what is worth running:
    #
    #  1. phi_s_max = 0.6 IS correctly sourced -- Du & Fogelson Eq. 2.25,
    #     xi(theta_b) = xi_0 + 1e7/(1+exp(-60*(theta_b-0.6))), citing Wufsus --
    #     BUT IT IS A DRAG ANCHOR, NOT A PRESSURE ANCHOR. This project put it
    #     into a pressure law. The mechanism it belongs to is already here:
    #     kozenyCarmanSolid.
    #  2. Du & Fogelson Sec 2.3 names our exact failure ("would collapse an
    #     aggregate to a point") and fixes it with linear springs of NONZERO
    #     REST LENGTH R (Eq. 2.17). The prefactor 1 - R/<|y|> vanishes at the
    #     rest length and CHANGES SIGN below it, so the network resists
    #     compression at EVERY density. We have no analogue. That is the real
    #     missing physics and it is not in this batch.
    #  3. phi_s0 = 0.01 is the BOTTOM OF A BAND THE ANCHOR CANNOT RESOLVE.
    #     Wufsus p.1818: no platelet contribution to permeability below
    #     f_p = 0.19, so k = 1.1e-14 m2 fits anything in 0.01-0.19. Du &
    #     Fogelson start at 0.10 and compact to 0.45-0.58.
    #
    # So: em_2vel is the CONTROL (phi_s0 = 0.01, matched to Batch 16/17, tests
    # only whether the lumen fix holds), and em_2vel_ps10 is the SAME CHILD
    # with skeleton_volume_fraction = 0.10 -- verified to differ in that one
    # key and nothing else. Authority against the deviatoric term goes from
    # 2.5e-04 to 3.6e-02 and parity moves from 59x nominal to 5.3x.
    #
    # PRE-REGISTERED, both children:
    #   em_2vel       -- cUnder tracks Batch 16 and crosses 0.05 at
    #                    t = 2.97e-04 (step 197 of 1855, ~3 min); dies near
    #                    t = 2.8e-03, ~30 min. maxVs/maxU back to ~6.4 from
    #                    237.7. THAT RATIO IS THE WHOLE POINT OF THIS CHILD.
    #   em_2vel_ps10  -- if phi_s0 is the answer it DEPARTS from that
    #                    trajectory before 2.97e-04 and phiSOverNominal levels
    #                    off near 5x. If it tracks the control anyway, p_n is
    #                    not the mechanism at ANY anchor and the next move is
    #                    the rest-length term.
    #
    # COST IS ASYMMETRIC. Control ~30 min. The variant is ~30 min if it fails
    # and the full 0.03 s window -- 10.7x, the nominal 5.6 h -- if it works.
    # A LONG RUN ON THE SECOND CHILD IS THE GOOD OUTCOME, NOT A HANG.
    #
    # CONCURRENCY 1: ~6.1 h worst case, ~1 h if both die on schedule.
    18: ("em_2vel em_2vel_ps10", 6.1, 'eulerian, THROAT, PAIRED: lumen-fix control + phi_s0 = 0.10'),

    # ------------------------------------------------------------------------
    # BATCH 19. THE FORM OF p_n, WITH THE ANCHOR HELD FIXED.
    #
    # Batch 18 answered the ANCHOR question and raised a new one. phi_s0 = 0.10
    # slowed the compaction by a third (cUnder 0.3132 against Batch 16's 0.4611
    # at t = 8.5e-04) at 4.7% of the deviatoric stress -- the first measured
    # effect the network pressure has ever had. But it ALSO made the diffuse
    # interface worse, and the per-cell field data says exactly why.
    #
    # THE DEFECT, MEASURED NOT ARGUED. At t = 5e-04:
    #   phi_s0 = 0.01: max|Vs| = max|U| = 0.015342 IN THE SAME CELL, and ZERO
    #                  cells anywhere above max|U|. Clean.
    #   phi_s0 = 0.10: 88 cells above max|U|, ALL in the diffuse edge (chi from
    #                  5.9e-04 to 0.118, median 9.1e-03), one at |Vs|/|U| = 646.
    # The shipped pole law does not vanish at zero network:
    #     p_n(0) = -K_n*phi_s0/phi_s_max  =  -0.0667 Pa at 0.01, -0.667 at 0.10
    # so the stress term is LINEAR in phi_s at the fringe, the force across the
    # interface goes as phi_s0^2, the momentum diagonal there goes as phi_s0,
    # and the spurious velocity therefore goes as phi_s0. Ten times the anchor,
    # ten times the artefact. That is the same defect class as the
    # 20260905-191135 lumen bug, one band further out.
    #
    # THE FIX IS THE SOURCE'S OWN FORM. Cogan & Guy Eq. 27 / Fig. 4 prescribe a
    # cubic with "a local maximum and root at phi_n = 0 and a minimum at a
    # nonzero reference value" -- a DOUBLE ROOT AT ZERO, which is precisely the
    # property that removes the fringe force:
    #     p_n = K_n * phi_s^2 * (phi_s - phi_s0) / (4*phi_s0^3)
    # zero and flat at 0, zero at phi_s0, minimum at 2*phi_s0/3, negative below
    # nominal so it still resists dilation, cubic above it. Normalisation is
    # STATED, not fitted: p_n = K_n at twice nominal. Fringe term falls from
    # O(phi_s) to O(phi_s^3) -- 1e-06 at chi = 1e-03. No pole, and none needed:
    # the 0.6 anchor was Du & Fogelson's DRAG coefficient all along and that
    # role is kozenyCarmanSolid's.
    #
    # TWO CHILDREN, ONE KEY APART.
    #   em_2vel_ps10_wi -- the pole law, phi_s0 = 0.10. MUST reproduce Batch
    #     18's em_2vel_ps10 step for step: cUnder 0.2075 at 7.0e-04, 0.3132 at
    #     8.5e-04, 0.5286 at 1.1e-03, death near 1.339e-03, maxVs/maxU peak
    #     99.7. If it does not, the write interval is not inert and the
    #     comparison is void.
    #   em_2vel_cub     -- the cubic. Same everything else.
    #
    # PRE-REGISTERED for the cubic, both falsifiable:
    #   1. THE FRINGE GOES QUIET. Whole-field maxVs/maxU collapses toward 1
    #      from 65-100, and VS_WHOLE_FIELD_LIMIT = 20 passes. If it does not,
    #      the phi_s0^2 scaling argument is wrong and the fringe was not the
    #      mechanism.
    #   2. COMPACTION SLOWS FURTHER. ~17% of the deviatoric stress at 1.5x
    #      nominal against the pole law's 4.7%, so cUnder should fall below
    #      em_2vel_ps10_wi's. NOT expected to stop it -- that wants O(1).
    #
    # WRITE INTERVAL 2.5e-05, not 5e-04. Batch 18 wrote TWO time directories
    # for a run that lasted 8.96e-04, so the clotDrag movie was one frame and
    # the control's blow-up at t = 8.4488e-04 had no field snapshot within
    # 3.4e-04 of it. 36-54 directories now. RAISE IT AGAIN if anything survives
    # past ~3e-03; at 0.03 s this writes 1200.
    #
    # NEW DIAGNOSTICS, both from Batch 18's blind spots:
    #   PHASE_CLOT_CONFIG networkPressureModel=  -- read it first and confirm
    #     the child ran the law you think it ran.
    #   PHASE_CLOT_PERMEABILITY kRatioMin/Max/Mean, kCellsBelowHalf,
    #     kCellsAboveTwice, kCellsOff10pc. kClotMin/kClotMax are pinned by the
    #     core/shell contrast (1e-14 against 1e-11) and read "nothing happened"
    #     all through Batch 18 while the field snapshot had 1552 of 2619 clot
    #     cells off nominal and one at 4x k0p.
    #
    # STILL UNEXPLAINED AND NOT ADDRESSED HERE: why Batch 18's phi_s0 = 0.01
    # control died at 8.96e-04, 3.1x earlier than Batch 16, in a sudden in-clot
    # event at x = 8.46e-05, y = 4.64e-06 near the bottom wall, growing 1.13x
    # per step after being clean (maxVs/maxU EXACTLY 1.00) for 8.3e-04. That is
    # a different failure from the fringe one. The write interval is what makes
    # it diagnosable next time it happens.
    #
    # COST: ~30 min if both die where Batch 18 did. Up to 5.6 h for the cubic
    # if it works, which is the good outcome.
    19: ("em_2vel_ps10_wi em_2vel_cub", 6.1, 'eulerian, THROAT, two-velocity, p_n FORM: pole vs cubic at phi_s0 = 0.10'),
    # BATCH 19 RETURNED (run 20260906-004750) AND BOTH CHILDREN WERE EXCLUDED,
    # both "phase out of bounds", both SIGFPE. BOTH PRE-REGISTERED PREDICTIONS
    # FAILED. Full record:
    # research/runs/20260906-004750-the-network-pressure-law-is-a-0.01-percent
    # -term-and-the-c-overshoot-is-interior.md
    #
    #   1. THE TWO p_n LAWS WERE INDISTINGUISHABLE. cUnder at matched time:
    #      0.006444/0.02352/0.04970/0.08325 (pole) against
    #      0.006457/0.02350/0.04953/0.08281 (cubic) at t = 1,2,3,4e-04, and
    #      phi_s/nominal agreed to four decimals.
    #
    #   2. THE MOMENTUM BUDGET SAID WHY. At t = 4e-04: accDrag -7297,
    #      accDragReaction +7268, accPressure +260, accPenalty -364,
    #      accBondHold -97, accViscous -13.7, accInertia +6.7,
    #      accElastic -0.71, accResidual 113. p_n lives in accElastic: 0.01%
    #      of the drag, 1/160 of the equation's own residual. Drag and its
    #      reaction cancel to 0.4%, so every physical term sits inside that
    #      cancellation.
    #
    #   3. RETRACTED: that em_2vel_ps10 ran at whole-field maxVs/maxU of
    #      65-100 over its first 1e-04. Measured: 21.4 over the first 1e-04,
    #      23.53 over the whole run. Batch 19's prediction (1) rested on it.
    #
    # WHAT THE SOURCES THEN SAID, and it re-aimed this batch away from the
    # interface question entirely (see the two ps45 children's own
    # `expectation`, which carries the citations):
    #
    #   Clot and blood are NOT immiscible. Cogan & Guy 2010 call it an
    #   "interpenetrating flow model" with phi_s + phi_n = 1 at every point,
    #   and note that each phase's own velocity is NOT divergence-free --
    #   only the volume-averaged one is. Du & Fogelson 2018: "Both fluid and
    #   bound platelets may be present at each spatial location." So VoF is
    #   ruled out on its premise, not on taste, and MULES was the wrong
    #   answer to the c overshoot.
    #
    #   THE RESISTING MECHANISMS ARE INACTIVE AT OUR ANCHOR. Du & Fogelson
    #   Eq. 2.25's packing drag contributes 4.2e-09 at theta_b = 0.01 and
    #   9.4e-07 at 0.10, against 2.5e+04 at 0.50. c = -1 is phi_s = phi_s0,
    #   i.e. NOMINAL density, not close packing: the measured cMin = -1.619
    #   is phi_s = 0.0162, still 37x below 0.6, and reaching 0.6 would need
    #   c = -119. The compaction is REAL and the model has no variable to
    #   hold it -- c is being asked to be both the indicator (bounded) and
    #   the density (free to rise 60x).
    #
    #   phi_s0 = 0.01 came from Wufsus's PLASMA GEL (~0.005), not from a
    #   clot. Retrieved human clots: platelets 20-80% of solid volume; MI
    #   aspirates 56% fibrin solids. Du & Fogelson's aggregates run to
    #   0.45-0.58; Xu 2017 uses solid fractions 0.6 (shell) and 0.8 (core);
    #   Zheng/Yazdani/LI/Humphrey/Karniadakis 2020 -- the PI's own paper --
    #   uses VF = 0.3419, 0.5129, 0.6335.
    #
    # BATCH 20 IS THAT TEST. Two children at phi_s0 = 0.45, one key apart, so
    # they also repeat Batch 19's pole-vs-cubic comparison AT AN ANCHOR WHERE
    # THE TWO LAWS CAN ACTUALLY DIFFER -- at 0.45 the pole sits 1.33x away and
    # the 1.31x compaction already measured at 0.10 lands on the clamp.
    #
    # CONTROL: run 20260906-004750's em_2vel_ps10_wi. Compare at MATCHED TIME
    # against cUnder 0.00644/0.02352/0.04970/0.08325/0.28075 and
    # phiSOverNominal 1.0032/1.0118/1.0248/1.0416/1.1404 at t = 1,2,3,4,8e-04.
    # Do NOT read this batch on survival time.
    #
    # HELD FIXED AND INCONSISTENT ON PURPOSE: k0p. Wufsus measure ~1.2e-17 m2
    # at fiber volume fraction 0.54 and our shell is 1e-11 m2; Xu use 1e-13
    # and 1e-15 against our 1e-11 and 1e-14. Lowering k0p to match would put
    # nu/k at 3.3e+11 -- 1000x the present core drag -- which is a separate
    # question. This batch varies ONE thing.
    #
    # em_2vel_ps10_clip (the c-clip measurement) is BUILT AND STAGED IN THE
    # CONTROLS BUT NOT RUN. clipPhaseField defaults off, so it changes nothing
    # for these two. It was aimed at "is MULES enough", which the sources
    # answered before it cost a run.
    #
    # COST: ~6 h if both run on, ~40 min if both die where Batch 19 did.
    20: ("em_2vel_ps45 em_2vel_ps45_cub", 6.1, 'eulerian, THROAT, two-velocity, THE LITERATURE ANCHOR: phi_s0 = 0.45, pole vs cubic where they can differ'),
    # BATCH 21 -- THE REGULARISATION FIX. Runs 20260906-052320 and -101812 both
    # killed em_2vel_ps45 at 4.56e-05 s and em_2vel_ps45_cub at 1.19e-04 s, with
    # identical wall times (33/84 s and 31/80 s), i.e. deterministically. Neither
    # died of compaction: phiSOverNominal was 1.0011 and 1.0054 at death.
    #
    # PHASE_CLOT_LUMEN_FORCE then measured where the elastic force lands --
    # vsElasBlood 62x (pole) and 71x (cubic) vsElasBulk, devBeBlood 2.15 against
    # devBeBulk 0.016 -- and a hand parse of the written fields found ALL 100 top
    # cells by |Vs| at c > 0.99 with Kres EXACTLY ZERO and Kd ~ 0.013 1/s.
    #
    # Kres = max(residualPhiS - phi_s, 0)/deltaT is a fixed ABSOLUTE threshold on a
    # quantity that scales with the anchor, so raising phi_s0 to 0.45 slid the
    # diffuse tail of c across it and switched the regularisation off in the lumen.
    # residual_chi = 0.01 measures the same ramp on chi, rescaled so the PEAK is
    # unchanged at 68 1/s. Region changes, strength does not. One key from Batch 20.
    #
    # NOT ESTABLISHED, AND THE BATCH IS BUILT TO FIND OUT: that the dead Kres band
    # is the CAUSE and not merely the SITE. It was non-empty at phi_s0 = 0.10 too
    # (489 cells, max |Vs| 0.2234) and that run survived 25x longer.
    #
    # READ PHASE_CLOT_LUMEN_WORST FIRST -- new this batch, it reports the argmax
    # cell of vsElas with its own c, chi, phiS, Dblk, As, Kd, Kres and devBe.
    #
    # BUDGET: the Batch 20 children died in 31 and 80 WALL SECONDS. If the fix
    # works they run long; if it does not, this batch costs about two minutes of
    # solver plus the boot and compile.
    21: ("em_2vel_ps45_rchi em_2vel_ps45_cub_rchi", 6.1, 'eulerian, THROAT, two-velocity, THE REGULARISATION FIX: residual_chi = 0.01 against Batch 20, pole and cubic'),
    # BATCH 22 -- THE PERMEABILITY. Run 20260906-210336's cubic child COMPLETED
    # 30 ms, the first two-velocity child ever to reach endTime. It was still
    # inadmissible (cUnder 0.1049 vs 0.05) and Alireza read the movie correctly:
    # the clot stops deforming after ~3 ms and then drifts downstream as a body.
    #
    # The strain saturation is PHYSICAL -- a neo-Hookean solid reaches
    # equilibrium and stops, and accElastic finally has authority (-100.9,
    # against -0.71 in Batch 19). The DRIFT is not: PHASE_CLOT_LOCK sat at
    # lockRatio 0.664 from 1 ms to 30 ms, i.e. the clot slowed the flow by one
    # third and no more, because k0p = 1e-11 is 100x Xu's shell and ~1e6 x
    # Wufsus's measurement. With bondSeeds 56 -> 7 nothing held it and the whole
    # body crept at 1.83e-04 m/s.
    #
    # This child takes XU'S PAIR: k0p 1e-13 (shell) and 1e-15 (core). One pair of
    # keys from em_2vel_ps45_cub_rchi; everything else identical.
    #
    # READ PHASE_CLOT_LOCK FIRST. PASS = lockRatio falls to <= 0.1 and stays,
    # and clotXbar drift falls well below 1.83e-04 m/s (compare at MATCHED TIME
    # against 8.14e-05 at 2.94 ms and 8.635e-05 at 30 ms).
    #
    # nu/k goes 3.3e+05 -> 3.3e+07 (shell). AN EARLY SIGFPE IS AN ADMISSIBLE
    # OUTCOME of this test, not a surprise -- this solver has never run there.
    22: ("em_2vel_ps45_cub_xuk", 6.1, 'eulerian, THROAT, two-velocity, THE PERMEABILITY: Xu k0p 1e-13/1e-15 against Batch 21'),
    # BATCH 23 -- THE DILUTE DRAG FLOOR. One key from Batch 22:
    #   skeleton_dilute_drag_rate = 1e+05 1/s, imposing K >= k1*chi.
    #   k0p STAYS at Xu 1e-13/1e-15, so exactly one thing changes
    #   against run 20260907-111028.
    # WHY: that run died at 5.907e-03 s of |Vs| = 3.90x max|U| in cells
    #   that are 98% BLOOD. Binning every cell by chi, compliance
    #   (elastic weight / total drag rate) is 1.5e-06 and 1.4e-05 below
    #   chi = 0.005 where |Vs|/max|U| is EXACTLY 1.00, and 3.0e-04 in
    #   chi 0.01-0.02 where it is 3.71. Kozeny-Carman is a packed-bed
    #   law being used two decades below its range; the floor pins
    #   compliance at 1/k1 = 1e-05 wherever it binds, flat, no band.
    # REJECTED FIRST, ARITHMETICALLY, AGAINST THE MEASURED FIELD:
    #   widening residualChi 0.01 -> 0.1 buys 2.2x; reweighting the
    #   deviatoric stress by solidFraction() buys 2.2x. The gap is
    #   20-30x. Neither is sized to it.
    # READ FIRST: PHASE_CLOT_DILUTE. It reports the chi 0.005-0.1 band
    #   directly -- maxVsBandOverMaxU, maxCompliance, flooredCells --
    #   which last time had to be dug out of the written fields because
    #   maxVsInClot reads 0 at EVERY sample of every run so far.
    # PASS: maxCompliance <= 1.1e-05 AND maxVsBandOverMaxU <= 1.2 AND
    #   maxVsX stays below the taper start 1.2e-04 AND it survives well
    #   past 5.907e-03 s.
    # FAIL, REAL: compliance is bounded (arithmetic) and |Vs| runs away
    #   anyway -> the driver is `be` advected into the lumen carrying
    #   stretch 2.18, not any drag or weight parameter.
    # OVER-DAMPED: clotBeXY collapsing toward 0.4 or lockRatio rising
    #   means k1 is too large -- lower it, CONFIG ONLY, no rebuild.
    23: ("em_2vel_ps45_cub_dfloor", 6.1, 'eulerian, THROAT, two-velocity, THE DILUTE DRAG FLOOR: K >= 1e5*chi against Batch 22'),
    # BATCH 24 -- RAISE THE FLOOR CONSTANT. One key from Batch 23:
    #   skeleton_dilute_drag_rate 1e+05 -> 1e+06. Nothing else moves.
    # WHY, FROM A CLOSED BALANCE and not a proxy: run 20260907-204203
    #   gives F := |Vs-U|*K equal to 0.95*chi*|grad p|, with the
    #   coefficient between 0.92 and 1.24 while K ranged over a FACTOR
    #   OF 12000 (685 to 8.3e+06) in one run at one instant. Where the
    #   floor binds K = k1*chi, the chi CANCELS, and the slip is
    #   0.95*|grad p|/k1 however little skeleton is there.
    #   The FORM is right (dilute-suspension slip IS concentration-
    #   independent); the CONSTANT was 27x too small. Inverting the
    #   case's OWN Kozeny-Carman at k0p=1e-13 with nu measured as
    #   exactly 3.333e-06 gives d = 4.68 um and 18*nu/d^2 = 2.74e+06.
    # READ FIRST: PHASE_CLOT_DILUTE. maxSlipBand and maxSlipBandOverMaxU
    #   are NEW in this package and report |Vs-U| directly.
    # PASS: maxSlipBandOverMaxU 0.78 -> ~0.08, maxSlipBand 0.0282 ->
    #   ~0.0028, maxCompliance 1e-06, maxVsBandOverMaxU stops short of
    #   ~1.15 instead of reaching 1.59, and it survives past 8.9418e-03 s.
    # FAIL, REAL: the slip falls as predicted and it dies at about the
    #   same time anyway -> the slip is not what ends these runs, and the
    #   next place is the OUTLET (Vs/Us are zeroGradient there while U is
    #   pressureInletOutletVelocity), NOT another drag parameter.
    # COST: with nu calibrated on 4738 floor-free cells, Kozeny-Carman is
    #   exact to 9.4e-12 everywhere, and at 1e+06 the floor exceeds it in
    #   0 of 4086 clot cells. 1e+07 was priced and REJECTED (3617/4086 at
    #   up to 3.66x). Watch clotBeXY, lockRatio, accElastic against Batch
    #   23: if they move, the zero-perturbation claim is falsified.
    24: ("em_2vel_ps45_cub_df1e6", 6.1, 'eulerian, THROAT, two-velocity, RAISE THE FLOOR: k1 1e5 -> 1e6 against Batch 23'),
    # BATCH 25 -- THE CARRIER. Not another parameter; it is the defect
    #   that has ended every two-velocity run.
    # THE RECORD, across every child in preflight/:
    #   carrier = phi_ (pressure-corrected, solenoidal, one-velocity):
    #     9 children, 9 COMPLETED.
    #   carrier = phiSkelOwn_ (raw skeleton flux, two-velocity):
    #     14 children, 14 FAILED -- 13 by SIGFPE with an IDENTICAL stack
    #     (symGaussSeidelSmoother::smooth <- advectMaterialLabel <-
    #     solvePhaseField <- prePredictor), 1 (cub_rchi) by going inert:
    #     its clotXbar is flat to four digits from 14.7 to 30 ms while
    #     clotUxbar reports 8.96 mm/s -- 134 um of travel owed, 0.006 um
    #     delivered.
    # POINTWISE, NOT A THRESHOLD: the 13 crashes scatter over 300x in time
    #   (4.4e-05 s to 1.42e-02 s). On the step before the crash in Batch 24
    #   the shell equation solved 8.08e-05 -> 5.18e-13 in 2 iterations and
    #   every reported diagnostic was smooth into it.
    # WHY THE CARRIER IS WRONG: phiSkelOwn_ is the skeleton row of a 2x2
    #   block that in the lumen is held up only by Kres = 67 1/s. Batch 24
    #   measured |Vs| = 0.0386 m/s -- 110% of max|U| -- in a cell with
    #   chi = 4.03e-11. That velocity was carrying c and shell.
    # THE FIX is not new code: blend to phi_ as chi -> 0, which the
    #   one-velocity branch of solvePhaseField has always done. Inside the
    #   clot chi -> 1 and the carrier is unchanged, so the clot still rides
    #   its own skeleton.
    # ALSO IN THIS PACKAGE: advectMaterialLabel now uses
    #   phaseSharedConvection() -- the SAME face weights as c -- instead of
    #   rebuilding vanLeer weights from a 0/1 label. That is the defect
    #   fvSchemes already records for Wp (run 20260822-103655); the label
    #   never inherited the fix.
    # TWO CANDIDATE CAUSES WERE PRICED AND KILLED FOR FREE, no run spent:
    #   (a) skeleton Courant / diagonal degradation -- measured max 0.121
    #       from Batch 24's written Vs, ZERO cells above 0.5;
    #   (b) the w/Vn blowup -- the run that did NOT crash reached
    #       wMin -72534 and Vn 2.72 m/s and carried on. Both dead.
    # READ FIRST: PHASE_CLOT_CARRIER, new in this package. maxCarrierCo is
    #   the Courant number of the flux that transports c and shell (maxCo in
    #   controlDict bounds phi_, NOT this); minShellDiagOverVbyDt is the
    #   shell matrix diagonal that symGaussSeidel divides by;
    #   shellDiagNonPositive/NonFinite count the cells that would fault.
    #   If it dies again, those four say why in one line, not a stack trace.
    # THE PAIR: em_2vel_carrier is the fix alone (erosion OFF).
    #   em_2vel_carrier_ero adds surface erosion at 2e-05 m/(s.Pa) above
    #   tau_emb = 3.0 Pa and differs in the erosion keys ONLY, so the two
    #   are a clean A/B. Erosion answers a separate question -- the clot
    #   cannot presently let go by ANY route: bulk fracture is unreachable
    #   (Gc/G = 1.39 m against a 40 um clot, Dg = 6.4e-06 of 1.0), wall peel
    #   arrests (utilisation 0.985 at 3 ms -> 0.389 at 7.9 ms), and release
    #   is a connectivity flood fill that cannot sever a connected tail.
    # SECOND FIX -- THE ROOT OF THE c UNDERSHOOT: A CELL WITH NO BLOOD LEFT
    #   COULD STILL GROW MORE CLOT. carrierDivergence() derives correctly that
    #   chi = 0.5(1-c) transported conservatively gives
    #       ddt(c) + div(c v) = div(v) - 2G.
    #   Evaluate on the pure states:
    #       c = +1 (pure blood): dc/dt = -2G           -- preserved at G = 0
    #       c = -1 (pure clot) : dc/dt = 2div(v) - 2G  -- NOT preserved
    #   Blood is a fixed point, clot is not. So growth kept running inside
    #   cells that were already 100% clot and c walked past -1.
    # THE SPLIT IS TOTAL ACROSS EVERY RUN ON RECORD: cUnder is EXACTLY 0 in
    #   all 9 one-velocity children (cMin = -0.99998), and nonzero in 14 of 16
    #   two-velocity ones. The one-velocity carrier is phi_, which IS
    #   solenoidal, so div(v) = 0 and c = -1 is a fixed point there. Same
    #   growth source, same schemes, same everything else.
    # MEASURED in Batch 24 at 7.9 ms: 213 of the 239 cells with c < -0.999
    #   still carry a live growth source, |grad c| there has median 2.78e+04
    #   (a tanh profile gives ~0 -- the carrier distorts the plateau, so the
    #   source never switches itself off), and in 55 cells the increment
    #   dt*Vn*|grad c| ALONE overshoots c = -1 within one step.
    # THE FIX bounds Vn, not the c equation:
    #       Vn <=  (1 + c)/(dt*|grad c|)     growth stops when blood is gone
    #       Vn >= -(1 - c)/(dt*|grad c|)     erosion stops when clot is gone
    #   Applied to Vn because lambda_, surfaceLossCoeff_ and the
    #   stoichiometric platelet uptake are all built from it -- bounding the
    #   speed keeps stoichNorm at 0.998. Bounding the c source alone would
    #   have grown clot without consuming the platelets that pay for it.
    #   It is NOT the clip: clipPhaseField removes material after the field
    #   has already left bounds; this withholds a source with nothing to act
    #   on, so the violation is never created.
    # INERT WHERE HEALTHY: at the interface (c in [-0.9,-0.1]) measured
    #   Vn/limit is at most 3.35e-04, three thousand times below binding. It
    #   binds in 55 of 31360 cells, all already pathological, and the nine
    #   one-velocity runs never reach it.
    # CORRECTION caught before build: a bare clamp on Vn_ would have BROKEN
    #   the stoichiometry. clotGrowth = int Vn*phaseGradient reads Vn_;
    #   plateletUptake = int interfaceDelta*adhesionSpeedEff*w does NOT. A
    #   bare clamp consumes platelets that produce no clot and stoichRatio
    #   (1.6277 vs target 1.6304) drifts unexplainably. The growth limiter
    #   now scales Vn_ AND adhesionSpeedEff_ by the SAME per-cell factor,
    #   which leaves clotGrowth/plateletUptake exactly unchanged because Vn
    #   is linear in adhesionSpeedEff. Gated on growthClosure 'reaction'.
    # READ PHASE_CLOT_VN_BOUND_GROWTH (growthLimited, worstFactor,
    #   uptakeScaled) and PHASE_CLOT_VN_BOUND (erodeLimited). PASS also
    #   requires stoichNorm to hold at 0.998 -- if it moves, the paired
    #   scaling is wrong.
    #   And qNegCells in PHASE_CLOT_PHASE_BOUNDS should now be 0. If the root
    #   fix holds, the divisor guard below never fires.
    # THIRD, now DEMOTED TO A BACKSTOP and it is separately observable so the
    #   bundle does not confound: THE PLATELET DIVISOR WAS AMPLIFYING BY
    #   EXACTLY 1e+06. qFloor guards a division against a SMALL q; it does
    #   nothing about a NEGATIVE one. max(-0.056, 1e-06) returns the floor,
    #   so a Wp that went negative with q is divided by +1e-06 instead of
    #   -0.056. Measured in Batch 24 at 7.9 ms: 230 cells with c < -1 hence
    #   q < 0, and w/Wp EXACTLY 1.000000e+06 in every one of them. wMin
    #   -30626.8 is precisely Wp -0.030627 over the floor; in healthy cells
    #   w/Wp never exceeds 8.5e+04. wMin = -3.1e+04 was never an
    #   instability, and VnMagMax = 1.15 m/s is the same artefact one step
    #   downstream -- all 30 cells with |Vn| > 0.01 have q < 0.
    #   It closed a FEEDBACK LOOP: Vn drives -Vn*phaseGradient in the c
    #   equation, and the growth source in the undershoot cells measures 18x
    #   the clot median (13.1 vs 0.71). All six divisor sites now use
    #   max(mag(q), qFloor), which is BIT-IDENTICAL wherever q >= 0.
    # WHY BUNDLING IS SAFE: the two fixes are spatially DISJOINT. The carrier
    #   blend acts where chi -> 0 (lumen and fringe); all 230 undershoot
    #   cells are at chi > 0.9, deep clot, ZERO at the interface and ZERO in
    #   the lumen. And each has its own diagnostic -- PHASE_CLOT_CARRIER for
    #   one, qNegCells/qNegAmpMax in PHASE_CLOT_PHASE_BOUNDS for the other.
    # WATCH qNegAmpMax: it should read ~1/|qMin| (order 18), NOT 1e+06. If
    #   it still reads 1e+06 the guard is not on the path that matters.
    # THIS IS THE AMPLIFIER, NOT THE ROOT. Why c leaves [-1,1] in the clot
    #   bulk at all is NOT measured -- candidate is the conservative
    #   convection and the explicit carrierDivergence failing to cancel
    #   discretely where the carrier has divergence inside the clot
    #   (trBulk = 131 1/s), with mobility = 0 so nothing restores c to the
    #   well minimum. qNegCells is the number that tracks the root.
    # PASS: em_2vel_carrier passes 8.94 ms, the furthest any k0p=1e-13
    #   two-velocity child has reached. FAIL: another SIGFPE in
    #   advectMaterialLabel -- with PHASE_CLOT_CARRIER naming the cell.
    25: ("em_2vel_carrier em_2vel_carrier_ero", 6.5, 'eulerian, THROAT, two-velocity, THE CARRIER: blend to phi_ in the lumen + shared label weights; A/B on surface erosion'),
    # BATCH 27 -- STAGE 1 OF THE CONTENT MODEL: REPRESENTATION, NOT PHYSICS.
    # em27_content_s1 differs from em26_blend_off in contentModel ALONE,
    # verified by a full recursive diff of the generated cases. Kinetics are
    # off and theta is REPORTED but never consumed by permeability, drag or
    # momentum, so the momentum solution should be unchanged.
    #
    # em26_blend_off IS RE-RUN AS A SAME-PACKAGE CONTROL. It already ran in
    # 20260909-040224, but that was a different package: this one adds the
    # Stage 1 fields (always constructed, AUTO_WRITE) and the Stage 0
    # assessor. Re-running it makes "Stage 1 changes no physics" a TESTED
    # claim -- the two logs must agree step for step outside the new
    # PHASE_CLOT_CONTENT records -- instead of an asserted one.
    #
    # THE ONE QUESTION: does conservatively transported solid content
    # reproduce the legacy chi = max(1-q,0)? Read PHASE_CLOT_CONTENT.
    #   PASS  equivMax at solver tolerance for all 272 steps, contentChange
    #         at round-off up to boundary flux, thetaCNeg = thetaHNeg = 0.
    #   FAIL  equivMax grows secularly -> the conservative content operator
    #         and the legacy c operator are NOT equivalent on this carrier,
    #         and Stage 2 must not be built until that is explained.
    #   THIRD, and informative: theta negative where legacy c stayed bounded
    #         -> first-order upwind on a compressible carrier is not
    #         positivity preserving here. Nothing is clipped, so it shows.
    #
    # COST: 2 children x 272 steps. em26_blend_off measured 465 s clock in
    # 20260909-040224; Stage 1 adds two scalar solves per step to the other
    # child. Estimate ~8-12 min per child, ~20 min for the pair at
    # CONCURRENCY 2. Extrapolated from measurement, not promised.
    27: ("em27_content_s1 em26_blend_off", 0.35, 'eulerian, THROAT, two-velocity, STAGE 1 CONTENT MODEL: conservative thetaC/thetaH transport vs legacy chi, kinetics off, with a same-package control'),
    # BATCH 28 -- T2a, THE OPERATOR IDENTITY TEST.
    #
    # WHY. Batch 27 (run 20260909-115458) measured equivMax growing to
    # 1.61e-02 under the shipped `content` operator, which I had
    # PRE-REGISTERED AS A FAILURE and which is nothing of the kind: the
    # theta equation and the legacy chi equation are different equations.
    # theta omits the growth source 0.5*Vn*phaseGradient and uses
    # first-order upwind where the legacy path uses vanLeer-blended
    # phaseWeights_. (mobility is 0.0 in this case -- verified in the
    # dictionary and by clotDiffusion = 0 in all 272 mass records -- so the
    # Cahn-Hilliard term is NOT a third difference.) I wrote an acceptance
    # criterion for an equation the solver does not solve.
    #
    # em28_content_matched removes both differences: same weights, plus the
    # mapped source theta_ref*(-0.5*laplacian(mobility,muC)
    # + 0.5*Vn*phaseGradient). Nothing then separates the two operators, so
    # ANYTHING LEFT OVER IS A CODING ERROR. It differs from
    # em27_content_s1 in contentOperator ALONE.
    #
    # em27_content_s1 IS RE-RUN in the same package, for two reasons: it
    # gives the one-key diff its control, and its equivMax is re-measured
    # with the time-level fix (chi was built from q_, which
    # updateDerivedFields() assigns BEFORE the phase solve -- the logged
    # 1.6102e-02 should read 1.6470e-02).
    #
    # READ PHASE_CLOT_CONTENT, AND READ IT AS A CURVE.
    #   PASS  em28 equivMax stays at the linear-solve floor. thetaC/thetaH
    #         carry tolerance 1e-12 relTol 0, so O(1e-12) per solve and at
    #         worst O(1e-9) over 272 steps if errors added coherently. It
    #         must NOT grow secularly the way em27's does.
    #   FAIL  em28 equivMax grows like em27's. Then the ASSEMBLY is wrong,
    #         not the model, and Stage 2 must not be built. Named in
    #         advance so the search is not improvised: the boundary
    #         treatment of div_w(carrier, uniform 1) against
    #         carrierDivergence(); the thetaC/thetaH source split; and the
    #         max(.,0) in chiLegacy that theta has no counterpart for.
    #   EXPECTED AND CORRECT: em28's contentVolume is NO LONGER conserved.
    #         The mapped source adds material. The balance to check is
    #         contentChange against contentSourceDt, which is why that
    #         field is now logged.
    #
    # COST: both children measured 438 s and 435 s clock in 20260909-115458.
    # ~15 min for the pair at CONCURRENCY 2. Measured, not promised.
    28: ("em28_content_matched em27_content_s1", 0.35, 'eulerian, THROAT, two-velocity, T2a OPERATOR IDENTITY: matched weights + mapped source vs the shipped content operator'),
    # BATCH 29 -- THE CONFIGURATION NO RUN ON RECORD HAS.
    # em27 and em28 both leave plateletContentModel OFF, so their content is
    # inert: it has no growth mechanism enabled and cannot move. A phase-weight
    # migration measured against them would prove nothing, because "nothing got
    # worse" is not a result when nothing was moving.
    # Every gate here is an IDENTITY -- the content budget closes against
    # Y*sum(dBSum), the pairing residual is zero, and the mixture flux equals
    # phiFf*phiFluidOwn + phiSf*phiSkelOwn -- so none of them depends on the
    # growth hitting any particular size.
    # NOT PHYSIOLOGICAL, and labelled so: the accepted transfer carries no
    # spatial localiser (standing backlog B8), so deposition is uniform and
    # fastest in the LUMEN. lambda is held at 42 1/s, where the unphysical
    # uniform gain is 4.2e-05 -- four decades below the 7.41e-03 C1 weight
    # discrepancy this run is measuring. No morphology claim follows from it.
    29: ("em29_stage2_baseline", 0.25, 'eulerian, THROAT, two-velocity, STAGE 2 CONSISTENCY BASELINE: contentOperator content WITH plateletContentModel. Assess with scripts/assess_stage2_baseline.py'),
        # BATCH 30 -- THE LOCALISER, WITH THE TOTAL HELD FIXED.
        # Run 20260915-073208 measured the defect em29 was built to expose but
        # could not fix: 86.5% of all deposition landed in cells that are >99%
        # BLOOD, at a per-volume rate 1.82x the rate in the clot core, because
        # dB ~ dt*lambda*N and N = epsP*nP is LARGEST where there is no solid.
        # Not merely unlocalised -- ANTI-localised.
        #
        # The controlled spec had the answer already. AM2020-phase-clot-v1 (P)
        # puts uptake on a surface delta, delta_Gamma[(k(s)+V_n-u.n)w], and the
        # LEGACY path implements exactly that at .C:8309. The localiser was
        # never missing from the model or the code -- only from this path. The
        # dimensional signature is the tell: the specified k is m/s and pairs
        # with a delta of 1/m; the old plateletBindRate was 1/s and paired
        # with nothing.
        #
        # THE TOTAL IS MATCHED ON PURPOSE. kEff = 6.045e-03 m/s makes
        # INT dB dV equal em29's -- checked against em29's OWN fields through
        # the exact kernel at 5e-05 s (0.16% apart) and 4e-04 s (0.07%), with
        # the required kEff drifting only 1.002x across the run. So this is a
        # ONE-VARIABLE diff: same amount of deposition, different place.
        #
        # PRE-REGISTERED: blood share 86.5% -> 1.6%; bindRateClotFrac ~0.03 ->
        # >0.8; and G1/G2b/G4 STILL PASS, because they are identities and hold
        # whatever the spatial pattern is. If a gate breaks, the rate field
        # broke the pairing rather than relocating it -- read pairMax first.
        #
        # NOT A PHYSICS PREDICTION. 6.045e-03 m/s is 131x the v2 closure's
        # adhesionSpeedEff = 4.6e-05 m/s and was picked to make the comparison
        # clean, not because it is calibrated. No morphology claim follows.
        30: ("em30_stage2_localised", 0.25, 'eulerian, THROAT, two-velocity, STAGE 2 LOCALISED: plateletBindLocalisation interface, kEff*interfaceDelta_ per AM2020-phase-clot-v1 (P), rate-matched to em29 so only the LOCATION of deposition differs'),
        # BATCH 31 -- C1/C2: THE MOMENTUM WEIGHTS MOVE TO THETA.
        # Step 3 of section 6 of the phase-volume coupling spec, and the FIRST
        # run in which theta does anything but get reported. em30 was
        # bit-identical to em29 in every phase-field and skeleton diagnostic
        # precisely BECAUSE theta was not consumed; this run consumes it.
        #
        # ONE KEY: phaseWeightBasis legacy -> content, so
        # phiSkel_ = min(max(thetaC+thetaH,0),0.9) instead of
        # min(max(1-q,0)*0.45,0.9). phiSf/phiFf are interpolants of phiSkel_/
        # phiFluid_ and of nothing else, so that one assignment moves the
        # flux-decomposition weight AND the weight platelet storage uses.
        #
        # WHY THE LOCALISER HAD TO LAND FIRST -- MEASURED, NOT ASSUMED. Content
        # weights put theta into the momentum equations in all 19119 cells
        # where solidFraction() is EXACTLY 0. Under em29's uniform rate theta
        # is 4.18e-05 there; run 20260905-191135 was killed by a lumen weight
        # of 1e-04, because the skeleton diagonal out there is only Kres ~ 68
        # 1/s against 3.3e+05 in the clot -- whole-field maxVs ran to 0.62 m/s
        # while every clot-cell metric stayed ordinary. Under em30's localiser
        # the same cells carry 2.66e-13, nine decades under that precedent.
        # The solver AND the generator refuse content weights on an
        # unlocalised path; lifting that needs an explicit recorded override.
        #
        # PRE-REGISTERED, each on the metric it is measured on:
        #  (1) PHASE_CLOT_PHASE_WEIGHT_CONFIG basis=content, else the run is
        #      void rather than null.
        #  (2) clampedLow=0 and clampedHigh=0 every step. THIS IS THE ONE THAT
        #      CAN FAIL: nothing bounds thetaC/thetaH anywhere in the solver,
        #      and phiFluid_ = 1 - phiSkel_ must stay strictly positive for the
        #      2x2 block. em30's theta ran [1.008e-19, 4.510e-01].
        #  (3) G4 relInternal <= 1e-12 (em30: 3.78e-16). G4 is an identity in
        #      WHATEVER weights are used, so it fails only on a PARTIAL
        #      migration -- the defect this run is actually exposed to.
        #  (4) G1, G2b unchanged in character; the weights are not in them.
        #  (5) NOT bit-identical to em30. The weight moves by up to 7.00e-03
        #      volume fraction (2.1% relative) at the interface, so maxVsOverU
        #      1.2212036074, clotCells 2586, maxPhiS 0.450980391685 should move
        #      at the percent level. Bit-identical = the branch never ran.
        #  (6) LUMEN SAFETY, priced linearly: spurious weight 2.66e-13 against
        #      the precedent's 1e-04, and the lumen diagonal is unchanged
        #      (Kres and isSkel read chi, not phiS, since residualChi=0.01), so
        #      spurious Vs ~ 1.6e-09 m/s against em30's maxVs 0.013147433975.
        #      FAIL if maxVs > 1.1x em30's, or if maxVs decouples from maxU.
        #
        # c1Max is REPORTED, NEVER GATED. It is the size of the change, not a
        # score. Sites 1/2/3/5 of the consumer map and all eleven indicator
        # sites are deliberately NOT migrated here. One step, not the migration.
        31: ("em31_stage3_contentweights", 0.25, 'eulerian, THROAT, two-velocity, STAGE 3 C1/C2: phaseWeightBasis content -- momentum/pressure weights from theta, not solidFraction(). Assess with scripts/assess_stage2_baseline.py, then read PHASE_CLOT_PHASE_WEIGHT_BASIS and G4.'),
        # BATCH 32 -- THE FORCE FOLLOWS THE WEIGHT.
        # em31 migrated the momentum WEIGHTS to theta and left the network
        # pressure on solidFraction(). The skeleton row of the 2x2 block then
        # carried -phi_s*grad(p) with phi_s = theta while F_s carried
        # -grad(phi_s p_n) with phi_s = solidFraction(): a force and a weight
        # disagreeing about the solid fraction inside ONE equation. The consumer
        # map ordered these sites BEFORE the weights for this reason; they were
        # done after, so this closes the gap.
        #
        # ONE KEY: phase_force_basis legacy -> content. It moves
        # networkPressure()'s argument and the ISOTROPIC elastic term. THE
        # DEVIATORIC WEIGHT DOES NOT MOVE -- max(chi,1e-4) reaches 1.0022 where
        # theta reaches 0.450977, so substituting there rescales the deviatoric
        # stress by theta_ref and drops the floor. That killed 20260905-191135.
        #
        # PRE-REGISTERED from em31's OWN fields through the cubic, not estimated:
        #  (1) PHASE_CLOT_PHASE_FORCE_CONFIG basis=content, else void not null.
        #  (2) pnMin and pnTermMin move LESS THAN 0.05%. The sharp test: the
        #      cubic p_n = C*phi_s^2*(phi_s-phi_s0) has its extremum at
        #      2*phi_s0/3 = 0.3, a STATIONARY point, and that is where |p_n| is
        #      largest -- so a 7e-03 shift in phi_s has zero first-order effect.
        #      Measured max|p_n| ratio between bases: 1.0000.
        #  (3) pnMax FALLS ~0.32% (measured 0.9968): 0.0045601 -> ~0.0045455.
        #      The positive branch sits near phi_s0 where the cubic is not
        #      stationary, which is why it moves and pnMin does not.
        #  (4) Integrated sum|p_n| +2.65%, sum|phi_s*p_n| +2.31%; up to 69% in
        #      individual cells near the zero crossing; 6 cells flip sign.
        #  (5) phiSMax ~0.450977 (content) vs phiSLegacyMax ~0.450980 (legacy).
        #  (6) ALL FOUR GATES STILL PASS. None contains the network pressure; if
        #      one breaks, a force change reached the conservation structure.
        #
        # NOT pre-registered: maxVsOverU/clotCells/maxPhiS. em31 showed the
        # skeleton is DRAG-LOCKED (Kd ~2.36e8 vs As ~4.06e5), so predicting
        # percent motion there was em31's pre-registration error. Do not repeat.
        32: ("em32_stage3_contentforces", 0.25, 'eulerian, THROAT, two-velocity, STAGE 3b FORCE BASIS: networkPressure + isotropic elastic on theta. Read PHASE_CLOT_NETWORK first, then assess_stage2_baseline.py.'),
        # BATCH 33 -- THE INDICATOR, WHICH IS NOT THE FRACTION.
        # em31 moved the momentum WEIGHTS to theta and em32 moved the FORCE.
        # Both are volume fractions. The eleven inline chi = max(1-q,0) sites
        # are NOT: none of them scales by theta_ref, so they consume an
        # INDICATOR that is 1/0.45 times larger than the fraction.
        #
        # ONE KEY: phase_indicator_basis legacy -> content, i.e.
        # max(1-q,0) -> max(theta,0)/theta_ref at eight sites: the deviatoric
        # elastic weight, the damage driver, the carrier blend (unreachable on
        # this child), the Kres ramp, the isSkel clamp, and both bond holds.
        # Writing theta instead of theta/theta_ref would rescale all of them
        # by 0.45 -- a constitutive recalibration disguised as a
        # representation change. 29 guard tests, one of which deletes the
        # division and requires the guard to fail.
        #
        # THE RATIO IS NOT CLAMPED ABOVE (decision, 2026-09-15). Both
        # indicators already exceed 1; the overshoot is published rather than
        # hidden. The one site where ind > 1 is unsafe is the carrier blend --
        # (1-chif) < 0 SUBTRACTS the blood flux -- and it keeps its own
        # min(..,1). The lower guard at zero is a separate decision and stays.
        #
        # PRE-REGISTERED from em32's OWN written 0.0004 fields (q, thetaC,
        # thetaH, bondHold; 31360 cells). These are a basis swap WITHOUT
        # feedback, so (2)-(5) are sign-and-order with tolerances, (1) is hard:
        #  (1) PHASE_CLOT_PHASE_INDICATOR_CONFIG basis=content, else VOID.
        #  (2) indMax ~1.002172, chiLegacyMax ~1.002179, each within 1e-04.
        #      They differ by 6.9 ppm while the POINTWISE difference is
        #      1.556e-02 -- 2239x larger. Agreement of the maxima is NOT
        #      evidence the bases agree; that is why both are printed.
        #  (3) indAboveOne between 100 and 200 (144 content, 148 legacy). The
        #      legacy count equals qNegCells EXACTLY -- chi>1 iff c<-1 iff q<0
        #      -- an independent cross-check between two unrelated reports.
        #  (4) THE SHARPEST TEST, and the sets that actually move: kresCells
        #      FALLS ~34 of 31360 (27295->27261) and isSkelCells RISES ~27
        #      (4312->4339). I predicted both signs backwards from the lumen
        #      argument; the flips are at the diffuse EDGE where theta has
        #      compacted past chi*theta_ref. If neither count moves, the key
        #      never reached the safety nets and the run is void.
        #  (5) Deviatoric elastic weight +0.64% in L2. Bond hold barely moves
        #      (holdChi max ratio 0.999993, integrated 0.999736), so a large
        #      change in release timing is NOT the hold -- trace it to (4).
        #
        # FAILURE, not null: counts in (4) unchanged; indMax above 1.01 (the
        # unclamped ratio running away, decision needs retaking); any lumen
        # cell acquiring a nonzero indicator.
        33: ("em33_stage3_contentindicator", 0.25, 'eulerian, THROAT, two-velocity, STAGE 3c INDICATOR BASIS: the eight inline chi sites on theta/theta_ref. Read PHASE_CLOT_PHASE_INDICATOR first (indMax, indAboveOne, kresCells, isSkelCells), then assess_stage2_baseline.py.'),
        # BATCH 34 -- FIRST STEP TOWARD RETIRING c. INSTRUMENT ONLY.
        # CONFIGURATION IDENTICAL TO em33: not one dictionary key differs.
        # The solver gains PHASE_CLOT_SURFACE_DELTA, a passive report of the
        # theta-derived surface delta beside the c-derived one. Nothing
        # consumes it.
        #
        # WHY. Retiring c needs a surface delta made from theta, and it is
        # NOT optional: BOTH growth paths are functions of 0.5*|grad c| --
        # the Stefan front (0.5*Vn*|grad c|, grows c) and the accepted
        # platelet transfer (bindRate = 0.5*|grad c|*kEff, grows theta and is
        # what G1/G2b gate). The design first claimed the delta could be
        # dropped for a volumetric source; plateletContent.H's contract says
        # otherwise and the claim was corrected before anything used it.
        #
        # THE FORM. INT delta dn must be 1. For c running -1..+1, 0.5*|grad c|
        # is exact. For theta running 0..theta_bulk,
        #   INT |grad theta|/theta_ref dn = theta_bulk/theta_ref = indMax,
        # a number already printed every step -- so the delta's calibration
        # error needs no new instrument to watch. Width-insensitive, because
        # INT |grad theta| dn = theta_bulk however smeared the profile is.
        # fvc::grad, never a hand finite difference (one was wrong by 2x).
        #
        # PRE-REGISTERED:
        #  (1) deltaRatio = deltaThetaInt/deltaCInt within 1% of indMax
        #      (em33 measured 1.00217). Both integrals are areas and both
        #      integrate to the clot surface. Far from indMax falsifies the
        #      delta argument and S3 does not proceed.
        #  (2) EVERY em33 NUMBER REPRODUCES -- indMax 1.00217179625,
        #      chiLegacyMax 1.00217874529, indAboveOne 144, kresCells 27261,
        #      isSkelCells 4339, maxVsOverU 1.21866559545, clotCells 2586,
        #      qNegCells 148. Any of them moving means the 'diagnostic-only'
        #      change was not, and that is the more important test here.
        #  (3) Gates unchanged; G6 still first-step-only on nP.
        #
        # FAILURE, not null: any em33 number moves; deltaCInt <= 0;
        # deltaRatio = -1 (no content model, instrument measured nothing).
        34: ("em34_surface_delta_instrument", 0.25, 'eulerian, THROAT, two-velocity, STAGE 4a INSTRUMENT ONLY: theta-derived surface delta reported beside the c-derived one, config identical to em33. Read PHASE_CLOT_SURFACE_DELTA (deltaRatio vs indMax) first, then confirm every em33 number reproduces.'),
        # BATCH 35 -- S1b. THE SAME CHILD AGAIN, BECAUSE THE INSTRUMENT WAS
        # WRONG AND THE TRAJECTORY IS ALREADY KNOWN EXACTLY.
        # CONFIGURATION IDENTICAL TO em33 AND em34: not one key differs.
        #
        # WHY RE-RUN A CHILD THAT ALREADY PASSED. Batch 34 passed its gate and
        # the gate was the wrong one. deltaRatio -- the ratio of the two
        # DOMAIN INTEGRALS -- landed at 0.27% of indMax and held for all 272
        # steps, so the TOTAL binding flux is preserved to 0.05%. Meanwhile
        # the DISTRIBUTION of that flux had already moved 2.2% in the mean and
        # 15.6% at p99, growing as t^1.83, measured from the written fields
        # after the fact. An integral cannot tell a conserved total from a
        # conserved distribution, and the design document said so one
        # paragraph before the gate was set on it.
        #
        # WHAT IS NEW. deltaTV: half the L1 distance between the two deltas
        # AFTER EACH IS NORMALISED TO UNIT MASS -- the fraction of the binding
        # source that relocates under the substitution, bounded [0,1].
        # deltaShiftMag/X/Y (with rmsC as its scale) and supportRatio
        # decompose a failure into MOVED versus SPREAD. deltaTheta is now a
        # WRITTEN FIELD beside interfaceDelta, so the next question can be put
        # to this run instead of costing another one.
        #
        # THE TWO THINGS THIS 5-MINUTE RUN BUYS.
        #  (1) THE GATE'S FLOOR. theta and c are seeded consistently, so
        #      deltaTV at the first write IS the instrument's resolution --
        #      two gradient operators on two fields describing one interface.
        #      The gate for the long run is TVbar < 0.1 (an order of magnitude
        #      below the 1.597x growth-rate defect the theta-native programme
        #      exists to remove). If the floor is within a DECADE of 0.1 the
        #      gate cannot fail and must be rebuilt before anything is read
        #      from it. Expected floor 1e-03 or below.
        #  (2) THAT A WRITTEN FIELD IS STILL DIAGNOSTIC-ONLY. Batch 34 proved
        #      bit-exactness across 15245 emitter lines and 272 steps. This
        #      adds an AUTO_WRITE field and mesh.C() arithmetic, so the same
        #      comparison must come back bit-exact AGAIN, against numbers now
        #      known to nine significant figures:
        #        indMax 1.00217179625   chiLegacyMax 1.00217874529
        #        indAboveOne 144        kresCells 27261
        #        isSkelCells 4339       maxVsOverU 1.21866559545
        #        clotCells 2586         qNegCells 148
        #        deltaCInt 9.64101000119e-11  deltaRatio 0.999457390128
        #
        # FAILURE, not null: ANY of those numbers moving; deltaTV negative or
        # above 1; deltaTV at the first write above 1e-02; deltaShiftMag or
        # supportRatio returning the -1 sentinel while deltaRatio does not.
        35: ("em34_surface_delta_instrument", 0.25, 'eulerian, THROAT, two-velocity, STAGE 4b INSTRUMENT ONLY: the surface-delta instrument rebuilt on a DISTRIBUTION statistic after the integral gate passed while the distribution moved. Read deltaTV first -- its FIRST value is the gate floor, not a result -- then confirm every batch-34 number reproduces bit-exactly.'),
        # BATCH 36 -- S2. ONE KEY: residual_chi 1.0e-02 -> 2.222e-04.
        # Nothing else differs from em34.
        #
        # WHY THIS VALUE. residualChi should mark the indicator at which the
        # skeleton row stops meaning anything -- where phi_s = residualPhiS,
        # i.e. ind = 1e-4/0.45 = 2.222e-04. The incumbent 1e-02 is the
        # chi-support the phi_s0 = 0.01 runs happened to have (1e-4/0.01) and
        # was never re-derived when the anchor moved to 0.45. THE PEAK IS
        # UNCHANGED at residualPhiS/deltaT = 68 1/s: rscale is exactly the
        # factor that makes it so. Region changes, strength does not.
        #
        # PRICED OFFLINE, no run spent deciding it, and the reconstruction was
        # VALIDATED FIRST -- rebuilt from run 20260916-095318's written
        # thetaC/thetaH it reproduces the emitted kresCells 27261 and
        # isSkelCells 4339 exactly, to the cell.
        #
        # READ IN THIS ORDER.
        #  (1) PHASE_CLOT_PHASE_INDICATOR at the FIRST STEP: kresCells must be
        #      EXACTLY 25835 and isSkelCells EXACTLY 5796. These are
        #      predictions of the KEY, made before the trajectory can diverge.
        #      Off by one cell and the offline pricing is wrong -- stop and
        #      re-derive rather than reading anything else.
        #  (2) PHASE_CLOT_SKELETON_VELOCITY, but read maxVs and maxU, NOT
        #      maxVsOverU. maxVsOverU/meanVsOverU/exceedFraction are computed
        #      over CLOT CELLS ONLY (c < 0 and drag > 1), and 0 of the 1426
        #      cells this change exposes are clot cells -- their c runs 0.980
        #      to 0.9996. Those three are STRUCTURALLY BLIND to this run and
        #      the design's original gate for S2 could not have failed.
        #      Whole-field maxVs/maxU is EXACTLY 1.000000000000 today, which is
        #      the correct regularised state: in the lumen the skeleton
        #      advects with the carrier and nothing else. It must stay there.
        #  (3) maxVsInClot must stay 0.
        #
        # THE EXPOSED CELLS WERE NOT BEING HELD UP BY Kres: the 1426 losing it
        # draw 4.4% (median) to 26.4% (max) of their damping from it. The 25835
        # that KEEP it draw a median 1.000000 of theirs from it -- Kd is ~0
        # there. That is the lumen, and it keeps its regulariser. Projected:
        # the band's |Vs| rises 0.4013 -> ~0.5156 of whole-field maxVs (upper
        # bound 0.5451), so it cannot become the whole-field maximum.
        #
        # FAILURE, not null: either cell count off by one; maxVs/maxU above
        # 1.01; maxVsInClot nonzero. A rise means the regularisation WAS doing
        # load-bearing work in the band and reading A was right.
        #
        # RESIDUAL RISK: rscale 0.01 -> 0.45, so the ramp is 45x steeper and
        # Kres varies 45x more sharply across the band. It sits in the 2x2
        # block diagonal; the detMin <= 0 tripwire is a FatalError, so a
        # conditioning failure stops the run rather than corrupting it.
        36: ("em35_s2_residualchi", 0.25, 'eulerian, THROAT, two-velocity, S2: residual_chi 1.0e-02 -> 2.222e-04, re-derived on the content basis. Read kresCells/isSkelCells at the FIRST step (exactly 25835 / 5796), then whole-field maxVs/maxU -- NOT maxVsOverU, which is clot-cell-only and cannot see this change.'),
    # BATCH 37 -- S3. ONE KEY: phase_delta_basis legacy -> content, i.e.
    # interfaceDelta_ switches from deltaC_ = 0.5*|grad c| to
    # deltaTheta_ = |grad(thetaC+thetaH)|/0.45. Nothing else differs from em35.
    #
    # THE KEY NOW DEFAULTS TO 'content' -- in the solver AND in
    # create_cases.py, which had to move with it because the generator writes
    # every basis key unconditionally and the solver default is otherwise
    # never reached. em35's JSON entry was given an explicit
    # phase_delta_basis 'legacy' so its definition keeps describing the run
    # that exists. Each run states its own value in
    # PHASE_CLOT_PHASE_DELTA_CONFIG; that line is the authority, not the JSON.
    #
    # READ IN THIS ORDER.
    #  (1) PHASE_CLOT_PHASE_DELTA_CONFIG must say basis=content. Absent or
    #      legacy -> the run is VOID, not null. Read nothing else from it.
    #  (2) PHASE_CLOT_SURFACE_DELTA deltaTV at 4e-04 must still be NONZERO
    #      (> 1e-03). Exactly 0 means the instrument got rewired onto
    #      interfaceDelta_ and is comparing the selected delta with itself --
    #      a triumphant pass measuring nothing. deltaC is written as its own
    #      field precisely to prevent this.
    #  (3) dBSum against em35 AT MATCHED TIME. 4e-04: 1.003586 (band
    #      1.0025-1.0047). 5e-05: 0.999952 (band 0.9995-1.0004). The SIGN
    #      CHANGES between them and that is the prediction, not an artefact.
    #  (4) stoichNorm at 4e-04 must MOVE, 0.992064 -> 0.992598 +/- 0.0003.
    #      Unchanged means the key reached one consumer and not the other.
    #  (5) NChange + dBSum at machine zero on every step, VnClosureResidual
    #      exactly 0. These are the conservation statements and they must not
    #      move at all.
    #
    # WHY THE DESIGN'S OWN GATE WAS RETIRED. It said the deposition changes by
    # no more than deltaRatio predicts. deltaRatio is 0.999457 -- a FALL of
    # 0.054%. Priced from em35's written fields the deposition RISES 0.359%:
    # opposite sign, 6.6x the size. Measured decomposition of the weight at
    # 4e-04: unweighted 0.999457, by w 0.999462, by epsP 0.999401, by nP
    # 1.003584, by N = epsP*nP 1.003557. nP carries the whole flip and epsP
    # pushes the other way. The theta delta is 3.1% wider, so it reaches into
    # blood that is less platelet-depleted.
    #
    # THE RECONSTRUCTION WAS VALIDATED FIRST, against four numbers em35's own
    # log printed at 4e-04: deltaRatio 0.999457383912 vs 0.999457384, deltaTV
    # 0.0155714061689 vs 0.0155714, supportRatio 1.03134485244 vs 1.031345,
    # bindRateClotFrac 0.476442070955 vs 0.476444.
    #
    # REPORTED, NOT GATED: clotGrowth, chiVolume, contentVolume, equivMax,
    # thetaMax, indMax, maxVsOverU, cMin, qNegCells. Predicting them needs
    # re-solving Wp_, whose sink coefficient moves -44% to +24% per cell.
    # 0.1-0.4% movement in clotGrowth is expected and is NOT a failure;
    # several percent is.
        37: ("em36_s3_deltabasis", 0.25, 'eulerian, THROAT, two-velocity, S3: the surface delta moves onto theta. ONE KEY from em35. Read PHASE_CLOT_PHASE_DELTA_CONFIG basis=content FIRST, then deltaTV nonzero, then dBSum against em35 at MATCHED time (1.003586 at 4e-04, 0.999952 at 5e-05 -- the sign changes and that is the prediction).'),
    # BATCH 38 -- S4. THE STEFAN FRONT STOPS GROWING THE CLOT.
    #
    # ONE KEY from em36: phase_growth_driver front -> transfer. The term
    # -Vn*phaseGradient leaves all three c branches and c receives the SAME
    # increment theta receives, dc = -(2/theta_ref)*dtheta, applied off the
    # same array at the same commit point. This is a MODEL change, not a
    # discretisation one: after it there is one accountant for the clot
    # instead of two that disagree by 2*alpha/D = 1.61x.
    #
    # READ ORDER, AND THE FIRST TWO ARE NOT THE RESULT.
    #  (1) PHASE_CLOT_GROWTH_DRIVER driver=transfer at startup. Absent, or
    #      driver=front, and the run is VOID rather than null -- it is em36
    #      again under a different name.
    #  (2) clotResidual in PHASE_CLOT_MASS at the FIRST step. Near 1e-20 is
    #      right; near 1e-15 means clotGrowth was not redefined with the
    #      source and every growth number in the run is meaningless.
    #  (3) G1, and it is EXACT rather than an estimate: clotDVdt at
    #      t = 1.470588e-06 must be 2.536982212e-15 against em36's
    #      3.580174910e-15, ratio 0.708620, band +/-0.1%. On step 1 the
    #      transfer's inputs are bit-identical between the two runs, because
    #      interfaceDelta_ is built from theta (S3) and the masks are still on
    #      the initial field.
    #  (4) |chiVolume - contentVolume/theta_ref| at 4e-04 <= 5.0e-20 against
    #      em36's 5.3790e-19. The structural gate; a wrong conversion factor
    #      fails it by 2x or more.
    #
    # DO NOT READ THE 0.62x OF S1b AS A GATE. That ratio is NOT CONSTANT:
    # 0.708620 at step 1 falling to 0.554268 at 4e-04, mean 0.626403. S1b's
    # 0.62 +/- 0.05 came from a trajectory fit and the LAST STEP IS OUTSIDE
    # IT, so reading it at the end of the window would fail a correct run.
    # It falls because the front is inexhaustible (+0.035% over em36's
    # window) and the transfer eats what it binds (nMin -32.74% while nMax
    # holds exactly at the inlet value). That decay IS the result.
    #
    # DO NOT GATE ON stoichNorm. OI-12: it held to 0.054% across em36 while
    # the committed transfer fell 21.76%, because plateletUptake has no N in
    # it. This package adds plateletUptakeCommitted beside it; the GAP between
    # the two ratios is the depletion.
    #
    # COST: one child, same 4e-04 s horizon and same ~272 steps as em36,
    # which took 192 s of solver time. 0.25 h is the same allowance em36 ran
    # under and is an ALLOWANCE, not a measurement for this batch.
    38: ("em37_s4_growthdriver", 0.25, 'eulerian, THROAT, two-velocity, S4: the front stops growing the clot; the accepted transfer is the only growth. ONE KEY from em36. Read PHASE_CLOT_GROWTH_DRIVER driver=transfer FIRST, then clotResidual at step 1 (1e-20, not 1e-15), then clotDVdt at t=1.470588e-06 (2.536982212e-15, exact, +/-0.1%).'),
    39: ("em38_s5_maskbasis", 0.25, 'eulerian, THROAT, two-velocity, S5: the release and bond-hold masks read phaseIndicator() instead of c. NO KEY CHANGES -- config is bit-identical to em37 and only the solver differs. Read indBasis=content and indSupportLimit=0.25 in PHASE_CLOT_RELEASED FIRST: if the package did not pick the new solver up, every other number simply reproduces em37 and looks like a pass. Then supported at 4e-04 (2953 vs em37 2942, band 2948-2958). clotCells moves 1 cell and is reported, not relied on.'),
    # BATCH 26 -- ISOLATION, AND THE MATRIX THAT KILLED 13 OF 14 RUNS.
    # Batch 25 bundled FOUR changes and could attribute nothing, and both its
    # children were rejected by the returned assessor. These three change ONE
    # key each, verified by a full recursive diff of the generated cases:
    #   em26_blend_off vs em26_blend_on differ in child_id and
    #   materialCarrierBlend no|yes and NOTHING ELSE.
    #   em26_transport_only differs from blend_off in alpha (8.1522e-11 -> 0)
    #   and growth_rate_multiplier only, so ANY qNegCells > 0 in it is
    #   transport-driven and cannot be blamed on growth or the cap. Codex
    #   proved that possible algebraically -- ddt(c)+div(cv) = div(v)-2G has
    #   NO fixed point at c = -1 under compression -- and nobody has ever
    #   measured it in this code. Decisive either way.
    #
    # HORIZON SET BY MEASUREMENT, NOT BUDGET. endTime 4e-04 s. The first raw
    # crossing is at 7.4627e-06 s (step 5); the first SIGNIFICANT one,
    # qNegMin < -1e-3, is at 2.6866e-04 s, identically in BOTH Batch 25
    # children. The 50 us window originally proposed would have seen the raw
    # crossing and nothing else and been misread as a null result.
    #
    # READ PHASE_CLOT_SHELL_MATRIX FIRST. New in package 53c4edb6. It is the
    # final assembled system -- diag + boundary internalCoeffs, off-diagonals,
    # source and psi -- i.e. the matrix symGaussSeidelSmoother actually
    # divides by. PHASE_CLOT_CARRIER reads shellEqn.diag() BEFORE solve() and
    # cannot see any of it. zeroDiagCells or any nonFinite* > 0 now FAILS the
    # child by gate: that is the SIGFPE caught as a measurement, not a crash.
    #
    # COST: ~268 steps per child at Batch 25's realised dt of 1.4925e-06 s,
    # against its 20,099 steps for 30 ms -- about 75x shorter. At its measured
    # ~0.585 s/step that is ~2.6 min per child, ~8 min for the three serial.
    # ESTIMATE, from Batch 25's rate; not a measured cost for this batch.
    26: ("em26_blend_off em26_blend_on em26_transport_only", 0.15, 'eulerian, THROAT, two-velocity, ISOLATION: one key each; blend ON/OFF and a growth-free transport-only control; PHASE_CLOT_SHELL_MATRIX is new'),
    # BATCH 40 -- THE OI-6 REPAIR AND S6a, AS THREE ONE-KEY DIFFS.
    #
    # READ em39 FIRST, AND READ IT FOR A NONZERO. PHASE_CLOT_MMC measures
    # |thetaFaceMom - thetaFaceMass|, and under phaseMomentumFluxBasis
    # `shared` it is zero BY CONSTRUCTION -- the two sites are then literally
    # the same expression. A pair with only em40 in it could not tell a
    # working repair from a diagnostic that can only ever print 0. em39
    # exists to make the gate failable: it must report mmcMax ~ 2.8e-02.
    # A zero there stops the read; nothing downstream means anything.
    #
    # THE CHAIN, each diff verified by recursive diff of the generated cases:
    #   em38            -> em39  two new keys at their defaults; nothing else.
    #                            Bit-identical to em38 except the three new
    #                            diagnostic blocks (MMC, QBASIS, COST).
    #   em39            -> em40  phaseMomentumFluxBasis legacy -> shared.
    #   em40            -> em41  phaseFluidFractionBasis legacy -> content.
    #
    # PRE-REGISTERED, from em38's own written fields (offline, face-by-face
    # off constant/polyMesh/owner+neighbour):
    #   em39  mmcMax = 2.654502e-02 at t=2e-04 and 2.807140e-02 at t=4e-04,
    #         EXACT -- the statistic is carrier-independent. mmcMeanRel in
    #         [0.0158, 0.0164]. dqMax 0.004393 / 0.015478, dqAbove1em2 0 / 342.
    #   em40  mmcMax = 0.0 EXACTLY, both times. Anything else means the
    #         weights did not reach the momentum flux.
    #   em41  q moves by exactly that pre-registered offset and no more.
    #
    # S6a IS A REPRESENTATION SWAP, NOT AN IMPROVEMENT. q stops being
    # 0.5*(1+c) and becomes 1 - (thetaC+thetaH)/theta_ref. Both forms are
    # built every step on EITHER basis, so em40 reports what em41 will do
    # before em41 does it. A narrower or wider clot in em41 is NOT a verdict:
    # the OI-6 criterion is about the momentum flux and says nothing about
    # which theta profile is correct. c is still solved and still written.
    #
    # COST: em38 ran 287 s of solver time over the same 4e-04 s horizon; the
    # three new diagnostics are per-step reductions and cost nothing
    # measurable. Three children at CONCURRENCY 2 is two waves. 0.4 h is an
    # ALLOWANCE derived from em38's realised cost, not a measurement.
    40: ("em39_oi6_fluxlegacy em40_oi6_fluxshared em41_s6a_qfromtheta", 0.4, 'eulerian, THROAT, two-velocity. OI-6 repair + S6a, three one-key diffs. Read PHASE_CLOT_MMC in em39 FIRST and require mmcMax ~ 2.8e-02: the diagnostic is identically zero under `shared`, so em39 is the only thing that proves it can fail. Then em40 mmcMax = 0.0 exactly. Then em41 PHASE_CLOT_QBASIS dqMax 0.015478 at 4e-04 with 342 cells past 1e-2. S6a is a representation swap -- a width change in em41 is not a verdict.'),
    #
    # BATCH 41 -- S6b, AND IT IS TWO KEYS BECAUSE THE DESIGN'S ONE DID NOT
    # REACH THE DEFECT.
    #
    # The S6 scoping document specified a single step, "phaseWeights_ off c",
    # and gave it Gate A-exact on |Wp - q|. Batch 40's own logs say that
    # cannot work. Wp = w*q, so at saturation the content must EQUAL the
    # fluid fraction -- and that survives discretisation only when Wp's
    # convection operator is the operator that DEFINES q. Before S6a,
    # q = 0.5*(1+c) and Wp rode c's weights, so it did. After S6a,
    # q = 1 - theta/theta_ref and theta rides FIRST-ORDER UPWIND (a Stage 1
    # contract, field-independent on purpose) while Wp still rides a limited
    # scheme. PHASE_CLOT_PLATELET WpRawQDevMax at t = 4e-04 s:
    #
    #     em39 / em40   q from c, same operator as Wp      1.643e-03
    #     em41          q from theta, Wp still limited     1.584e-02   (9.6x)
    #
    # and 1.584e-02 IS the c/theta divergence (dqMax 1.5479e-02): Wp is
    # tracking the field q no longer comes from. Changing which field the
    # LIMITER reads does not touch that, so the step is split.
    #
    #   em41  RE-RUN AS THE CONTROL, and read it first. Same dictionary,
    #         new binary. It must reproduce the returned em41 of run
    #         20260919-213132 bit for bit -- WpRawQDevMax 1.584e-02, dqMax
    #         0.0154792016639, dqAbove1em2 343, facesUpwinded 3651 at step 1,
    #         MMC zero in 272/272. Anything else and the two new keys are not
    #         the no-op at their defaults that they are supposed to be, and
    #         em42/em43 are unreadable. This is the em39-against-em38 check
    #         one stage on, and it is the only thing that can catch a legacy
    #         path perturbed by the edit.
    #   em42  phaseLimiterBasis content. The limiter moves to the UNCLAMPED
    #         (thetaC+thetaH)/theta_ref. Frees the weights from c -- which is
    #         what actually blocks S6c -- and reaches three equations: c,
    #         shell_ and Wp. It must NOT fix WpRawQDevMax: pre-registered
    #         1.0e-02 to 2.5e-02, em41's order and not em39's. Its FIRST step
    #         must reproduce em41's exactly (facesUpwinded 3651,
    #         WpRawQDevMax 5.02236272326e-06 to 10 digits), because at step 1
    #         dqMax is 3.33e-16 and vanLeer's r is affine-invariant.
    #   em43  phasePlateletFluxBasis content. Wp takes contentFaceWeights_ --
    #         the face values theta actually transported with. THE NUMBER:
    #         WpRawQDevMax must fall to em39's order, pre-registered 5.0e-04
    #         to 5.0e-03. The floor is NOT zero and Gate A-exact does not
    #         apply here: em39's operators are identical by construction and
    #         it still reports 1.643e-03, which is the adhesion sink and the
    #         explicit laplacian. Below 1.0e-05 means Wp has been aliased to
    #         q, not solved.
    #
    # WHAT THIS BUYS AND WHAT IT COSTS. Wp's convection goes from limited to
    # first-order upwind. More numerical diffusion in the platelet content is
    # the price of the identity -- a smoother Wp is not evidence of anything.
    # shell_ deliberately does not move: it is a label on pathlines with no
    # Wp = q identity, and upwinding it would smear the core-shell boundary
    # and its two decades of permeability contrast.
    #
    # COST: em41 realised 349.8 s of solver time over this 4e-04 s horizon on
    # the batch-40 binary. Three children at CONCURRENCY 2 is two waves. 0.4 h
    # is an ALLOWANCE from that realised cost, not a measurement.
    # BATCH 42 -- S6c-1, AND IT DECIDES WHETHER c CAN BE DELETED AT ALL.
    #
    # The S6 scoping doc gave the deletion a bit-identity gate on the premise
    # that "after S6a and S6b nothing but diagnostics reads c". THE PREMISE IS
    # FALSE. updateDerivedFields() builds, with no key at all:
    #
    #     normal_ = grad(c)/max(|grad c|, gradFloor)
    #
    # and normal_ reaches the physics by one path that IS reachable here --
    # shearPa_ -> shearRate_ -> adhesionEnhancement_ -> adhesionSpeed_ ->
    # adhesionSpeedEff_ -> Vn_, the growth rate. The solver's own config line
    # has said "phaseNormal(stays-on-c-until-S5)" since S3; S5 ran and did not
    # take it. Enumerated from source against em43's own dictionary in
    # research/designs/2026-09-20-S6c-normal-basis-and-deletion.md.
    #
    #   em44  ZERO KEYS against em43. Its dictionary is byte-identical to
    #         em43's apart from the new phaseNormalBasis legacy line, VERIFIED
    #         at build time by diffing the generated phaseClotProperties. It
    #         must reproduce em43 bit for bit once the new
    #         PHASE_CLOT_NORMAL_BASIS lines and the two amended *_CONFIG
    #         strings are normalised away. What it adds is the INSTRUMENT.
    #   em45  phaseNormalBasis content. normal_ becomes
    #         -2*grad((thetaC+thetaH)/theta_ref)/max(|.|, gradFloor).
    #
    # THE NUMBER IS dAdhesionSpeedMax, and read selfCheck BEFORE it. The
    # instrument re-derives the adhesion speed rather than reading it off the
    # model, so selfCheck = max|recomputed(active basis) - adhesionSpeed_|
    # must be 0 on every line; nonzero means that copy has drifted and every
    # other figure on the line is uninterpretable.
    #
    #   dAdhesionSpeedMax = 0 at every step  -> the clamp holds, the swap is
    #     inert on this configuration, and S6c-2 deletes c under a TRUE
    #     bit-identity gate.
    #   nonzero -> the offset is priced here instead of surprising the
    #     deletion run. Not a failed run; it is the answer.
    #
    # WHY IT MIGHT BE ZERO: adhesionEnhancement_ = 1 + 19*min(max((shearRate
    # - 3000)/5000, 0), 1), so while shearRate < gammaLow = 3000 the
    # enhancement is EXACTLY 1 on any normal whatsoever. Whether that holds on
    # the current parameters was never measured -- the 782 1/s figure behind
    # the 2026-09-16 deferral is from an older run, and em43 neither writes
    # nor logs shearRate. cellsAboveGammaLowL/C report it directly.
    #
    # AT STEP 1 THE TWO CHILDREN MUST AGREE. Both build both normals, so
    # dNormalMax, dShearPaMax and dAdhesionSpeedMax are shared-state
    # quantities there and must match to full printed precision. This is the
    # counterpart of batch 41's facesUpwinded miss: that counter differenced
    # two fields and the step replaced one of them, so it could never have
    # held. These difference two fields the key SELECTS BETWEEN and modifies
    # neither.
    #
    # COST: em43 realised 282.6 s of solver time over this 4e-04 s horizon.
    # Two children at CONCURRENCY 2 is one wave. 0.3 h is an ALLOWANCE.
    42: ("em44_s6c1_normallegacy em45_s6c1_normalcontent", 0.3, 'eulerian, THROAT, two-velocity. S6c-1: the LAST live consumer of c. Read em44 FIRST and require it to reproduce em43 bit for bit -- it is a ZERO-KEY control and its dictionary is identical to em43\'s but for the new phaseNormalBasis legacy line. Then read selfCheck on PHASE_CLOT_NORMAL_BASIS: nonzero and nothing else on the line means anything. Then dAdhesionSpeedMax -- 0 at every step means deleting c is bit-identical in the physics and S6c-2 proceeds; nonzero is the priced offset, not a failure. Step 1 must agree between the two children to full printed precision.'),
    41: ("em41_s6a_qfromtheta em42_s6b_limitertheta em43_s6b_wpcontentflux", 0.4, 'eulerian, THROAT, two-velocity. S6b in two keys. Read em41 FIRST and require it to reproduce run 20260919-213132 bit for bit -- it is the control for the edit itself. Then em42 must NOT fix WpRawQDevMax (1.0e-02 to 2.5e-02, em41\'s order): the limiter field was never the mechanism. Then em43 must bring it to em39\'s order, 5.0e-04 to 5.0e-03. Below 1e-05 is Wp aliased to q, not a result. MMC must stay 0 in 272/272 throughout.'),
    43: ("em46_s6c2_deleted", 0.15, 'eulerian, THROAT, two-velocity. S6c-2: c, muC, mobility, epsilonC and deltaC are DELETED from the solver -- 72 sites, 19 functions. NO KEY CHANGES: em46 constant/phaseClotProperties, system/fvSchemes, system/fvSolution and system/controlDict were diffed byte for byte against the ones em45 actually ran and are IDENTICAL, so the control is not a sibling child but the ARCHIVED run 20260920-060033 on package 474b91fc. Read PHASE_CLOT_CONTENT_BALANCE FIRST -- if it is absent the package did not pick the new solver up and every other number reproduces em45 and looks like a pass. Then require contentResidual to hold its drift under 5%. Then the physics: every surviving PHASE_CLOT_ tag except COST must be md5-identical to em45, because nothing live read c by the time it was deleted. Anything that moves means a consumer was missed, and the movie is not the evidence -- the log is.'),
    # BATCH 44 -- S7 VERIFICATION, AND IT IS A ZERO-KEY CONTROL.
    #
    # WHAT S7 CHANGED IN THE SOLVER, and it is only two things. Three
    # lookupOrDefault<word> defaults moved legacy -> content
    # (phaseFluidFractionBasis, phaseLimiterBasis, phaseNormalBasis),
    # because S6c-2 refused five values and left three of them as the
    # fallback, so omitting a key selected a fatal. And the
    # refuseDeadValue lambda gained the switch and the requirement string,
    # because its message was telling 185 cases to set a value that is
    # also refused. No equation, no field, no scheme, no dictionary key
    # that em46 reads.
    #
    # WHY em46 IS THE CONTROL, AND WHY IT IS THE ONLY ONE WORTH RUNNING.
    # A default is read only when the key is ABSENT, and create_cases.py
    # writes all five refused keys into every child. So the three flips
    # cannot reach em46 -- verified rather than reasoned: em46's generated
    # constant/phaseClotProperties, system/fvSchemes, system/fvSolution
    # and system/controlDict are BYTE-IDENTICAL (cmp) to the four files
    # run 20260920-235402 actually ran, and constant/ differs only by
    # polyMesh, which blockMesh writes at run time.
    #
    # Same dictionary, new binary. That is the whole test, and no sibling
    # child can sharpen it.
    #
    # THE PRE-REGISTRATION, against run 20260920-235402 (package d213d440):
    #
    #   exit 0, 272 steps, endTime 4e-04 reached, ALL_CHILDREN_ADMISSIBLE,
    #     MAIN_CASE_COMPLETE.
    #   57 of 58 PHASE_CLOT_ tags md5-identical line for line. COST is
    #     wall clock and is the one exception. Between the previous two
    #     runs four tags moved -- DAMKOHLER, GROWTH_STOICH, CONTENT_BALANCE
    #     and COST -- and all four moved for reasons S7 does not touch, so
    #     54 of 58 becomes 57 of 58 here, not 54.
    #   all 174 members of the written-field archive byte-identical at all
    #     8 solver write times plus t=0, none added and none removed.
    #   contentResidual worst-case still ~4.3e-22 a step;
    #     clot_drift_fraction still ~-1.2e-11; mass_gate_ok true.
    #
    # WHAT A FAILING RUN LOOKS LIKE -- stated before reading one, because
    # a gate nobody can describe failing is a gate that cannot fail:
    #
    #   (a) any PHASE_CLOT_ tag except COST differs. Then the lambda
    #       rewrite reached the constructor, or a default is being read at
    #       a site the generator does not write, and the S7 claim that the
    #       migration is inert where keys are explicit is false.
    #   (b) construction aborts naming a refused value. Then the rewrite
    #       changed WHICH values are refused rather than only the message.
    #       em46 names the surviving value for all five keys, so this can
    #       only happen if the refusal predicate inverted.
    #   (c) it completes and reports a number nobody predicted. Read the
    #       log. The movie shows a field and is never a gate status.
    #
    # WHAT THIS BATCH CANNOT TEST, recorded rather than glossed. The
    # refusal MESSAGE fires only on an archived case, and archived cases
    # are by definition never run, so no Colab session reaches it. It is
    # held offline instead: check_refused_dictionary_values.py derives
    # each call site's switch and requirement string and compares them
    # against the same precondition table it prices cases with, with a
    # mutation that points one site at the wrong switch.
    #
    # AND THE LADDER HAS COLLAPSED -- read this before planning batch 45.
    # Hashing every generated input file, the 18 runnable children are
    # only EIGHT distinct configurations. em43 = em44 = em45 = em46 byte
    # for byte, and em35 = em36 = em37 = em38 = em39. Every rung was a
    # one-key diff and S6c-2 deleted the keys those diffs named; S7's
    # migration of em44 off phaseNormalBasis legacy closed the last pair.
    # They still generate and their old batches still name them, but none
    # of them is a distinct experiment any more. The next real experiment
    # is the 30 ms horizon, not another rung.
    #
    # COST: 285 s realised on run 20260920-235402. One child, so
    # CONCURRENCY is irrelevant. 0.15 h is that cost with headroom.
    44: ("em46_s6c2_deleted", 0.15, 'eulerian, THROAT, two-velocity. S7 VERIFICATION and a ZERO-KEY control: em46\'s four input dictionaries are byte-identical to the ones run 20260920-235402 ran, so this is the same dictionary on a new binary. Require 57 of 58 PHASE_CLOT_ tags md5-identical (COST is the exception) and all 174 written-field members byte-identical. Anything else that moves means S7 reached the constructor. Read the log, not the movie.'),
    45: ("em47_horizon_probe", 0.85, 'eulerian, THROAT, two-velocity. THE HORIZON PROBE, and it is staged INSTEAD OF the 3e-2 s run, not before it as a formality. ONE KEY against em46: end_time_s 0.0004 -> 0.004. Cost is NOT the reason -- dt is pinned at 1.4706e-06 by the ELASTIC Courant limit (Co max 0.0279 against maxCo 0.5, 18x headroom), so 3e-2 s is 20400 steps / ~5.8 h and will not inflate. The reason is that 20260921-025148 ended FOUR STEPS into a regime change: DEBOND.utilisation sat at +1.2/s for 200 steps then turned to +553/s at t = 3.9559e-04, the same step PLATELET.wMin hit its whole-run minimum of -22.0857. Read qUnder FIRST. It rose 0 -> 2.17e-03 with d(qUnder)/dt STILL RISING at the last eighth, which puts the assessor field bound (cUnder = 2*qUnder > 0.01) at t <= 7.1e-04 and C_BOUND_LIMIT 0.05 at t <= 2.9e-03 -- 2.4% and 9.6% of the 3e-2 s target. EXPECT INADMISSIBLE on \'phase out of bounds\'; that is the prediction landing, not the run failing, and the log is written per step whatever the grade. The outcome worth hoping for is qUnder NEVER reaching 0.005 in 4e-03 s, which green-lights 3e-2 s and makes me wrong.'),
    46: ("em48_polecubic_30ms", 7.0, "eulerian, THROAT, two-velocity. THE 30 ms RUN, on the repaired compaction law. TWO KEYS against em47_horizon_probe: end_time_s 0.004 -> 0.03 and network_pressure_model cubic -> poleCubic (write_interval_s 5e-05 -> 2.5e-04 so 30 ms writes 120 directories, not 600). WHY THE LAW CHANGED: em47 reached phiSOverNominal 1.1913 at 4e-03 s and was still accelerating at +216/s. The SAME cubic reached endTime 0.03 s at 1.0524 in run 20260906-210336, so this is not a new defect -- k0p moved to Xu's values (1e-11 -> 1e-13 shell, 1e-14 -> 1e-15 core) and the Darcy load scales as 1/k. MEASURED in em47's written p field: 12.1 Pa across the clot against a cubic reply of 2.59 Pa. poleCubic is the cubic times (phiSMax-phi_s0)/(phiSMax-phi_s): identical slope at nominal (derived, not fitted), FOUR TIMES WEAKER at the diffuse fringe that killed poleAtMax, and divergent at close packing so the bound comes from a force balance. READ FIRST: PHASE_CLOT_CONFIG networkPressureModel (must say poleCubic, else VOID), then pnCapCells -- THE GATE THAT CAN FAIL, 0 means the force balance did the bounding and nonzero means the 0.99*phiSMax clamp did -- then phiSOverNominal at matched times against em47's 1.000473/1.003143/1.009304/1.025381/1.051664/1.191305 at t = 2,5,10,20,30,40 e-04. THE SHELL IS PREDICTED MARGINAL (G=4 Pa under 12 Pa should nearly close-pack); if pnCapCells fires there the finding is that G and k0p are not a self-consistent pair, NOT that the law failed. THE DELIVERABLE is whether the clot MOVES: em47 released 29 of 56 wall bonds with clot_drift_fraction -1.95e-09, which is why the movie showed no detachment."),
    47: ("em50_G4000", 28.4, "PAUSED 2026-09-22 ON ALIREZA'S INSTRUCTION -- DO NOT RUN. No stiffer G sweeps until the S10 root-cause fixes land (research/designs/2026-09-22-S10-root-causes-em49.md): em49 showed a G-proportional spurious wall traction at t=0 that would tear this child off on step 1 as well. eulerian, THROAT, two-velocity. DR LI'S STIFFNESS REQUEST, RUNG 2 -- NUMBERED BELOW RUNG 1 ON PURPOSE. check_notebook_batch_selector requires BATCH == max(BATCHES), so the child meant to run FIRST must carry the HIGHER number; rung 1 is batch 48. Identical to batch 48 except G_pa 400 -> 4000 and G_core_pa 4000 -> 40000. COST: 28.4 h at 4 ms, because dt falls 31.6x against em48 and the KELVIN-VOIGT limit (0.5*rho*nu/Gcore = 4.375e-08 s) BINDS BELOW the wave CFL (4.770e-08 s) at this modulus -- the first child on record where it does. A 30 ms window would be 212.7 h, i.e. 8.9 days, which is a feasibility limit and not a price. RUN SEPARATELY FROM BATCH 48; together they are 36.6 h and no session holds that. The Stage 6.80 lumen mode is 29x worse than em48 here, so if anything dies it is most likely maxVs OUTSIDE the clot with maxVsInClot still 0."),
    48: ("em49_G400", 10.3, "eulerian, THROAT, two-velocity. DR LI'S STIFFNESS REQUEST, RUNG 1 -- RUN THIS ONE FIRST (rung 2 is batch 47; see the note there for why the numbers are inverted). ONE PHYSICAL CHANGE against em48_polecubic_30ms: G_pa 4 -> 400 with G_core_pa 40 -> 4000, keeping the 10:1 core/shell ratio. He wrote on 2026-09-22 that at 4 Pa the clot behaves 'more like a fluid with a higher viscosity than blood rather than a deformable solid' and asked for 400/4000/40000; 40000 was not built, on Alireza's instruction. end_time_s 0.03 -> 0.005 (raised from 0.004 on Alireza's approval: 10.3 h nominal, ~11.8 h if the stiffer operator costs 15% more per solve, against his stated 12 h ceiling), write_interval_s 2.5e-04 -> 2.5e-05 (200 frames). WHY 4 ms AND NOT 30: dt is set by the ELASTIC WAVE COURANT on the CORE modulus, dt = 0.5*hMin/sqrt(Gcore/rho) with hMin = 5.8876e-07 m from the THROAT cells, which reproduces em48's logged dt to 0.2%. dt falls 10x here, so 30 ms would cost 61.7 h. em48 writes every 2.5e-04 s so a MATCHED-TIME comparison is available at any endpoint; 4 ms is also em47's horizon and is where em50_G4000 stops, keeping the two rungs comparable to each other. THE DISCRIMINATOR: em48's clot length ratio is 1.706 at 4 ms and ~2.03 at 5 ms, measured on its written q field. Stiff-and-solid predicts <= 1.15. The extra millisecond buys a wider gap for a NULL result, NOT the bond-release event -- debondedFaces is 29 by 4 ms, 30 by 4.4488 ms, then flat until 17.66 ms. If it lands near 1.7 the deformation is NOT elastic-limited and the PI's fix does not work, which is the more important result. WATCH maxVs against maxVsInClot: the lumen residual elastic stress scales as floor*G while Kres scales as 1/dt, so the Stage 6.80 lumen mode is 10x worse here."),
    49: ("em51_G400_s10", 3.8, "eulerian, THROAT, two-velocity. S10 DEBUG CHILD -- em49 WITH THE FOUR ROOT-CAUSE REPAIRS (research/designs/2026-09-22-S10-root-causes-em49.md), on Alireza's instruction 2026-09-22: homogeneous G 400/400 (the 10x core removed for turnaround; k0p_core kept), network_pressure_reference deposited (no step-1 interface tension), content_transport_scheme limitedLinear (shared limiter, no upwind smearing), released_carrier material (released clot rides the solved Vs, slip wall), and the wall-flux constraint (unconditional). dt 4.770e-07 s, 10482 steps, ~3.8 h at em49's 1.247 s/step plus 5%. Gates G1-G4 pre-registered in the child's expectation; compare against em49 (run 20260922-104613) at matched times."),
    50: ("em52_G400_fct", 4.0, "RAN as run 20260923-031942: all four S10 fixes held; died at 3.235 ms of a void-cell runaway (research/runs/20260923-031942-em52-fct-fixes-held-void-cell-runaway.md). eulerian, THROAT, two-velocity. S10 DEBUG CHILD, SECOND ATTEMPT: em51 with ONE key changed, content_transport_scheme limitedLinear -> fct. em51 (batch 49, run 20260923-012704) died on step 2 of negative content: an implicit limited operator is not an M-matrix. fct = implicit UPWIND predictor + Zalesak-limited (linear - upwind) correction, one shared limiter; priced offline on the real mesh with implicit assembly (0 negatives in 200 steps; the harness reproduced em51's failure first). Fixes 1, 2, 4 as em51 (G1 and G4 passed on em51's one step). ~4 h."),
    51: ("em54_G4_s11 em53_G400_s11", 3.4, "RAN as run 20260923-085741: G1, G3', G4, G5, G6 passed (no voids, no compaction stripes); both stopped on the convexExcess domain check (em53 0.512 ms, em54 2.530 ms) after the clot's dilute fringe ran away at the upstream wall corner -- its drag was the calibrated k1 floor, 10x below the network's own (research/runs/20260923-085741-em53-em54-s11-fringe-runaway.md). eulerian, THROAT, two-velocity. S11 ROOT-CAUSE FIXES of run 20260923-031942 (research/designs/2026-09-23-S11-root-causes-em52.md): skeleton momentum in the multiphaseEuler phase-continuity form, the 2x2 block inverted on faces, series face transmission of the network stress (and of the BSD coefficient), the network pressure as a compact face-gradient phase pressure, a convex volumetric law with bulk modulus exactly K_n (convexExcess), be relaxed only where no network was deposited (new content mixed in stress-free), and dt bounded by the skeleton-carrier Courant number and the network-pressure diffusion number. em54 (G 4/4, ~20 min, ~1050 steps) runs beside em53 (G 400/400, em52's twin, ~3.2 h); CONCURRENCY 2. Gates G1, G3', G4 as em52 and G5-G9 (voids 0; argHighPassRms <= 1e-3; fringe high-pass <= 2e-3; dilute slip and face Vs <= 0.1 m/s; reaches 5 ms), calibrated so em52 fails each, pre-registered in the children's expectations."),
    52: ("em56_G4_s12 em55_G400_s12", 3.2, "RAN as run 20260923-135545: em56 reached 5 ms (exit 0, no runaway, drag identity exact) but failed G6/G7/G10 late; em55 stopped at 0.801 ms when a grid-scale density mode ran away in the far fringe at the upstream wall corner (research/runs/20260923-135545-em55-em56-s12-grid-scale-mode.md). eulerian, THROAT, two-velocity. S12 -- THE DRAG WEIGHTED AS THE STRESS ALREADY IS (research/designs/2026-09-23-S12-intrinsic-occupancy-drag.md): dragOccupancyBasis intrinsic, the interphase drag of a partial cell = its occupancy times the network's OWN Kozeny-Carman drag at its intrinsic density a = phi_s0 + theta - thetaNetRef, ind*(1-a)^2*nu/k_KC(a); the calibrated dilute floor k1 is retired (refused on top, OI-31). em56 = em54 and em55 = em53 with that ONE change (plus skeleton_dilute_drag_rate 0, which it requires). Precedent: Li, Lowengrub, Raetz & Voigt 2009 Eq. 2.27; multiphaseEuler dispersedDragModel::K = alpha_d*Ki; Wufsus et al. 2013. Children run in matrix order: em56 (G 4, ~21 min) FIRST, then em55 (G 400, ~2.8 h at em53's measured 0.96 s/step). Gates G1, G3', G4-G9 as em53/em54, plus G10 PHASE_CLOT_FRINGE fringeOverBodySlipRms <= 8 at every record (em53/em54 read 12.5-38.8 on this expression; priced 1.4-4.2) and G11 fringeDragOverIntrinsic = 1 within 1e-9, pre-registered in the children's expectations."),
    53: ("em57_G4_s13 em58_G400_s13", 3.4, "RAN as run 20260923-204553: em57 reached 5 ms (G6/G7/G10 still failed, marginally; strict body zigzag halved); em58 reached 2.119 ms and blew up in 3 steps when a far-fringe wall tail at principal stretch 3.2-6.2 outran a dt bound that read G, not the tangent (research/runs/20260923-204553-em57-em58-s13-tail-elastic-instability.md). eulerian, THROAT, two-velocity. S13 -- THE GRID-SCALE RESTORING (research/designs/2026-09-23-S13-grid-scale-restoring.md): network_pressure_weight networkPresence (the convexExcess partial pressure is weighted by network PRESENCE, not occupancy -- its additive argument is already partial, and S11 counted the occupancy twice, leaving the fringe occ^2 of its own volumetric resistance) and be_stretch_gradient fluxConsistent (be's stretch from the flux that carries be and the content, the construction the one-velocity path already has, so a grid-scale compaction strains the network it compacts). em57 = em56 and em58 = em55 with those two keys only. Priced with the per-cell dispersion relation of the linearised skeleton row on batch 52's own fields: unstable cells 0% everywhere at every written time with both fixes (shipped: em55 far fringe 2.4% at 0.7 ms; em56 far fringe 33% and body 5.5% at 5 ms). em57 (G 4, ~24 min) FIRST, then em58 (G 400, ~3 h). Gates G1-G11 as batch 52 -- G6 (argHighPassRms <= 1e-3), G7 (fringe high-pass <= 2e-3), G8, G9 and G10 are the ones batch 52 failed -- pre-registered in the children's expectations."),
    54: ("em59_G4_s14 em60_G400_s14", 4.0, "RAN as run 20260924-014332: rupture ran as designed (em59 first rupture 3.63 ms in the body, em60 1.47 ms in the wall tail; G12 and G13 passed) and BOTH children stopped (3.70 ms, 1.66 ms) when the rupture label, carried on shell_'s content-limited weights, left [0, 1] and phiSkel*(1 - ruptured) went negative (research/runs/20260924-014332-em59-em60-s14-rupture-label-unbounded.md). eulerian, THROAT, two-velocity. S14 -- NETWORK RUPTURE AT THE CRITICAL STRETCH (Alireza's choice, 2026-09-23; research/designs/2026-09-23-S14-network-rupture.md): where network reaches principal stretch 2.8 (Tutwiler et al. 2020, source 28; single fibres 2.5-3.3) it ruptures -- content kept as debris (packing and drag), shear load path and elastic memory gone, the rupture carried with the material like the shell label -- and the elastic dt bound reads the tangent stiffness G[(8/9)b1 + (2/9)(b2+b3)] instead of G. em59 = em57 and em60 = em58 plus network_rupture criticalStretch, network_critical_stretch 2.8, elastic_stability_stiffness tangent. Priced on batch 53's fields: em60 ruptures only a few far-fringe tail cells at the upstream wall corner (<= 11 cells, 0.0003% of content by 2.1 ms); em59's soft body passes 2.8 from ~3.5-4 ms (1.7% of content at 4 ms, ~12% by 5 ms). em59 (G 4, ~25 min) FIRST, then em60 (G 400, ~3-5 h: the tangent bound may halve dt at the worst times). Gates G1, G3', G4, G5, G8, G9, G11, plus G12 (stretch at rupture within 10% of 2.8: the rupture resolves the stretch it acts on), G13 (tangent elastic number <= 0.25) and G14 (the rupture label within [0, 1]); G6/G7/G10 as gates for em60, WATCH for em59 (a Laplacian high-pass reads smooth curvature at G 4) -- all pre-registered in the children's expectations."),
    55: ("em59_G4_s14 em60_G400_s14", 4.0, "SUPERSEDED BEFORE RUNNING by batch 56 (S15, 2026-09-24); still runnable with this package by setting BATCH = 55, as the S14 control. eulerian, THROAT, two-velocity. S14 RE-RUN, INPUTS UNCHANGED -- the fix is in the solver: the rupture label is transported as the content is (implicit upwind M-matrix predictor + Zalesak-limited correction with min(content lambda, its own)), bounded in [0, 1] by construction; replayed on batch 54's returned meshes it stayed exactly in [0, 1] where the old operator reproduced -0.091/1.037. em59 (G 4, ~25 min) FIRST, then em60 (G 400, ~3-5 h). Gates as batch 54 plus G14 by construction. em60 is EXPECTED to exceed G8 at rupture events (drag-limited recoil 0.12-0.21 m/s measured) -- reported alongside it: the same velocities >= 50 us after the last rupture (batch 54: 0.022/0.029 m/s). em60 is EXPECTED to fail G7/G10 as em58 did: the fringe texture in clotDrag predates any rupture."),
    56: ("em61_G4_s15 em62_G400_s15", 4.0, "eulerian, THROAT, two-velocity. S15 -- THE ROOT FIXES FOR OI-34 AND OI-35 (Alireza, 2026-09-24; research/designs/2026-09-24-S15-composition-label-and-progressive-damage.md). (1) The core/shell label IS thetaH/(thetaC+thetaH), the shell share of the FCT-transported solid contents: bounded by construction, no transport, no clip (batch 54: the clipped label had drifted up to 0.18-0.25 in ~220 core/shell cells, k0p up to 3x wrong). (2) Progressive network damage, Kliuchnikov et al. 2025 (source 32) Eq. 6: d = 1 - exp(-(l1 - 2.105)/2), S = S0 (1 - d), irreversible -- replaces S14 all-or-nothing rupture whose one-cell step release drove the 0.12-0.21 m/s recoil; hardening up to l1 ~ 4.1, tangent <= 5.51 G for every stretch. em61 = em59, em62 = em60, plus material_label_basis contentComposition and network_rupture progressiveDamage only. em61 (G 4, ~25 min) FIRST, then em62 (G 400, ~3-5 h). Gates G1, G3', G4, G5, G8 (UNCHANGED; predicted to PASS now), G9, G11, G12p (damageStepMax <= 0.05), G13, G14; G7/G10 expected to fail at G 400 as em58 (fringe texture not targeted)."),
    57: ("em63_G400_s16 em64_G4_s16 em65_G4_s16_final", 14.0, "eulerian, THROAT, two-velocity. S16 -- TERMINAL RUPTURE WITH A TIMED RELEASE, DAMAGE CARRIED AS CONTENT (Alireza, 2026-09-24; research/designs/2026-09-24-S16-timed-terminal-rupture.md, APPROVED; waivers in the docket DECISIONS.md). em63 = em62 and em64 = em61 plus the S16 keys only: network_rupture terminalRelease (d* = 1 at lc 2.8, Eq. 6 below it; d relaxes to d* over tau 3.3e-4 s by backward Euler), damage_transport contentPartition (the partition theta - D*, D* - D, D rides the content transport; the shared limiter with the no-flux convention), skeleton_floor_basis occupancy, network_pressure_damage tensile. em65 = em64 plus UFinal 1 / pFinal 1 (H-P4, opt-in). CONCURRENCY 2: em64 and em63 start together, em65 takes em64's slot (~40 min each); em63 3-12 h priced, 32 h worst case, stopped by a 14 h wall budget (stopAtClockTime writeNow). Gates: G1, G3', G4, G5, G11, G14, S15a, G-id (offline), G12r, S16w, S16p (offline), config; G8 predicted 0.079-0.090 m/s at G 400 (SUSPECTED); G6/G7/G10 expected to fail at G 400. em65: start-up gate, onset 0.9-1.6 ms, tear band by 5 ms."),
    58: ("s17_D1_nc3 s17_D2_nc4 s17_D2b_nc6 s17_D3_dt18 s17_D3b_dt25 s17_CB6 s17_CB4 em67_G4_s17 em66_G400_s17 em68_G400_s17_law em69_G400_s17_noimpl em70_G4_s17_noimpl", 14.0, "eulerian, THROAT, two-velocity. S17 -- FIBRE-FORM RELEASE, IMPLICIT ELASTIC COUPLING, A TIME-ACCURATE START-UP (Alireza, 2026-09-26; research/designs/2026-09-26-S17-frozen-release-implicit-elastic-startup.md, revision 4, APPROVED). A = network_rupture fibreRelease (the pending share is a fibre tension along its own axis, rotated as a material line, stretch non-increasing, P-LS blend projection; be := I where no intact network remains); B = final_relaxation_unity with n_correctors 6; C = elastic_coupling implicitTangent (Gamma_t in the BSD pair, a real Vs predictor, nuEl out of the dt rate). LAUNCH ORDER (design section 6): the seven 0.25 ms discriminators first (D1/D2/D2b/D3/D3b from em65, CB6/CB4 from em66; the face-wise outlet and clot-surface second differences), then em67 (G 4, A+B+C, 3 h budget) with em66 (G 400, A+B+C, 14 h budget), then em68 (A only, 2.5 ms) with em69 (A+B explicit, 2.5 ms), then em70 (G 4, A+B explicit, 3 h budget). CONCURRENCY 2. Each control differs from its production child only in its named switches (checked by create_cases.py). Gates: G1, G3', G4, G5, G11, G14, S15a, G12r, S16w, S16m, config (S16 and S17), G6/G7/G10 restored, G8 at 0.1 m/s unchanged; new: G-tr (fails the run above 1.05), G-rot, G-eig, G-uni, G-proj, G-dep (unit), S17split, P2; em66 <= 20905 steps to 5 ms; D2b >= 1 means B does not ship; CB6 >= 1 fails; G-id em68 vs em63 (offline)."),
    59: ("em72_G4_s17b em71_G400_s17b em73_G400_s17b_noimpl em74_G400_s17b_noA3", 14.0, "eulerian, THROAT, two-velocity. S17b -- SLACK FIBRES (V1), FIELD-UNIT FIBRE ARITHMETIC, THE WALL FACE IN BOTH RECONSTRUCTS (Alireza, 2026-09-27; research/designs/2026-09-27-S17b-slack-fibre-field-units-wall-face.md, revision 2, APPROVED). A1 = fibre_slack v1 (a fibre carried to its rest length goes slack: its taut content, C_B and q_P go to 0 at once, the content stays in P2 until the tau law releases it); A2 = fibre_arithmetic fieldUnits (the six partition/fibre predictors in field units, the fibre read only above DBL_MIN); under A1 thetaPending, thetaTaut, thetaPendingB and thetaPendingP (+Final) run one fixed 3-sweep Gauss-Seidel entry (section 8 a); A3 = skeleton_wall_face zeroed (the wall face term zeroed in both the Vs and U reconstructs). em71 = em66 + A1 + A2 + A3 (G 400, 14 h budget); em72 = em67 + A1 + A2 + A3 (G 4, 3 h); em73 = em69 + A1 + A2 + A3, explicit (2.5 ms, 6 h budget); em74 = em71 without A3 (2.5 ms, 6 h budget; design section 8 e: em73 dying inside the H-NA10 loop after its G8 FAIL confirms the prediction). LAUNCH: em72 with em71, then em73 with em74, CONCURRENCY 2. Gates: no fibre-invalid fatal; the V1 slack gate |dd| = 0 (the other slack checks are conformance, plus a watch on the pre-event residual, section 8 d); theta_T <= theta_P; G-proj; W2 and P2 <= 0.9702 x 1.01; energy; the start-up on sign alternation, step 1 to the first damage (section 8 h; later alternation a WATCH), face-wise c_n and the outletMeanUx proxy both reported; G-id em74 vs em66 on physical lines to the first transfer; the S17 carried gates; G8 at 0.1 m/s scored FAIL on any exceedance (em73 predicted to FAIL near 2.0-2.1 ms by H-NA10); em74 vs em71 sign rule against the logged wall-face projection. DISK: free >= 5 GiB before the launch (design section 5)."),
    60: ("em76_G4_s18", 3.0, "eulerian, THROAT, two-velocity. S18, THE SOFT STAGE (Alireza, 2026-09-29; research/designs/2026-09-29-S18-no-frozen-pressure-content-weight-network-viscosity.md, rev. 10, APPROVED). em76 = em72 + A (frozen_pending_pressure none: p_used excludes the frozen q_P/S) + B (wall_patch_split split: the S17 split on the wall patch, which the wall bond reads) + C (network_pressure_weight contentRatio, R2b: theta/ref on both branches, inside the gradient) + E (cplus_order shared: one C+ order for the passengers and the taut share); no R3, no consistent correction, the shipped Vs entry (D, I, H deferred to OI-44). 5 ms, 3 h budget (em72: 8203 s). Gates: ANY VS_SPLIT slip > 0.1 m/s in 0-5 ms FAILS (never scored on maxFaceVs); conformance: frozenPower = 0, projRaised = 0, S17split on the wall patch; the carried S17b gates; the vs-exceed-fraction gate is a WATCH (DECISIONS). The onset window [1.17, 1.22] ms, the tearing clause and the debond watch reference em72: scored OFFLINE against batch 59, NOT-EVALUABLE on Colab. The moved gate (clotXbar >= 6.77 / 15.77 um at 1.0 / 2.0 ms) catches only a frozen clot."),
    61: ("xu5_g250 xu5_g500 xu5_g1000 xu5_g2000", 13.0, "eulerian, STRAIGHT 300 x 50 um CHANNEL, two-velocity, the soft S18. XU ET AL. 2017 FIGURE 5 (Alireza, 2026-09-29/30; research/designs/2026-09-29-case-xu2017-fig5.md, rev. 4, APPROVED 2026-09-30; docket research/investigations/20260929-1430-case-xu2017-partially-obstructive-clot/). Four children, one variable: the wall shear rate 250 / 500 / 1000 / 2000 1/s (Figure 5 a/e, b/f, c/g, d/h), each with its OWN velocity inlet U_mean = gammaDot H / 6 (pressure_driven false). Each child = em76 + Xu's idealised dome (half-ellipse shell x 50.9-131.9 um, top 34.7 um; seated core 40 x 14.6 um at x 90.8 um; label width 3 um), G 1.2 / G_core 24 Pa, k0p 4.07e-13 / 1e-15, Xu's plasma (rho 1025, 4 cP), growth off and no platelet inflow, the unbreakable bond (1e9 Pa; the em72 held layer), releaseCoreExempt off; 10 ms at dt 2.604167e-6 s (3840 steps), write 0.25 ms (40 writes, ~1.12 GiB each), stopAtClockTime 6.5 h each. LAUNCH: 250 with 500, then 1000 with 2000, CONCURRENCY 2: ~9.2-13 h in one session. The 2000 1/s child is an OI-44 probe. Gates (design section 4): A0 config echo, A1 first-write fields, A2 S18 structural, A3 the carried S17b gates, A4 no growth + the release line every step, A6 outlet crossing, A7 the bond, P0 flow (outletMeanUx / U_mean 1 +- 0.005), P1 gap hydraulics 1.152 [1.03, 1.30], N1 dt, N3 G8 <= 4 u_max, P3-P9 as pre-registered; A5 energy is scored OFFLINE (scripts/s17_offline_checks.py energy-inlet, FAIL at R > 0.9915). A child with bulk-core damage is not extended beyond 10 ms (OI-44)."),
    62: ("em78_G4_s18p_ctl em79_G4_s18p_Cnp em80_G4_s18p_Afz em81_G4_s18p_Esh", 6.0, "eulerian, THROAT, two-velocity, S18 + the log-only S18p probe. THE em76 ATTRIBUTION PROBE (Alireza, 2026-10-01: probe batch; research/designs/2026-10-01-S18p-em76-attribution-probe.md, rev. 2; docket research/investigations/20261001-2159-run223505-s18-em76-tear/). Each child = em76 + proj_site_log yes (PHASE_CLOT_S18P per-site P-LS counters once per step, PHASE_CLOT_S18P_CELL dump; no field, flux, solve or dt change). em78 = the control (must reproduce em76 bit for bit, V1); em79 reverts C (network_pressure_weight networkPresence); em80 reverts A (frozen_pending_pressure shipped); em81 reverts E (cplus_order shipped). 5 ms, wall_budget_h 3.0 each (em76: 4216 s at concurrency 2). LAUNCH: em78 with em79, then em80 with em81, CONCURRENCY 2: two waves, ~2.4 h (6 h at the budget). Colab assessor: V2 report-only (one PHASE_CLOT_S18P per step, T + P = PHASE_CLOT_S17 projCount/projRaised, non-empty after 1.2 ms on em78). The score (mean over 4.0-5.0 ms of child - em72 drained), V1 and the bands 3 / 6.5 / 10 are scored OFFLINE against em72 and em76."),
    63: ("fs_G4_kxu_v250 fs_G40_klo_v250", 13.0, "eulerian, THROAT, two-velocity, the A + B + E baseline (S18 networkPresence). A FREE SHELL-ONLY CLOT AT THE THROAT OPENING, VENOUS 250 1/s (Alireza, 2026-10-02; Dr He Li: is a detached shell stiff enough to block the downstream vessel; research/designs/2026-10-02-case-free-shell-throat.md, rev. 3, APPROVED 2026-10-02; docket research/investigations/20261002-1219-case-free-shell-throat/). fs_G4_kxu_v250 = loose (G 4 Pa, k 4.07e-13); fs_G40_klo_v250 = dense (G 40 Pa, k 1e-15). A 30 um disc resting UNBONDED on the bottom wall just inside the taper. 30 ms, wall_budget_h 13 each (G 40 is clock-stopped at ~17-26 ms by design), both at CONCURRENCY 2 in one session. A step-1 watcher stops a child through stopAtFile if any bond, bond-hold seed or support appears (watcher-stop.txt)."),
    64: ("fsp_ctl fsp_k13 fsp_dt15 fsp_G40", 1.0, "eulerian, THROAT, two-velocity, the A + B + E baseline. THE FREE-SHELL FLUX-OSCILLATION PROBE (Alireza, 2026-10-03: Prepare it now; research/designs/2026-10-03-probe-fs-flux-oscillation.md rev. 1; docket research/investigations/20261003-0841-run000038-free-shell-live/). Each = fs_G4_kxu_v250 to 0.4 ms (wall budget 1 h) plus one key: ctl (none, must reproduce batch 63 bit for bit), k13 (k0p 1e-13), dt15 (maxDeltaT 1.5e-6), G40 (G 40). Read: outletFlux every step, flip fraction and out-of-band steps over 0.05-0.40 ms. ~30-40 min at CONCURRENCY 2."),
    65: ("fsb_off fsb_k13 fsb_G40 fsb_ctl fsb_ctl_r3", 1.0, "eulerian, THROAT, two-velocity, the A + B + E baseline. S19, THE BLOCK-COUPLED STEP (Alireza, 2026-10-04: Approve, but include R3 as an option; research/designs/2026-10-03-S19-block-coupled-step.md rev. 4; docket research/investigations/20261003-0205-np-block-coupled-solve/). One direct UMFPACK solve per step for (Ux, Uy, Vsx, Vsy, p) replaces the predictors and the 6 segregated correctors (OI-47). fsb_off = fsp_k13 with the key ABSENT (V1: bit-identical to fsp_k13); fsb_k13 / fsb_G40 / fsb_ctl = fsp_k13 / fsp_G40 / fsp_ctl + momentum_pressure_coupling blockCoupled; fsb_ctl_r3 = fsb_ctl + network_viscosity kelvinVoigt (R3). 0.4 ms, wall budget 1 h each. Read: PHASE_CLOT_S19 every step (G0 engagement, G4 contRes and qInlet/qOutlet, G6 linRes, G7 kMatch = 0, G8r r3Power <= 0), G1 against batch 64 (U rel. L2 <= 1.75e-2 at 0.2 and 0.4 ms), G2 on fsb_ctl (SMOOTH), G3 ClockTime median <= 1.5 x the batch-64 parent. ~40-60 min at CONCURRENCY 2."),
    66: ("fsb_off fsb_k13 fsb_G40 fsb_ctl fsb_ctl_r3", 1.0, "eulerian, THROAT, two-velocity, the A + B + E baseline. S19a (Alireza, 2026-10-04: Approve as designed; research/designs/2026-10-04-S19a-umfpack-row-equilibration.md rev. 2; docket research/investigations/20261004-1608-run203938-s19-umfpack-oom/): batch 65's five children UNCHANGED on the S19a package, where UMFPACK factors A itself (CSC, UMFPACK_A) and so equilibrates its rows; batch 65 died of UMFPACK_ERROR_out_of_memory in step 1. Read: G6 linRes and linResScaled <= 1e-10, G9 numericPeakGB <= 1.0 and tNumeric <= 5 s every step, noffdiag (watch, expected 0), V1 fsb_off bit-identical to fsp_k13. ~40-60 min at CONCURRENCY 2."),
    67: ("fsb_off fsb_k13_id fsb_k13 fsb_G40 fsb_ctl fsb_ctl_r3", 1.0, "eulerian, THROAT, two-velocity, the A + B + E baseline. S19b (Alireza, 2026-10-04: Approve as designed; research/designs/2026-10-04-S19b-lumen-elimination.md rev. 2; docket research/investigations/20261004-1803-s19b-lumen-elimination/): under blockCoupled the lumen Vs unknowns are ELIMINATED by default (Vs = U there; n 156,800 -> ~106,578), so no identity row is left for any equilibration to trip on; fsb_k13_id = fsb_k13 with block_coupled_lumen identityRows, the S19a control. Read: C1 on fsb_k13_id, D1 on every default blockCoupled child, E1 offline (scripts/s19b_e1_score.py, fsb_k13 against fsb_k13_id), G0-G9, V1 fsb_off bit-identical to fsp_k13; watch noffdiag (expected 0). ~1.3 h at CONCURRENCY 2."),
    68: ("fsf_G40_klo_v250 fsf_G4_kxu_v250 fsb_G40_klo_rest fs_G40_klo_rest_seg", 22.0, "eulerian, THROAT, two-velocity, the A + B + E baseline, S19b (blockCoupled, default lumen elimination). THE TWO FREE SHELL-ONLY CLOTS, CENTRED OFF BOTH WALLS, ON THE BLOCK-COUPLED STEP WITH R3 (Alireza, 2026-10-05: \"I want clot to be completely free\"; \"Centred, off both walls\"; approved as designed, R3 per case, the control pair; research/designs/2026-10-05-case-free-shell-centred-s19b.md rev. 3; docket research/investigations/20261005-0010-run042536-b67-s19b-validation/). fsf_G40_klo_v250 (dense, G 40, k 1e-15) and fsf_G4_kxu_v250 (loose, G 4, k 4.07e-13): batch 63's pair, disc centre (129.4, 24.5) um, gaps 9.51/9.50 um >= 3 epsilonC, R3 kelvinVoigt (tau_v per child), endTime 25 ms; frames (8 reader-derived fields) every 0.2 ms, full writes every 5 ms and on any stop. Then the OI-48 control pair (fs_G40_klo_v250 exactly, resting, 2 ms): fsb_G40_klo_rest (block) and fs_G40_klo_rest_seg (segregated). CONCURRENCY 2: slot 1 = the dense child (22.0 h budget; reach 21.9 ms at 5.45 s/step, 19.0 ms at +15%); slot 2 = loose (16.0 h budget, ~8 h expected) then the pair (3.0 h budget each, ~2 h expected). NOTHING REACHES DRIVE UNTIL ALLRUN HAS FINISHED: a session shorter than ~23 h loses the whole run."),
    69: ("em78_id xu5_id fsf_G4_kxu_r3_40 fsf_G4_kxu_nor3_40 fsf_G40_klo_v250", 22.2, "BATCH 69 (SELECTOR ENTRY; set LAUNCH to 1, 2 or 3. LAUNCH 3 = the stiff clot fsf_G40_klo_v250 rerun on this release (S20), Alireza 2026-10-06: does the clog survive a sharp edge (docket 20261006-0950)? Its own session, 22 h budget, no 5 ms watcher; compare with run 20261005-110516, one 40 ms child per Colab session after its rider, the two sessions in parallel; each launch runs at concurrency 1, forced below): the loose free clot to 40 ms on S20, R3 on against R3 off (research/designs/2026-10-05-case-loose-40ms-r3-pair.md rev. 4). LAUNCH 1 = em78_id then fsf_G4_kxu_r3_40 (R3 kelvinVoigt, tau_v 1.5 ms), LAUNCH 2 = xu5_id then fsf_G4_kxu_nor3_40 (network_viscosity none); the matrix order puts the embolus and Xu series (21, 22) before throat_free_shell (23), so each rider runs first. Each is batch 68's fsf_G4_kxu_v250 except endTime 40 ms and a 22.0 h clock budget (40 ms ~ 13.1 h priced on batch 68's wall clock, 15.1 h with 15%; 2.0 h of the 24 h session left); frames every 0.2 ms, full writes every 5 ms, ~1.25 GB per session. NOTHING REACHES DRIVE UNTIL ALLRUN HAS FINISHED. Runs on P2 = S21 + S20 (the clean solver with the S20 fix), after the S21 validation session. The riders are S20 V1 and V3: the identity twins em78_id (session 1) and xu5_id (session 2), 0.5 ms each (~105 steps, wall_budget_h 1.0 each), REPORTED and not gated. The runner's 5 ms watcher (run_main_case_colab.py, watch_child) judges each 40 ms child at its 5 ms frame, writes runs/<child>/watcher-verdict.json, and stops a child that is not PASS through the stopWatcher stopAtFile function object; the R3-off width is reported only."),
}

# CONCURRENCY. Children are INDEPENDENT cases, so running N side by side is
# THROUGHPUT, not parallelism: no decomposePar, no processor boundaries, and
# each child's solver still runs `nProcs : 1`, which is the mode every
# diagnostic in this solver assumes. Two children at once produce BIT-IDENTICAL
# logs to two children run one after the other -- only wall time changes.
#
# That is the reason to use it here. MPI decomposition would be a different
# proposition entirely: `yWorstBlood` is picked on a LOCAL maximum and its own
# comment says it is "only meaningful in serial", and 24500 cells over 2 ranks
# is too small to scale anyway.
#
# 0 = every core Colab reports. 1 = serial, as every returned run so far.
#
# WHAT TO EXPECT ON 2 CORES. Batch 0 is 5 children, so 2 slots means THREE
# waves (2, 2, 1), not a clean halving: 8.5 h -> ~5.1 h at perfect scaling.
# OpenFOAM's linear solvers are memory-bandwidth bound, so two of them sharing
# a socket each run somewhat slower than alone -- budget 5-6 h, not 4.25.
#
# The gain is not only time. Colab sessions drop, and a 5 h run is materially
# less exposed than an 8.5 h one.
import os
CONCURRENCY = 2
if CONCURRENCY == 0:
    CONCURRENCY = os.cpu_count() or 1
os.environ['AM2020_CONCURRENCY'] = str(CONCURRENCY)
print(f'concurrency: {CONCURRENCY}  (cores visible: {os.cpu_count()})')
if CONCURRENCY > 1:
    print('  NOTE: realised_cost_seconds will be CONTENDED wall time, not\n'
          '        per-child cost. result.json records realised_cost_concurrency\n'
          '        beside it. Use ExecutionTime from the solver log for cost models.')

# BATCH 70, the S21 validation, ran on P1 (release-c53a61a6, digest-f874e5ff9947), not on
# this release: it compares the clean solver WITHOUT S20 against R0, so it is not offered here.
BATCH = 69
# LAUNCH selects one launch of a two-launch entry (BATCH 69: two sessions; BATCH 70, the S21
# validation: 1 = R0, 2 = P1). BATCH 69: each launch is its own Colab session. BATCH 70:
# both launches in ONE runtime, chosen by R0_FIRST at the top of this cell.
LAUNCH = 1
if BATCH == 70:
    LAUNCH = 1 if R0_FIRST else 2   # S21 validation: 1 = R0 reference, 2 = this release
elif R0_FIRST:
    raise ValueError('R0_FIRST is for BATCH 70 (the S21 validation) only; set it False')
LAUNCHES = {
    69: {1: ("em78_id fsf_G4_kxu_r3_40", 22.2), 2: ("xu5_id fsf_G4_kxu_nor3_40", 22.2),
         3: ("fsf_G40_klo_v250", 22.2)},   # the stiff rerun on S20 (docket 20261006-0950)
}
# BATCH 69 runs each launch's children one at a time, in matrix order: the rider (em78_id or
# xu5_id) finishes before the 40 ms child starts (design rev. 4 section 2, concurrency 1). The
# value here overrides CONCURRENCY above for that batch only.
LAUNCH_CONCURRENCY = {69: 1}
if BATCH in LAUNCH_CONCURRENCY:
    CONCURRENCY = LAUNCH_CONCURRENCY[BATCH]
    os.environ['AM2020_CONCURRENCY'] = str(CONCURRENCY)
    print(f'BATCH {BATCH}: concurrency forced to {CONCURRENCY} (rider first, then the 40 ms child)')
ONLY, _hours, _path = BATCHES[BATCH]
if BATCH in LAUNCHES:
    if LAUNCH not in LAUNCHES[BATCH]:
        raise ValueError(f'BATCH {BATCH} has launches {sorted(LAUNCHES[BATCH])}; LAUNCH = {LAUNCH}')
    ONLY, _hours = LAUNCHES[BATCH][LAUNCH]
    print(f'BATCH {BATCH} launch {LAUNCH}: {ONLY}')
print(f'Batch {BATCH}: {len(ONLY.split())} children, ~{_hours:.1f} h, path = {_path}')

run_id = datetime.now().strftime('%Y%m%d-%H%M%S-am2020-main-case')
print('Run ID:', run_id, flush=True)
process = subprocess.Popen(['python3', str(runner), '--run-id', run_id] + runner_args + (['--only', ONLY] if ONLY else []), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
assert process.stdout is not None
for line in process.stdout:
    print(line, end='', flush=True)
return_code = process.wait()

run_drive = CONTENT / 'drive/MyDrive/OpenFOAM_Blood/runs' / run_id
artifact = run_drive / 'artifacts' / ARTIFACT
if return_code != 0:
    runner_result = artifact / 'runner_result.json'
    print('\n--- archived runner result ---')
    print(runner_result.read_text() if runner_result.is_file() else f'MISSING: {runner_result}')
    # The case directory is named case-after-run-<CASE>, not case-after-run.
    # This path was wrong until 2026-08-14 and the glob silently matched
    # nothing, so run 20260814-221530 reported a bare exit code and the
    # FatalIOError that caused it never reached the screen. Globbed now
    # rather than spelled out, so it cannot drift from the archiver again.
    child_dirs = []
    for case_root in sorted(run_drive.glob('case-after-run-*/runs')):
        child_dirs.extend(sorted(case_root.glob('*')))
    if not child_dirs:
        print(f'no child directories found under {run_drive}')
    for child_dir in child_dirs:
        exit_file = child_dir / 'run-exit-code.txt'
        exit_text = exit_file.read_text().strip() if exit_file.is_file() else 'MISSING'
        if exit_text != '0':
            print(f'\n[{child_dir.name}] run-exit-code={exit_text}')
            for name in ('log.initialise','log.blockMesh','log.checkMesh','log.phaseClotAxisymmetricFluid'):
                log = child_dir / name
                if log.is_file():
                    print(f'--- {child_dir.name}/{name} tail ---')
                    print('\n'.join(log.read_text(errors='replace').splitlines()[-60:]))
    # The assessor writes result.json under artifacts/<ARTIFACT>/<CASE>/, not
    # directly under artifacts/<ARTIFACT>/. Spelled out, this path was one
    # level short, so run 20260815-182036 raised "without an assessor result"
    # while the result sat on disk a directory below and the run had in fact
    # produced its full measurement. Globbed, like the log paths above.
    result_paths = sorted(artifact.rglob('result.json'))
    if not result_paths:
        raise RuntimeError(f'Runner returned {return_code} without an assessor result; evidence at {run_drive}')
    print(f'assessor result: {result_paths[0]}')
    print(f'\nAn excluded child is information, not a crash. The next cell reports it. Evidence: {run_drive}')


In [ ]:
import json
from pathlib import Path

art = run_drive / 'artifacts' / 'main_case_v1'
runner = json.loads((art / 'runner_result.json').read_text())
res = runner['gates']['AM2020_planar_mural_detachment']['result']

print('MAIN CASE:', runner.get('verdict'))
print(f"admissible: {res['children_admissible']}/{res['children_total']}\n")
for bad in res['inadmissible']:
    print(f"  EXCLUDED {bad['child_id']}: {bad['why']}")

print('\n' + '=' * 72)
print('GROWTH RATE vs FLOW RATE')
print('=' * 72)
print(f"{'child':<18}{'shear 1/s':>10}{'U m/s':>12}{'Vn m/s':>14}{'max shear':>12}")
for row in res['growth_vs_flow']['rows']:
    vn = row.get('Vn_m_s')
    print(f"{row['child_id']:<18}{row['wall_shear_rate_1_s']:>10.0f}"
          f"{row['mean_velocity_m_s']:>12.2e}"
          f"{(f'{vn:.4e}' if vn else row.get('reason','-')):>14}"
          f"{(row.get('max_shear_rate_1_s') or 0):>12.1f}")
fit = res['growth_vs_flow']['flow_dependence']
if fit['exponent'] is not None:
    print(f"\nfitted exponent n in Vn ~ U^n : {fit['exponent']:.3f}"
          f"   (transport-limited prediction {fit['predicted_exponent']:.3f})")
else:
    print(f"\nno exponent: {fit['reason']}")

print('\n' + '=' * 72)
print('REQUIRED ADHESIVE STRENGTH  (welded bond; this is what detachment would need)')
print('=' * 72)
print(f"{'child':<18}{'shear':>8}{'G Pa':>10}{'k0p m2':>12}{'required Pa':>14}")
for row in res['required_strength']['rows']:
    req = row['required_strength_pa']
    print(f"{row['child_id']:<18}{row['wall_shear_rate_1_s']:>8.0f}{row['G_pa']:>10.1f}"
          f"{row['k0p_m2']:>12.1e}"
          f"{(f'{req:.4f}' if req is not None else row['reason']):>14}")
p = res['required_strength']['flow_not_stiffness']
print(f"\nload set by flow rather than stiffness? {p['supported']}")
if p.get('stiffness_spread_at_fixed_flow'):
    print(f"  spread in required strength across a {p['stiffness_span']:.0f}x span in G:")
    for flow, spread in p['stiffness_spread_at_fixed_flow'].items():
        print(f"    at {flow} 1/s : {spread:.2f}x")
elif p.get('reason'):
    print(f"  not evaluable: {p['reason']}")

print('\n' + '=' * 72)
print('DEMONSTRATION  (bond live)')
print('=' * 72)
for row in res['demonstration']['rows']:
    frac = row['released_fraction']
    print(f"{row['child_id']}: strength {row['adhesive_strength_pa']} Pa, "
          f"peak traction {row['peak_traction_pa']:.3f} Pa")
    print(f"   released {row['faces_released']}/{row['bonded_faces_initially']} faces"
          f" ({(frac or 0) * 100:.0f}%) over {row['steps_with_releases']} steps,"
          f" largest single step {row['largest_single_step_release']}")
    print(f"   first release at t = {row['first_release_time_s']} s;"
          f" progressive={row['progressive']}; complete={row['released_everything']}")
    # THE FOOTPRINT. Run 20260812-062701 peeled all 60 faces, the clot
    # started to move, and then growth carried it onto virgin wall faces that
    # were still initialised bonded -- the count went to 61, then 62. One
    # bonded face re-anchors the whole body, so the clot ratcheted downstream
    # in stops and starts, welding itself to healthy endothelium as it slid.
    # The footprint is now latched at the injury site. It must stay flat.
    print(f"   FOOTPRINT: {row['footprint_faces_start']} -> "
          f"{row['footprint_faces_end']} adhesive faces; "
          f"re-adhered {row['faces_reformed']} face(s) "
          f"on {row['steps_with_reformation']} step(s)")
    if row['footprint_grew']:
        print("   !! THE FOOTPRINT GREW. Growth is welding the clot to wall")
        print("      that was never injured, and every new bond arrests the")
        print("      whole body. The injury latch has failed.")
    tr = row['transport']
    if tr['available']:
        print(f"   TRANSPORT: centroid {tr['centroid_start_x_m'] * 1e6:.1f} -> "
              f"{tr['centroid_end_x_m'] * 1e6:.1f} um "
              f"({tr['displacement_m'] * 1e6:+.1f} um, "
              f"{tr['displacement_cells']:+.1f} cells) by t = {tr['transport_end_time_s']} s")
        print(f"   clot cells {tr['clot_cells_start']} -> {tr['clot_cells_end']} "
              f"({tr['volume_change_fraction'] * 100:+.1f}% volume); "
              f"moved downstream = {tr['moved_downstream']}")
        # THE TWO EDGES, SEPARATELY. A centroid cannot tell translation from a
        # body being eaten from behind: run 20260812-041322 ran its trailing
        # edge 17 cells and its leading edge 9, shrank by 8, and every metric
        # above still said "moved downstream".
        lead, trail = tr['leading_travel_cells'], tr['trailing_travel_cells']
        if lead is not None and trail is not None:
            print(f"   EDGES: leading {lead:+.1f} cells, trailing {trail:+.1f} cells"
                  f"  (translation means these AGREE)")
            print(f"   body length {tr['body_length_start_m'] * 1e6:.1f} -> "
                  f"{tr['body_length_end_m'] * 1e6:.1f} um "
                  f"({tr['length_change_fraction'] * 100:+.1f}%); "
                  f"translating = {tr['translating']}")
            # HEIGHT, not just length. Run 20260812-200349 held its length
            # at exactly the 90 um injury width while its height grew 70%,
            # because a spurious Uy advected it into the wall it sits on.
            if tr.get('height_change_fraction') is not None:
                print(f"   HEIGHT: top {tr['top_y_start_m'] * 1e6:.1f} -> "
                      f"{tr['top_y_end_m'] * 1e6:.1f} um "
                      f"({tr['height_change_fraction'] * 100:+.0f}%); "
                      f"core {tr['core_cells_start']} -> {tr['core_cells_end']} "
                      f"cells ({tr['core_growth_fraction'] * 100:+.0f}%)")
                if tr['core_growth_fraction'] > 0.15:
                    print("   !! THE CORE IS GROWING. Smearing shrinks a core, so")
                    print("      this is the body being stretched, not diffused.")
                    print("      Check UyDiscarded/crossFraction in the log.")
        # Ubody is projected onto the INLET NORMAL, a fixed geometric axis, so
        # a residual cross-stream component is a bug and not a small number.
        # Run 20260813-084207 projected onto the MEASURED free-stream direction
        # instead, which tilts up over a wall-mounted obstacle: 1-5% of Ux
        # survived the projection and lifted the phase field 6.6 um into the
        # wall. axisTilt is that tilt, now reported instead of absorbed.
        cross = row.get('body_velocity_cross_max_m_s')
        if cross is not None:
            print(f"   AXIS: max |Uy| after projection = {cross:.3g} m/s; "
                  f"free-stream tilt off the axis = "
                  f"{(row.get('axis_tilt_max') or 0) * 100:.1f}%")
            if row.get('body_velocity_max_m_s') and \
                    cross > 1e-3 * row['body_velocity_max_m_s']:
                print("   !! Ubody STILL HAS A CROSS-STREAM COMPONENT. The "
                      "projection axis is not geometric.")
        if row.get('body_velocity_max_m_s') is None:
            print("   !! NO BODY VELOCITY PARSED -- the log format changed and "
                  "the plausibility check is dead. Do not read the transport.")
            if tr['translating'] is False:
                print("   !! NOT A TRANSLATION. The body changed length OR height")
                print("      while the centroid moved, so the clot is being eroded,")
                print("      smeared or stretched, not carried. Ignore")
                print("      displacement_cells in that case.")
        print(f"   c_min {tr['c_min_start']:.3f} -> {tr['c_min_end']:.3f}; "
              f"material {tr['material_change_fraction'] * 100:+.1f}%; "
              f"dissolving = {tr['dissolving']}")
        # THE EXIT, reported separately from the translation. Transport is now
        # measured over the window the body was INSIDE the domain, because
        # run 20260813-013923 translated coherently and then left, and
        # measuring across the exit reported core -100%, length -100%,
        # c_min -0.997 -> -0.004: a dissolution signature produced by a clot
        # that simply departed.
        if tr.get("reached_outlet") is not None:
            print(f"   EXIT: reached outlet = {tr['reached_outlet']}"
                  + (f", from t = {tr['exit_started_s']} s"
                     if tr.get("exit_started_s") else "")
                  + f"; transport measured to t = {tr['transport_end_time_s']} s")
        if tr.get("residue_fraction") is not None:
            print(f"   RESIDUE: {tr['residue_fraction'] * 100:.0f}% of the "
                  f"material is still in the domain at t = "
                  f"{tr['final_time_s']} s "
                  f"({tr['clot_cells_final']} cells below c = 0)")
            if tr.get("stranded_residue"):
                print("   !! STRANDED RESIDUE. The core left but its material")
                print("      did not. It carries real Brinkman drag and blocks")
                print("      the channel. Colour by clotDrag to see it.")
    else:
        print(f"   TRANSPORT: not measured - {tr['reason']}")
        # THE RESIDUE. Run 20260812-200349 released everything, the body
        # reached the outlet and left -- and 83% of its material stayed
        # behind as a frozen wedge that blocked the channel. The old code
        # printed only "no clot material at one of the transport endpoints"
        # and every number was None, so the actual failure was invisible.
        if tr.get("material_start") is not None:
            print(f"   MATERIAL: {tr['material_start']:.1f} -> "
                  f"{tr['material_end']:.1f} "
                  f"({(tr['residue_fraction'] or 0) * 100:.0f}% left in the "
                  f"domain); c_min {tr['c_min_start']:.3f} -> "
                  f"{tr['c_min_end']:.3f}")
        if tr.get("stranded_residue"):
            print("   !! STRANDED RESIDUE. The core left but its material did")
            print("      not. That residue carries real Brinkman drag and")
            print("      blocks the channel. Colour by clotDrag to see it.")
        elif tr.get("left_cleanly"):
            print("   CLEAN EXIT: the clot carried its material out of the domain.")

print('\n' + '=' * 72)
print('DEFORMATION vs FLOW  (ARTERIAL - deliberately outside the PI\'s venous brief)')
print('=' * 72)
dfm = res['deformation']
print(f"  venous reference at 200/s: {dfm['venous_reference_strain_percent']}% strain")
print(f"  {'shear 1/s':<11}{'traction Pa':<13}{'Us um':<9}{'strain %':<10}"
      f"{'cells':<8}{'predicted %':<13}{'converged':<11}{'visible'}")
for row in dfm['rows']:
    print(f"  {row['wall_shear_rate_1_s']:<11.0f}"
          f"{row['traction_pa'] or float('nan'):<13.2f}"
          f"{row['Us_max_um']:<9.3f}{row['strain_percent']:<10.2f}"
          f"{row['displacement_cells']:<8.2f}"
          f"{row['predicted_strain_percent']:<13.2f}"
          f"{str(row['skeleton_converged']):<11}"
          f"{row['visible_without_warping']}")
for gone in dfm['excluded']:
    print(f"  EXCLUDED {gone['child_id']}: {gone['why']}")
if dfm['rows_discarded_for_convergence']:
    print(f"  !! {dfm['rows_discarded_for_convergence']} row(s) have an UNCONVERGED")
    print('     skeleton. Their strain is what the solver reached at its iteration')
    print('     cap, not what the model predicts. Do not quote those numbers.')
if dfm['deforms_visibly']:
    print(f"  Visible deformation without warping from "
          f"{dfm['lowest_visible_shear_1_s']:.0f} 1/s upward.")
else:
    print('  No child deformed enough to see without warping. Raising the shear')
    print('  further, or reporting Warp By Vector, are the options.')
print(f"  NOTE: {dfm['_this_series_is_arterial']}")

print(f"\ncohesive (bulk-fracture) release anywhere: {res['cohesive_release_anywhere']} cells")
print('   Expected to be 0: AT2 bulk fracture needs about 2900% strain on this')
print('   formulation. A non-zero value is a finding, not a failure.')

print('\n' + '=' * 72)
print('LIMITATIONS TO STATE WITH ANY OF THE ABOVE')
print('=' * 72)
for item in res['numerical_limitations']:
    print(f"  - {item}")

cost = {k: v for k, v in res['realised_cost_seconds'].items() if v}
if cost:
    print(f"\nrealised cost: {sum(cost.values()) / 60:.0f} min total, "
          f"{max(cost.values()) / 60:.0f} min for the slowest child")
print('\n' + '=' * 72)
print('PARAVIEW')
print('=' * 72)
print(f"  {run_drive}/case-after-run-AM2020_planar_mural_detachment/runs/<child>/case.foam")
print("  Best child to show:  demo_full_release")
print()
print("  c          the clot. Contour at c = 0 for the outline, or Threshold")
print("             c from -1 to 0 for the body. c < 0 is clot, c > 0 is blood.")
print("  released   1 on clot no longer connected to an intact bond. This is")
print("             the embolus; Threshold released > 0.5 to isolate it.")
print("  Us         skeleton displacement. Warp By Vector, scale 50-100, to")
print("             see the peel lift off the wall - it is about 1e-7 m, so")
print("             it is invisible at true scale.")
print("  U, w       flow, and the platelet field that drives growth.")
print()
print("  For DEFORMATION, show deform_s4000 (or the highest converged child):")
print("             Warp By Vector on Us. At arterial shear the displacement is")
print("             microns rather than a tenth of a micron, so a warp of 5-10")
print("             is enough - check the strain table above before choosing.")
print()
print("  Frames 0 -> 1.2 s are growth and peeling: the clot grows and the bond")
print("  fails, but the body does not travel. Frames after 1.2 s are the")
print("  transport stage at dt = 4e-4 s, where the released material is")
print("  carried downstream. Step through those to see it leave.")

# --- COHESIVE TEAR ----------------------------------------------------------
ct = res.get('cohesive_tear', {})
if ct.get('rows'):
    print()
    print('COHESIVE TEAR -- does the clot break through the BODY?')
    art = ct.get('driver_artefact')
    if art is not None:
        print(f"  damage driver: interface/bulk = {art:.2f}", end='')
        if art >= 1.0:
            print('  <-- ARTEFACT')
            print('  !! THE PHASE INTERFACE IS DRIVING THE DAMAGE, NOT THE MATERIAL.')
            print('     The Us pin holds the skeleton at zero in the fluid, so grad(Us)')
            print('     across the 4-cell interface is spurious and the fracture')
            print('     criterion fires there first. Run 20260814-055141 measured 13.5.')
            print('     Nothing below this line can be read as a material property.')
        else:
            print('  (the material drives it -- results are readable)')
    if ct['wall_held_everywhere'] is not True:
        print('!! A WALL FACE RELEASED. The bond was supposed to be unbreakable,')
        print('   so the clot had a second way out and no tear here means what')
        print('   it looks like. Stop and read faces_released.')
    for row in ct['rows']:
        state = row['tear_mode'] if row['tore'] else 'held'
        where = ''
        if row['tore']:
            where = (f", at y = {row['tear_y_mean_m'] * 1e6:.1f} um "
                     f"({row['tear_height_fraction'] * 100:.0f}% of clot height), "
                     f"{row['tear_cells']} cells, t = {row['first_tear_time_s']}")
        print(f"  Gc = {row['Gc_j_m2']:.0e} "
              f"({row['orders_below_physiological']:.1f} orders below "
              f"physiological): {state}{where}")
        mat = row.get('material_remaining_fraction')
        if mat is not None:
            print(f"     material left: {mat * 100:.0f}% of initial; "
                  f"intact clot above the damage: "
                  f"{row['intact_cells_above_damage']} cells")
        if row['tear_mode'] == 'surface_ablation':
            print('     !! SURFACE ABLATION, NOT A TEAR. The damage reached the outer')
            print('        surface with nothing intact beyond it, so material came off')
            print('        layer by layer instead of a piece being cut out. No fragment')
            print('        can form this way. A localised tear needs somewhere to')
            print('        concentrate -- a neck, or a stiff core against a soft shell.')
        if row['tear_at_wall']:
            print('     !! THE TEAR IS AT THE WALL. That is the bond failing under')
            print('        another name -- the whole-body release Dr Li rejected.')
        if row['shows_what_the_PI_asked_for']:
            print('     ^^ THIS IS THE PICTURE: wall held, body tore, fragment left.')
    print(f"  required Gc = {ct['required_Gc_j_m2']}, "
          f"lowest that held = {ct['lowest_Gc_that_held_j_m2']}")
    if ct['bracket_note']:
        print('  !!', ct['bracket_note'])
    else:
        print(f"  bracketed: the body needs a toughness "
              f"{ct['orders_below_physiological']:.1f} orders below the measured "
              f"{ct['physiological_Gc_j_m2']} J/m2 before it tears at 250 1/s.")
    print('  ', ct['what_this_does_not_show'])

print('\n' + '=' * 72)
print('SOURCE 33 REPLICATION  (Xu et al. 2017 -- core/shell, growth, 1000 1/s)')
print('=' * 72)
cs = res.get('source33_replication', {})
for row in cs.get('rows', []):
    print(f"{row['child_id']}")
    print(f"  material   shell G {row['G_shell_pa']} Pa / k {row['k0p_shell_m2']:.1e} m2"
          f"  |  core G {row['G_core_pa']} Pa / k {row['k0p_core_m2']:.1e} m2")
    print(f"  toughness  shell {row['Gc_shell_j_m2']} J/m2, core {row['Gc_core_j_m2']} J/m2"
          f"   ({row['orders_below_physiological']:.1f} orders below physiological"
          f" -- 0.0 means NOTHING was lowered)")
    print(f"  realised k {row['k_clot_min_m2']:.2e} .. {row['k_clot_max_m2']:.2e} m2,"
          f" floored fraction {row['permeability_floored_fraction']}")
    print()
    print('  --- is the experiment valid ---')
    print(f"  wall held (0 faces released) : {row['wall_held']}  ({row['faces_released']} faces)")
    print(f"  damage driven by the material: {row['driver_is_the_material']}"
          f"  (interface/bulk {row['interface_over_bulk']})")
    print(f"  permeability is the material : {row['permeability_is_real']}")
    if row['skeleton_diverged']:
        print('  *** SKELETON DIVERGED -- a 4 Pa shell may be outside what'
              ' St Venant-Kirchhoff can carry at this shear rate ***')
    print()
    print('  --- did it look like Xu figure 5 ---')
    print(f"  core  centroid shift {row['core_centroid_shift_m']}"
          f"  height {row['core_height_change_fraction']}"
          f"  length {row['core_length_change_fraction']}")
    print(f"  shell centroid shift {row['shell_centroid_shift_m']}"
          f"  height {row['shell_height_change_fraction']}"
          f"  length {row['shell_length_change_fraction']}")
    print(f"  shell moved more than core   : {row['shell_moved_more_than_core']}"
          f"  (ratio {row['shell_over_core_shift']})")
    print(f"  Us  shell {row['Us_shell_max_m']}  core {row['Us_core_max_m']}"
          f"  ratio {row['Us_shell_over_core']}"
          f"  (predicted {row['Us_shell_over_core_predicted']} from G_core/G_shell)")
    print()
    print('  --- growth, and anything that came free ---')
    print(f"  clot area change {row['growth_area_fraction']}  (grew: {row['grew']},"
          f" multiplier {row['growth_rate_multiplier']}x on alpha only)")
    print(f"  material remaining {row['material_remaining_fraction']}")
    print(f"  released cells {row['fragment_cells_max']}, moved downstream"
          f" {row['fragment_moved']}")
    print(f"  Dg_max {row['Dg_max']} -- at physiological toughness this SHOULD be ~0;"
          f" fracture is not the mechanism being tested")
for name, failed in cs.get('the_experiment_is_valid_only_if', {}).items():
    if failed:
        print(f"  *** INVALID: {name} not met by {failed} ***")


In [ ]:
# ZIP THE RUN AND DOWNLOAD IT STRAIGHT TO THE LOCAL MACHINE.
#
# Faster than waiting for Drive to sync: the archive is built on Colab's local
# disk and pushed over the browser connection, so nothing round-trips through
# Drive twice. The Drive copy still exists and is still the archive of record --
# this is a convenience path, not a replacement for it.
#
# SLIM = True drops the per-timestep field directories and keeps the assessment,
# the logs, the case inputs and the environment record. That is everything
# needed to read the result; it is NOT enough to open the run in ParaView.
# Use it when the full bundle is too large to download reliably.
SLIM = False

import shutil
from pathlib import Path
from google.colab import files

run_drive = Path('/content/drive/MyDrive/OpenFOAM_Blood/runs') / run_id
assert run_drive.is_dir(), f'no archived run at {run_drive}'

staged = Path('/content/download') / run_id
if staged.exists():
    shutil.rmtree(staged)
staged.parent.mkdir(parents=True, exist_ok=True)

def keep(path: Path) -> bool:
    """In SLIM mode, drop written time directories but keep 0/ and everything else."""
    if not SLIM:
        return True
    for part in path.relative_to(run_drive).parts:
        if part in ('0', 'constant', 'system'):
            return True
        try:
            # a numeric directory name is a written output time
            if float(part) > 0.0:
                return False
        except ValueError:
            continue
    return True

# Copy to local disk first. Zipping directly out of the mounted Drive reads
# every byte back over the network, which is the cost this cell exists to avoid.
copied = 0
for src in run_drive.rglob('*'):
    if src.is_dir() or not keep(src):
        continue
    dst = staged / src.relative_to(run_drive)
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    copied += 1

bundle_base = Path('/content') / (run_id + ('-slim' if SLIM else ''))
archive = shutil.make_archive(str(bundle_base), 'zip', root_dir=staged)
size_mb = Path(archive).stat().st_size / 1e6
print(f'{copied} files -> {Path(archive).name}   {size_mb:.1f} MB')

if size_mb > 500:
    print()
    print('LARGER THAN 500 MB. Browser downloads of this size often stall or')
    print('truncate. Set SLIM = True and re-run this cell if it fails; the')
    print('Drive copy at the path below is unaffected either way.')
    print(f'  {run_drive}')

files.download(archive)
